*[cell 0]*

# Final report, practical application (calibration on all data) — publication statistics

Assembles the manuscript's results from the pipeline's exports and writes `LVL_OUTS/final_replication_publication_report.md` (every table, in order, with an appendix of the supporting analyses the manuscript does not cite) and `LVL_OUTS/final_replication_supp_tables.tex` (the supplementary tables: the H1 detector-pair matrices of Tables 2.10a–d and the four H3 tables). It reads exports (and the bytes of lvl1.py, lvl2.py and the sweeps' script copies, for the code checks of Table 7.3) and runs no other notebook's code, apart from executing lvl2.py once per LVL_2 run, read-only and in a subprocess, through its Step 5, to rebuild the run's block matrices for the diagnostics of the exact-zero folded units (Tables 4.30–4.32; `[cell 9]`; nothing is written). The statistics it computes itself: the method-level association and multivariate-outlier test of Tables 2.8–2.9 (mean η²_H against mean Corr%; the LVL1 notebook's own Mahalanobis export, of mean η²_H and mean blocks, is only reproduced as a check in Table 7.3); the comparisons across datasets (the change in DSC between the datasets, the summaries across the datasets and the claims table); the BH families stated under its tables (the 'q (all …)' columns, the q of Tables 5.6a–e and the per-scale q of Table 5.20); the descriptives of Tables 5.18–5.19 and the agreement summary of Table 5.11; the exact-zero diagnostics of Tables 4.30–4.32 (the carried labellers refitted by lvl2.py's own Step 5 with the exact-zero folded values replaced, and the stability refits counted by the copies of the exact-zero blocks they draw). A missing input stops it (`[cell 1]`).

| Section | Content | Source |
|---|---|---|
| 0 | Sample, session duration, behavioural features, drift models | demographic_report, LVL0_report, LVL1 notebook |
| 1 | LVL1 — calibration on all data | LVL1_sweep_report |
| 2 | LVL1 — H1 segmentation across the datasets | LVL1 notebook; Tables 2.8–2.9: this report, from the LVL1 sweep report's and the LVL1 notebook's exports |
| 3 | LVL2 — classification calibration on all data | LVL2_sweep_report; LVL2 T2 + T3 run |
| 4 | LVL2 — H2, robustness, classification across datasets, exact-zero folded units | LVL2 runs (Step 5e); Tables 4.30–4.32: this report, from a read-only rebuild of the runs' block matrices |
| 5 | LVL2 — H3 interrater reliability, sensitivities | LVL2 runs (Steps 8–10) |
| 6 | Consolidated results and exclusions | this report |
| 7 | Notes, checks, provenance; the Markdown and LaTeX exports | this report |

**Datasets.** T2 only (the T2 run), T3 only (the T3 run) and the T2 + T3 run's Pooled, overall (one fit on both conditions), with its restrictions to each condition, Pooled, T2 and Pooled, T3. The configuration was calibrated on all data (T2 and T3), so every dataset is in-sample; the pooled datasets share their participants with T2 only and T3 only. H1 (the LVL1 notebook's exports) has T2+T3 calibration (the LVL1 sweep's sessions, each detector at its own optimal configuration), T3 only and the pooled datasets.

**Cohorts.** The manuscript calls the two conditions cohorts: T2 = Cohort A (CA), T3 = Cohort B (CB). This notebook, its exports and the report keep T2 and T3; only make_replication_figures prints CA and CB.

**Conventions.** Scaled MAD (sMAD) = 1.4826 × median absolute deviation. Summaries are median [Q1, Q3] (sMAD). Bootstrap p = (count + 1)/(B + 1). BH = Benjamini–Hochberg FDR; families are stated under each table. Per trial = per stimulus-trial removed, × 10⁻⁶. † marks the selected method.

In [ ]:
# [cell 1]
# ===== INPUTS: set these before running =====
LVL0_HANDOFF = "lv0_out_20261106_180438.csv"            # LVL_0 handoff in LVL_OUTS (LVL0 report, sample description)
LVL1_HANDOFF = "lv1_out_20261106_190046"                # LVL_1 handoff prefix in LVL_OUTS (H1, behavioural features)
LVL1_SWEEP   = "_sweep_lvl1_all"                         # LVL_1 T2+T3 calibration sweep (its report's exports)
LVL2_SWEEP   = "_sweep_lvl2_all_PELT_Attention_comp"        # LVL_2 T2+T3 calibration sweep (its report's exports)
RUN_TAGS = {"T2":   "lv2_out_PELT_Attention_comp_T2_20261107_192158",
            "T3":   "lv2_out_PELT_Attention_comp_T3_20261107_192158",
# the pooled T2 + T3 run
            "both": "lv2_out_PELT_Attention_comp_T2+T3_20261103_224209",
            }

import ast, difflib, json, os, re, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown
import platform, sys, scipy, sklearn, matplotlib, statsmodels
print(f"interpreter {sys.executable} (Python {platform.python_version()}) | numpy {np.__version__} | pandas "
      f"{pd.__version__} | scipy {scipy.__version__} | scikit-learn {sklearn.__version__} | matplotlib "
      f"{matplotlib.__version__} | statsmodels {statsmodels.__version__}")

# ---- locations: everything is inside /home/sjsuss626/EMST_files/sjs_dissertation ----------------
LVL_OUTS   = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/lvl_outs")
SWEEPS_DIR = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/sweeps")
FIG_DIR    = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/figures/pub_figures")
CODE_DIR   = Path("/home/sjsuss626/EMST_files/sjs_dissertation/code/practical_application_emst")              # lvl1.py, lvl2.py
REPORT_MD  = LVL_OUTS / "final_replication_publication_report.md"     # the report this notebook writes
REPORT_TEX = LVL_OUTS / "final_replication_supp_tables.tex"           # the supplementary tables, as LaTeX
ALPHA = 0.05
PIPELINE_VERSION = "0.0.1"     # development series; must equal the level notebooks' [cell 2] literal

_pins = dict(LVL0_HANDOFF=LVL0_HANDOFF, LVL1_HANDOFF=LVL1_HANDOFF, LVL1_SWEEP=LVL1_SWEEP, LVL2_SWEEP=LVL2_SWEEP,
             RUN_TAGS=RUN_TAGS)
if any(v is None for v in _pins.values()):
    raise SystemExit(f"set every input at the top of this cell; not set: {[k for k, v in _pins.items() if v is None]}")
LV0 = LVL_OUTS / LVL0_HANDOFF.removesuffix(".csv")
HANDOFF = LVL_OUTS / LVL1_HANDOFF
RUN = {k: LVL_OUTS / v for k, v in RUN_TAGS.items()}
_need = ([Path(f"{LV0}.csv"), Path(f"{LV0}_parameters.json")]
         + [Path(f"{LV0}_report_{k}.csv") for k in ("session_duration", "session_duration_tests", "rt_limits",
                                                   "drift_models", "model_selection")]
         + [Path(f"{LV0}_demographics_{k}.csv") for k in ("ledger", "flow", "single_session", "single_session_counts",
                                                         "balance", "table", "intervals", "out_of_range")]
         + [Path(f"{HANDOFF}{s}") for s in ("_meta.json", "_parameters.json", "_report_table_nonparametric.csv",
                                           "_report_age_continuous.csv", "_report_age_groups.csv", "_results.csv",
                                           "_features.csv")]
         + [Path(f"{HANDOFF}_h1_{k}.csv") for k in ("desc_overall", "desc_winner", "desc_detector", "tests", "summary",
                                                   "method_association", "mahalanobis", "ranks")]
         + [LVL_OUTS / f"{LVL1_SWEEP}_report_{k}.csv" for k in ("knob_ss", "selection_freq", "best_per_detector",
                                                                     "best_eta", "best_corr", "configs")]
         + [LVL_OUTS / f"{LVL1_SWEEP}_selection.json", LVL_OUTS / f"{LVL2_SWEEP}_selection.json"]
         + [LVL_OUTS / f"{LVL2_SWEEP}_report_{k}.csv" for k in ("dsc_summary", "dsc_variance", "dsc_picks", "dsc_maxima",
                                                              "dsc_units", "dsc_configs", "dsc_bands", "stability", "dose_fits",
                                                              "dose_contrasts", "dose_contrasts_per_config", "dose_picks")]
         + [SWEEPS_DIR / LVL2_SWEEP / "apriori" / "inattent999000_heuristic.json"]
         + [SWEEPS_DIR / s / "_provenance.json" for s in (LVL1_SWEEP, LVL2_SWEEP)] + [CODE_DIR / "lvl1.py", CODE_DIR / "lvl2.py"]
         + [Path(f"{p}{s}") for p in RUN.values() for s in ("_config.json", "_parameters.json", "_step8_multimetric.csv",
                                                             "_step8_null_overall.csv", "_step9_design_tests.csv",
                                                             "_step9_null_contrasts.csv", "_step9_cell_tests.csv",
                                                             "_post_index.csv", "_post_replicates.npz", "_units.csv")])
_missing = [str(p) for p in _need if not p.exists()]
if _missing:
    raise SystemExit("inputs not found (run the stage that writes each one first; README, execution order):\n  "
                     + "\n  ".join(_missing))
# the post hoc tables of every run, as listed in its index
POST_FILES = {k: pd.read_csv(f"{p}_post_index.csv") for k, p in RUN.items()}
_missing = [f for k, d in POST_FILES.items() for f in d.file if not (LVL_OUTS / f).exists()]
if _missing:
    raise SystemExit(f"post hoc exports listed in the runs' indexes but not found: {_missing}")

META = json.load(open(f"{HANDOFF}_meta.json"))
SEL1 = json.load(open(LVL_OUTS / f"{LVL1_SWEEP}_selection.json"))
SEL2 = json.load(open(LVL_OUTS / f"{LVL2_SWEEP}_selection.json"))
CONFIG = {k: json.load(open(f"{p}_config.json")) for k, p in RUN.items()}
RUNLOG = {k: json.load(open(f"{p}_parameters.json")) for k, p in RUN.items()}
WINNER_LVL1 = SEL1["detector"]                       # the selected LVL_1 detector
ANALYSIS_METHOD = WINNER_LVL1
FAM_LABEL = {"cluster": "k-means", "mixture_gmm": "GMM"}
PICKS = {f"{FAM_LABEL[f]} ({d['k']})": (d["config"], d["arm"]) for f, d in SEL2["families"].items()}   # label -> (config, arm)
EXPONENTS = {f"{FAM_LABEL[f]} ({d['k']})": tuple(v for k, v in d["LVL_2_CONFIG"].items() if "exponent" in k)
             for f, d in SEL2["families"].items()}   # label -> (entropy exponent, RT-variance exponent)
SCALES = list(json.loads(json.dumps(eval(META["lvl1_config"]["scale_types"]))))
print(f"LVL_1 detector {WINNER_LVL1} | carried solutions {PICKS} | scales {SCALES}")

In [ ]:
# [cell 2]
# ---- descriptive summaries, p-values, FDR ------------------------------------------------------------
def smad(x):
    """Scaled MAD = 1.4826 x median absolute deviation."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return 1.4826 * np.median(np.abs(x - np.median(x))) if len(x) else np.nan


def summ(x):
    """n, median, Q1, Q3, scaled MAD of the finite values of x."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if not len(x):
        return dict(n=0, median=np.nan, q1=np.nan, q3=np.nan, smad=np.nan)
    return dict(n=len(x), median=np.median(x), q1=np.percentile(x, 25), q3=np.percentile(x, 75), smad=smad(x))


def summ_str(x, nd=3):
    """'median [Q1, Q3] (sMAD)' of x."""
    s = summ(x)
    return "—" if not s["n"] else f"{s['median']:.{nd}f} [{s['q1']:.{nd}f}, {s['q3']:.{nd}f}] ({s['smad']:.{nd}f})"


def p_two(star, null=0.0):
    """Two-sided bootstrap p: 2 x the smaller tail, each with the +1 correction, capped at 1."""
    star = np.asarray(star, float); star = star[np.isfinite(star)]
    if not len(star):
        return np.nan
    n = len(star)
    return min(1.0, 2 * min((np.sum(star <= null) + 1) / (n + 1), (np.sum(star >= null) + 1) / (n + 1)))


def bh(p):
    """Benjamini-Hochberg q-values; NaN stays NaN."""
    p = np.asarray(p, float); q = np.full(len(p), np.nan); ok = np.isfinite(p)
    if ok.any():
        q[ok] = multipletests(p[ok], method="fdr_bh")[1]
    return q


def ci(star, lo=2.5, hi=97.5):
    star = np.asarray(star, float); star = star[np.isfinite(star)]
    return (np.percentile(star, lo), np.percentile(star, hi)) if len(star) else (np.nan, np.nan)


def ci_str(lo, hi, nd=3):
    return "—" if not (np.isfinite(lo) and np.isfinite(hi)) else f"[{lo:.{nd}f}, {hi:.{nd}f}]"


# ---- tables: one registry feeds both the notebook display and the Markdown export -----------------
TABLES = []            # (section, key, title, string table, note)
NUM = {}               # key -> numeric table (for reuse in the cross-dataset tables)
FIGURES = []           # (section, key, title, path, note): figures, linked in the Markdown export
SECTION_TITLES = {0: "Sample, preprocessing and behavioural features", 1: "LVL1 — calibration on all data",
                  2: "LVL1 — H1 segmentation across the datasets", 3: "LVL2 — classification calibration on all data",
                  4: "LVL2 — H2 flagged vs unflagged blocks and robustness",
                  5: "LVL2 — H3 interrater reliability", 6: "Consolidated results",
                  7: "Statistical notes, exclusions and source provenance"}
# Supporting analyses the manuscript does not cite: printed in the appendix (by table number), after section 7.
APPENDIX = ("Table 2.6.", "Table 3.5.", "Table 3.6b.", "Table 3.7.", "Table 3.8.", "Table 3.9.", "Table 3.10.",
            "Table 3.11.", "Table 3.12.", "Table 3.13.", "Table 3.14.", "Table 4.4.", "Table 4.5.", "Table 4.10.",
            "Table 4.11.", "Table 4.12.", "Table 4.13.", "Table 4.14.", "Table 4.15.", "Table 4.16.", "Table 4.17.",
            "Table 4.21.", "Table 4.22.", "Table 4.23.", "Table 5.1a.", "Table 5.1b.", "Table 5.3f.", "Table 5.3g.",
            "Table 5.5.", "Table 5.6g.", "Table 5.10.", "Table 5.11.", "Table 5.12.", "Table 5.13.", "Figure 5.1.",
# the pooled datasets' supporting tables
            "Table 2.3c.", "Table 2.3d.", "Table 2.3e.", "Table 4.18.",
            )
in_appendix = lambda title: title.startswith(APPENDIX)
CHECKS = []            # (check, passed, detail) -> section 7


def check(name, ok, detail=""):
    CHECKS.append((name, bool(ok), detail))
    if not ok:
        print(f"(!) CHECK FAILED: {name} {detail}")


_SHA16 = {}


def sha16(path):
    """(bytes, first 16 hex digits of the SHA-256) of a file: the sweeps' script fingerprints and the figure sidecars'
    input records."""
    path = Path(path)
    if path not in _SHA16:
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 22), b""):
                h.update(chunk)
        _SHA16[path] = (path.stat().st_size, h.hexdigest()[:16])
    return _SHA16[path]


def cfg_val(v):
    """A configuration value as JSON (tuples -> lists); a string holding a Python literal (the LVL_1 handoff records
    str(value)) is read back first."""
    if isinstance(v, str):
        try:
            v = ast.literal_eval(v)
        except (ValueError, SyntaxError):
            pass
    return json.loads(json.dumps(v, default=str))


def fmt_p(v):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "—"
    return "< .001" if v < .001 else f"{v:.3f}".replace("0.", ".", 1)


def _fmt_cell(v, nd):
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}" if abs(int(v)) >= 10_000 else str(int(v))
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v):
            return "—"
        s = f"{v:.{nd}f}"
        return "0" + s[2:] if s.startswith("-0") and float(s) == 0 else s
    return "—" if v is None or (isinstance(v, str) and v == "") or (isinstance(v, float) and v != v) else str(v)


def show(section, key, title, df, note="", nd=3, p_cols=None, nd_cols=None, index=False):
    """Format a numeric table to strings ONCE, display it and register it for the Markdown export."""
    df = df.reset_index() if index else df.copy()
    NUM[key] = df.copy()
    is_p = lambda n: (n in ("p", "q") or re.match(r"^(p|q)[ _(]", n) is not None or n.endswith(("_p", "_q"))
                      or re.search(r":\s*(p|q)( \(.*\))?$", n) is not None)
    p_cols = set(p_cols or [c for c in df.columns if is_p(str(c))])
    nd_cols = nd_cols or {}
    sdf = pd.DataFrame({c: [fmt_p(v) if c in p_cols else _fmt_cell(v, nd_cols.get(c, nd)) for v in df[c]]
                        for c in df.columns})
    TABLES.append((section, key, title, sdf, note))
    display(Markdown(f"**{title}**"))
    try:
        display(sdf.style.hide(axis="index"))
    except Exception:
        display(sdf)
    if note:
        display(Markdown(f"*Note.* {note}"))
    return sdf


def show_panels(section, key, title, df, id_cols, panels, note="", **kw):
    """A wide table as consecutive panels sharing the identifying columns (narrow enough for a PDF page)."""
    for j, (lab, cols) in enumerate(panels.items()):
        show(section, f"{key}_{'ab'[j] if len(panels) == 2 else j}", f"{title} — panel {'ABCD'[j]}: {lab}",
             df[id_cols + cols], note if j == len(panels) - 1 else "", **kw)


def show_figure(section, key, title, path, note=""):
    """Register a figure of make_replication_figures.ipynb (with its provenance sidecar) for the Markdown export."""
    path = Path(path)
    side = path.with_suffix(".json")
    ok = path.exists() and side.exists()
    check(f"{title.split('.')[0]}.{title.split('.')[1]}: figure and provenance sidecar present", ok, path.name)
    if ok:
        FIGURES.append((section, key, title, path, note))
        display(Markdown(f"**{title}**" + (f"  \n*Note.* {note}" if note else "")))

In [ ]:
# [cell 3]
# ---- data: every export this report reads ----------------------------------------------------------------
# the datasets: name -> (run, conditions, kind)
DATASETS = {"T2 only":         ("T2",   ["T2"],       "independent"),
            "T3 only":         ("T3",   ["T3"],       "independent"),
# the pooled run's datasets
            "Pooled, overall": ("both", ["T2", "T3"], "pooled"),
            "Pooled, T2":      ("both", ["T2"],       "pooled-derived"),
            "Pooled, T3":      ("both", ["T3"],       "pooled-derived"),
            }
DS = list(DATASETS)
POOLED_DS = [ds for ds, d in DATASETS.items() if d[2] != "independent"]      # the pooled datasets among them
IND_DS = ["T2 only", "T3 only"]           # the independent datasets
H3_DS = DS                                # H3 is reported for every dataset
H1_DS = ["T2+T3 calibration", "T3 only"] + POOLED_DS   # the datasets of the LVL1 notebook's H1 exports
DESIGNS = ["dynamic", "control", "session", "participant"]
L0 = lambda k: pd.read_csv(f"{LV0}_report_{k}.csv")
DEMO = lambda k: pd.read_csv(f"{LV0}_demographics_{k}.csv")
H1X = lambda k: pd.read_csv(f"{HANDOFF}_h1_{k}.csv")
L1R = lambda k: pd.read_csv(f"{HANDOFF}_report_{k}.csv")
S1 = lambda k: pd.read_csv(LVL_OUTS / f"{LVL1_SWEEP}_report_{k}.csv")
S2 = lambda k: pd.read_csv(LVL_OUTS / f"{LVL2_SWEEP}_report_{k}.csv")
_post_csv = lambda f: pd.read_csv(LVL_OUTS / f) if (LVL_OUTS / f).stat().st_size > 1 else pd.DataFrame()  # no rows: empty file
POST = {run: {f.removeprefix(f"{RUN_TAGS[run]}_post_").removesuffix(".csv"): _post_csv(f)
              for f in d.file if f.startswith(f"{RUN_TAGS[run]}_post_") and f.endswith(".csv")}
        for run, d in POST_FILES.items()}


RUN_DATASET = {d[0]: ds for ds, d in DATASETS.items() if d[2] != "pooled-derived"}     # run -> the dataset of its own fit


def P(key, datasets=None):
    """A post hoc table over the runs (each run's rows carry its datasets; a table without a Dataset column gets its
    run's dataset), optionally restricted to some datasets."""
    parts = [(POST[r][key] if "Dataset" in POST[r][key].columns else POST[r][key].assign(Dataset=RUN_DATASET[r])
              [["Dataset"] + list(POST[r][key].columns)]) for r in RUN if key in POST[r] and len(POST[r][key])]
    d = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if datasets is not None and len(d) and "Dataset" in d.columns:
        d = d[d.Dataset.isin(datasets)].reset_index(drop=True)
    return d


def replicates(run):
    z = np.load(f"{RUN[run]}_post_replicates.npz")
    names = json.loads(str(z["index"]))
    return {n: z[f"r{i}"] for i, n in enumerate(names)}


REPL = {run: replicates(run) for run in RUN}
STEP8 = {run: pd.read_csv(f"{p}_step8_multimetric.csv") for run, p in RUN.items()}
NULL8 = {run: pd.read_csv(f"{p}_step8_null_overall.csv") for run, p in RUN.items()}
TESTS9 = {run: pd.read_csv(f"{p}_step9_design_tests.csv") for run, p in RUN.items()}
NULL9 = {run: pd.read_csv(f"{p}_step9_null_contrasts.csv") for run, p in RUN.items()}
CELLS9 = {run: pd.read_csv(f"{p}_step9_cell_tests.csv") for run, p in RUN.items()}

# consistency of the inputs: one LVL_1 handoff and one calibration behind every run; one behavioural heuristic
for run, c in CONFIG.items():
    check(f"LVL_2 run {run}: LVL_1 handoff = {HANDOFF.name}", Path(str(c.get("lvl1_handoff", c.get("lvl1_input", "")))).name
          .startswith(HANDOFF.name), str(c.get("lvl1_handoff", c.get("lvl1_input"))))
    _cfg = c["lvl2_config"] if "lvl2_config" in c else RUNLOG[run]["parameters"]
    _diff = {k: (_cfg.get(k), v) for f in SEL2["families"].values() for k, v in f["LVL_2_CONFIG"].items()
             if _cfg.get(k) != v}
    check(f"LVL_2 run {run}: LVL_2_CONFIG = the LVL2 sweep's selection", not _diff, str(_diff))
check("LVL_1 handoff built from the pinned LVL_0 handoff", META.get("lvl0_input") == Path(f"{LV0}.csv").name,
      str(META.get("lvl0_input")))
_l1cfg = META["lvl1_config"]
check("LVL_1 handoff configuration = the LVL1 sweep's selection",
      all(str(_l1cfg.get(k)) == str(v) for k, v in SEL1["LVL_1_CONFIG"].items()), str(SEL1["LVL_1_CONFIG"]))
check("LVL1 sweep and LVL_1 handoff on the same LVL_0 handoff", SEL1["lvl0_input"] == Path(f"{LV0}.csv").name,
      SEL1["lvl0_input"])
check("LVL2 sweep on the pinned LVL_1 handoff", Path(SEL2["lvl1_input"]).name.startswith(HANDOFF.name), SEL2["lvl1_input"])
# the calibration's lineage (the sweep reports' selection files): every configuration key, the code, the detector and the
# calibration condition
_PROV = {s: json.load(open(SWEEPS_DIR / s / "_provenance.json")) for s in (LVL1_SWEEP, LVL2_SWEEP)}
_EXEMPT1 = {"n_jobs": "number of parallel workers", "parallel_verbose": "log verbosity", "save_figures": "figure output only"}
_EXEMPT2 = {"conditions": "run design: each run's own conditions, checked against its run",
            "save_figures": "figure output only"}
_exempt_txt = lambda e: "; ".join(f"{k} ({r})" for k, r in e.items())
_full1, _l1all = SEL1.get("LVL_1_CONFIG_full"), META["lvl1_config"]
if _full1 is None:
    check("LVL_1 handoff configuration = the LVL1 sweep's full configuration at its selection (every key)", False,
          "the LVL1 selection has no LVL_1_CONFIG_full (rerun LVL1_sweep_report)")
else:
    _dif1 = {k: (_full1.get(k, "missing"), _l1all.get(k, "missing")) for k in sorted(set(_full1) | set(_l1all))
           if k not in _EXEMPT1 and (k not in _full1 or k not in _l1all or cfg_val(_full1[k]) != cfg_val(_l1all[k]))}
    check("LVL_1 handoff configuration = the LVL1 sweep's full configuration at its selection (every key)", not _dif1,
          f"{len(set(_full1) - set(_EXEMPT1))} keys compared; differing: {_dif1 or 'none'}; exempt: {_exempt_txt(_EXEMPT1)}")
_full2 = SEL2.get("LVL_2_CONFIG_full")
_RUN_CONDS = {run: DATASETS[ds][1] for run, ds in RUN_DATASET.items()}
for run in RUN:
    _name = f"LVL_2 run {run}: its configuration = the LVL2 sweep's full configuration at the selection (every key both record)"
    if _full2 is None:
        check(_name, False, "the LVL2 selection has no LVL_2_CONFIG_full (rerun LVL2_sweep_report)")
        continue
    _rec = RUNLOG[run]["parameters"]
    _dif2 = {k: (_full2[k], _rec[k]) for k in _full2 if k in _rec and k not in _EXEMPT2 and cfg_val(_full2[k]) != cfg_val(_rec[k])}
    _c2 = {k: v for k, v in CONFIG[run].items() if k in _full2}            # the knobs _config.json records again
    _c2.update({k: (v["value"] if isinstance(v, dict) and "value" in v else v)
                for arm in CONFIG[run].get("detector_settings", {}).values() if isinstance(arm, dict)
                for k, v in arm.items() if k in _full2})
    _dif2.update({f"{k} (_config.json)": (_full2[k], v) for k, v in _c2.items()
                if k not in _EXEMPT2 and cfg_val(_full2[k]) != cfg_val(v)})
    _n2 = len([k for k in _full2 if k in _rec and k not in _EXEMPT2])
    _cd = sorted(_rec.get("conditions") or [])
    check(_name, not _dif2 and _n2 > 0 and _cd == _RUN_CONDS[run],
          f"{_n2} keys compared ({len(_c2)} also in _config.json); differing: {_dif2 or 'none'}; conditions {_cd}; "
          f"recorded by one side only: {sorted((set(_full2) ^ set(_rec)) - set(_EXEMPT2)) or 'none'}; "
          f"exempt: {_exempt_txt(_EXEMPT2)}")


# ---- check (c) for lvl2.py: the lines on which it may differ from the LVL2 sweep's script copy ----------------------
# A line that differs (difflib, on either side) is admitted by (i) when it is a comment or blank line and by (ii) when it
# sets one of the selection's LVL_2_CONFIG knobs in NOTEBOOK CELL 28 (the lines README step 10 writes). A line no rule
# admits fails the check.
import io, tokenize
LVL2_DIFF_RULES = {"(i)": "comment or blank line",
                   "(ii)": "line setting a selected LVL_2_CONFIG knob in NOTEBOOK CELL 28"}


def lvl2_script_diff(ref, cur, knobs):
    """The lines that differ between the LVL2 sweep's script copy (ref) and lvl2.py (cur), both sides, sorted by the rule
    of check (c) that admits each: ({NOTEBOOK CELL: {rule: number of lines}}, [the lines no rule admits])."""
    sides = []
    for path in (ref, cur):
        text = Path(path).read_text()
        code = set()                # line numbers (from 1) holding a token other than a comment or a line break
        for t in tokenize.generate_tokens(io.StringIO(text).readline):
            if t.type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT, tokenize.DEDENT,
                              tokenize.ENDMARKER):
                code.update(range(t.start[0], t.end[0] + 1))
        lines, cells, c = text.splitlines(), [], None
        for ln in lines:
            m = re.match(r"# =+ NOTEBOOK CELL (\d+) =+\s*$", ln)
            c = int(m.group(1)) if m else c
            cells.append(c)
        sides.append((lines, cells, code))
    admitted, bad = {}, []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, sides[0][0], sides[1][0], autojunk=False).get_opcodes():
        if op == "equal":
            continue
        for name, (lines, cells, code), rows in (("script copy", sides[0], range(i1, i2)),
                                                 ("lvl2.py", sides[1], range(j1, j2))):
            for i in rows:
                key = re.match(r'\s*"(\w+)"\s*:', lines[i])
                rule = ("(i)" if i + 1 not in code else
                        "(ii)" if cells[i] == 28 and key and key.group(1) in knobs else None)
                if rule is None:
                    bad.append(f"{name} line {i + 1} (NOTEBOOK CELL {cells[i]}) is admitted by no rule of check (c): "
                               f"{lines[i].strip()[:70]}")
                else:
                    admitted.setdefault(cells[i], {}).setdefault(rule, 0)
                    admitted[cells[i]][rule] += 1
    return admitted, bad


def lvl2_diff_text(admitted):
    """Check (c)'s detail: every NOTEBOOK CELL with differing lines and the number of lines each rule admitted there."""
    parts = []
    for c in sorted(admitted, key=lambda c: -1 if c is None else c):
        where = "the script's header" if c is None else f"NOTEBOOK CELL {c}"
        parts.append(f"{where}: " + ", ".join(f"{r} {admitted[c][r]}" for r in LVL2_DIFF_RULES if r in admitted[c]))
    rules = "; ".join(f"{r} {txt}" for r, txt in LVL2_DIFF_RULES.items())
    return "; ".join(parts) + f" [lines per rule: {rules}]"


_fp1 = sha16(CODE_DIR / "lvl1.py")[1]
_cp1 = SEL1.get("script_copy")
_cp1_fp = sha16(_cp1)[1] if _cp1 and Path(_cp1).is_file() else "missing"
check("Code: lvl1.py = the script the LVL1 sweep ran (fingerprint)",
      _fp1 == SEL1["pipeline_fingerprint"] == _PROV[LVL1_SWEEP]["pipeline_fingerprint"] == _cp1_fp,
      f"lvl1.py {_fp1}; selection {SEL1['pipeline_fingerprint']}; sweep {_PROV[LVL1_SWEEP]['pipeline_fingerprint']}; "
      f"script copy {_cp1}: {_cp1_fp}")
_fp2 = sha16(CODE_DIR / "lvl2.py")[1]
_cp2 = SEL2.get("script_copy")
_cp2_fp = sha16(_cp2)[1] if _cp2 and Path(_cp2).is_file() else "missing"
_knobs2 = sorted({k for f in SEL2["families"].values() for k in f["LVL_2_CONFIG"]})
_adm2, _bad2 = (({}, []) if _fp2 == SEL2["pipeline_fingerprint"] or _cp2_fp == "missing"
                else lvl2_script_diff(_cp2, CODE_DIR / "lvl2.py", _knobs2))
check("Code: lvl2.py = the script the LVL2 sweep ran, or differs from it only on comment or blank lines and on the "
      "selection's knobs in NOTEBOOK CELL 28",
      _cp2_fp == SEL2["pipeline_fingerprint"] == _PROV[LVL2_SWEEP]["pipeline_fingerprint"] and not _bad2,
      f"lvl2.py {_fp2}; selection {SEL2['pipeline_fingerprint']}; sweep {_PROV[LVL2_SWEEP]['pipeline_fingerprint']}; "
      f"script copy {_cp2}: {_cp2_fp}; "
      + ("identical" if _fp2 == SEL2["pipeline_fingerprint"] else "no script copy to compare" if _cp2_fp == "missing" else
         f"differing lines by NOTEBOOK CELL and the rule admitting them: {lvl2_diff_text(_adm2)}"
         + (f"; admitted by no rule: {_bad2[:6]}" if _bad2 else "")))
_DET = {"LVL1 selection": SEL1["detector"], "LVL2 selection": SEL2["analysis_method"],
        **{f"run {r}": CONFIG[r].get("analysis_method") for r in RUN},
        **{f"run {r} (parameters)": RUNLOG[r]["parameters"].get("analysis_method") for r in RUN}}
check("Detector: the LVL2 sweep and every LVL_2 run use the LVL1-selected detector", len(set(_DET.values())) == 1, str(_DET))
_CAL = {"LVL1 selection": SEL1.get("condition"), "LVL2 selection": SEL2.get("condition"),
        "LVL1 sweep": _PROV[LVL1_SWEEP].get("condition"), "LVL2 sweep": _PROV[LVL2_SWEEP].get("condition")}
check("Calibration on all data (T2 and T3): both sweeps and their selections",
      all(v == "both" for v in _CAL.values()), str(_CAL))
_PVER = {"LVL_0": json.load(open(f"{LV0}_parameters.json")).get("pipeline_version"),
         "LVL_1": json.load(open(f"{HANDOFF}_parameters.json")).get("pipeline_version"),
         **{f"LVL_2 run {r}": RUNLOG[r].get("pipeline_version") for r in RUN}}
check("Pipeline version: the LVL0, LVL1 and LVL2 run logs record PIPELINE_VERSION",
      all(v == PIPELINE_VERSION for v in _PVER.values()),
      f"PIPELINE_VERSION {PIPELINE_VERSION}; " + "; ".join(f"{k} {v or 'not recorded'}" for k, v in _PVER.items()))
_heur = {"a-priori (T2+T3 sweep)": json.load(open(SWEEPS_DIR / LVL2_SWEEP / "apriori" / "inattent999000_heuristic.json"))["heuristic"]}
_heur.update({f"{k} run": c["detector_settings"]["inattent_behav"] for k, c in CONFIG.items()})
HEURISTIC = pd.DataFrame({lab: {k: str(v["value"]) for k, v in h.items()} for lab, h in _heur.items()})
check("one behavioural heuristic: the a-priori standard and every LVL_2 run",
      not HEURISTIC.isna().any().any() and all(HEURISTIC[c].equals(HEURISTIC.iloc[:, 0]) for c in HEURISTIC.columns))
_REP_KEYS = {"repeat_exposure_h2", "repeat_exposure_h3", "repeat_exposure_h3_excluded"}
check("every run exports the same post hoc tables (the repeat-exposure sensitivity: the runs whose data hold repeat exposures)",
      len({tuple(sorted(set(POST[r]) - _REP_KEYS)) for r in POST}) == 1
      and all((len(POST[r]["repeat_exposures"]) > 0) == _REP_KEYS.issubset(POST[r]) for r in POST),
      str({r: len(POST[r]) for r in POST}))
print({r: len(POST[r]) for r in POST}, "post hoc tables per run;", {r: len(REPL[r]) for r in REPL}, "replicate sets")

In [ ]:
# [cell 4]
# ---- 0. sample, session duration, behavioural features, preprocessing and drift models ---------------------------
# demographic_report (sample), LVL0_report (duration, RT limits, drift models) and the LVL1 notebook (behavioural
# features, age check), each on the pinned handoffs.
_P0 = json.load(open(f"{LV0}_parameters.json"))["parameters"]
_L1 = {k: cfg_val(META["lvl1_config"][k]) for k in ("pr_stim_drop", "min_rt_thresh", "max_rt_thresh", "rt_threshold_na_handling")}


def _tset(c):
    """A trial set in words: a configuration's practice stimuli and RT limits (a limit of 0 is off)."""
    lim = " and ".join(x for x in (f"RT ≥ {c['min_rt_thresh']:g} s" if c["min_rt_thresh"] else "",
                                   f"RT ≤ {c['max_rt_thresh']:g} s" if c["max_rt_thresh"] else "") if x)
    return ((f"the stimuli after the first {c['pr_stim_drop']} of each session" if c["pr_stim_drop"] else "every stimulus")
            + ", " + (lim or "every RT"))


_NA_TXT = {"bfill": "the RT features fill it with that scale's next valid RT in the session; the rolling variance only "
                    "where too few values remain in its window",
           "ffill": "the RT features fill it with that scale's previous valid RT in the session; the rolling variance only "
                    "where too few values remain in its window",
           "partial": "the RT features average the stimulus's other scales",
           "drop_stimulus": "the whole stimulus is missing in every LVL_1 feature"}
_bal = DEMO("balance")
show(0, "0a", "Table 0.1. Sample by condition: composition and T2 vs T3 comparisons", _bal, nd=2,
     note=f"Analysed sample (LVL_0 handoff, participants aged ≤ {_P0['max_age']}). χ² tests without continuity correction; "
          "Mann–Whitney U two-sided. Sex as recorded (female / male).")
_dur = L0("session_duration")
show(0, "0b", "Table 0.2. Session duration (summed response times) by condition and session", _dur,
     nd_cols={"Sessions": 0})
show(0, "0b2", "Table 0.2b. Session duration: T2 vs T3 and S1 vs S2 tests", L0("session_duration_tests"), nd=1,
     nd_cols={"n": 0},
     note="Duration = sum of the raw response times of every response in the session (five scales per stimulus), in minutes; "
          "time between responses is not recorded. S1 vs S2 among participants with both sessions. q = BH over the four tests.")
_np = L1R("table_nonparametric")
_cols = ["T2-overall", "T3-overall", "T2-S1", "T2-S2", "T3-S1", "T3-S2", "S1-overall", "S2-overall", "task-overall"]
_pc0 = ["T2 vs T3 p_FDR", "S1 vs S2 (T2) p_FDR", "S1 vs S2 (T3) p_FDR"]
show(0, "0c", "Table 0.3. Behavioural features by condition and session: Mdn (Q1–Q3) and BH-corrected p",
     _np[["feature", "scale"] + _cols + _pc0].rename(columns={c: c.replace(" p_FDR", " q") for c in _pc0}),
     p_cols=[c.replace(" p_FDR", " q") for c in _pc0],
     note="LVL1 notebook (descriptives) on the pinned LVL_1 handoff. Columns spanning both sessions (T2, T3, S1-overall, "
          "S2-overall, task-overall) average each participant's sessions first. T2 vs T3: Mann–Whitney U; S1 vs S2: Wilcoxon "
          "signed-rank among participants with both sessions; q = BH within each feature (6 rows × 3 comparisons). RT, "
          f"both RT variances (raw RT in seconds) and the session-level entropy use LVL_0's trial set ({_tset(_P0)}), "
          f"not the analysed one ({_tset(_L1)}; Table 0.4); the sliding-window entropy is LVL_1's rolling entropy of the "
          "analysed stimuli.")
show(0, "0d", "Table 0.4. Preprocessing: RTs outside the limits", L0("rt_limits"), nd_cols={"n": 0},
     note=f"The limits are LVL_0's: the responses outside them and the first {_P0['pr_stim_drop']} stimuli of each session "
          "(practice) are left out of LVL_0's drift-model fits and of Table 0.3's RT, RT-variance and session-level "
          "entropy descriptives. The analysed data "
          f"apply LVL_1's own limits ({_tset(_L1)}): "
          + (f"RTs above {_L1['max_rt_thresh']:g} s are set to missing with their rating "
             f"({_NA_TXT[_L1['rt_threshold_na_handling']]})" if _L1["max_rt_thresh"] else "there is no upper RT limit")
          + ("; " + (f"RTs below {_L1['min_rt_thresh']:g} s likewise" if _L1["min_rt_thresh"] else
                     f"there is no lower RT limit, so the responses faster than {_P0['min_rt_thresh']:g} s keep their RT "
                     "and rating and enter every LVL_1 feature and every later analysis (the behavioural heuristic marks "
                     f"RTs below {RUNLOG['T2']['parameters']['inattent_behav_fast_rt_thresh']:g} s as fast)"))
          + ("; practice stimuli are analysed like every other stimulus." if not _L1["pr_stim_drop"] else "."))
show(0, "0e", "Table 0.5. Drift models: the selected model per span and its fit per scale", L0("drift_models"),
     note="Each span's model is chosen by LVL_0 as the candidate with the best mean R² over the five scales, fitted to the "
          "median RT in seconds at each trial position (Table 0.6; practice stimuli excluded). The residuals LVL_1 uses are "
          "log RT minus the same model fitted, per session, to the median log RT at each trial position (the log-scale R²; "
          "the curves of Figure 0.1; the LVL0 notebook's decay-fit export), then age-adjusted. Practice stimuli are "
          "excluded from this fit only; they are segmented and scored like every other stimulus. The drift models are fitted "
          "to T2 and T3 together, as are LVL_0's Box-Cox transforms and age adjustment.")
show(0, "0f", "Table 0.6. Drift-model selection: mean R² of every candidate, per span", L0("model_selection"), nd=4,
     note="Mean over the five scales of the R² of each candidate fitted to the median RT per trial position (the LVL0 "
          "notebook's decay-fit export, LVL_0's selection rule); the highest mean in each span is the model used.")
_flow = DEMO("flow")
show(0, "0g", "Table 0.7. Sample flow: the dataset, the age exclusion and the analysed sample", _flow,
     note="demographic_report on the dataset and the pinned LVL_0 handoff (same participants and sessions, checked there).")
show(0, "0h", "Table 0.8. Sample characteristics by age group, condition (T2 / T3) and sex", DEMO("table"), nd=2,
     nd_cols={"n": 0},
     note="One row per participant of the analysed sample. Adolescents: 10–17 years. Depression and anxiety: T-scores. IBS = "
          "interval between sessions in hours among participants with both sessions: median and MAD (unscaled, median "
          "absolute deviation from the median).")
show(0, "0i", "Table 0.9. Interval between sessions by age group",
     DEMO("intervals").rename(columns={"Cohort": "Age group",
                                       "More than 7 days apart, % of cohort": "More than 7 days apart, % of age group"}), nd=2,
     nd_cols={"Participants": 0, "Both sessions": 0, "Same day, n": 0, "Different days, n": 0,
              "More than 7 days apart, n": 0},
     note="Same day = both sessions on the same calendar day; days apart = interval in hours / 24, among participants whose "
          "sessions were on different days; more than 7 days apart: share of the age group.")
show(0, "0j", "Table 0.10. Participants with one analysed session, by age group and recorded reason",
     DEMO("single_session_counts").rename(columns={"cohort": "age group"}),
     note="Reasons from the ledger of demographic_report (the dataset build's records; one entry per participant).")
show(0, "0k", "Table 0.11. Out-of-range responses in the analysed sample", DEMO("out_of_range"), nd=2,
     nd_cols={"Responses": 0, "Out of range, n": 0, "Participants with any": 0},
     note="Responses outside a scale's valid categories (the original task version accepted keys 1–9 on every scale). They "
          "stay in the data; LVL_1 sets them to missing for the rating-based features.")
_ac, _ag = L1R("age_continuous"), L1R("age_groups")
show(0, "0l", "Table 0.12. Age check: stimulus-level features before the age adjustment against age (continuous)",
     _ac[["family", "scale", "n", "slope", "slope_lo", "slope_hi", "r", "p", "p_FDR", "row"]], nd=4,
     nd_cols={"n": 0}, p_cols=["p", "p_FDR"],
     note="LVL1 notebook (descriptives). One value per participant (mean over sessions and stimuli). OLS slope per year of "
          "age with its 95% CI and Pearson r; p_FDR = BH over the five scales within each feature family; the composite rows "
          "are supplementary (uncorrected).")
show(0, "0m", "Table 0.13. Age check: adolescents (< 18) vs adults, stimulus-level features before the age adjustment",
     _ag.assign(**{"Δ median": _ag.median_adolescent - _ag.median_adult})[
         ["family", "scale", "n_adolescent", "n_adult", "median_adolescent", "median_adult", "Δ median",
          "mad_adolescent", "mad_adult", "U", "r_rb", "p", "p_FDR", "row"]], nd=3,
     nd_cols={"n_adolescent": 0, "n_adult": 0, "U": 1}, p_cols=["p", "p_FDR"],
     note="Mann–Whitney U, two-sided; r_rb = rank-biserial r (positive: adolescents higher); Δ median = adolescents − adults; "
          "MAD unscaled; p_FDR as Table 0.12.")
show_figure(0, "f0_1", "Figure 0.1. Response-time decay curves and log-RT, predicted and residual distributions",
            FIG_DIR / "main_text" / "fig_decay_curves.png",
            note="make_replication_figures from the LVL0 notebook's decay-fit export and the LVL_0 handoff. A–C: decay fits "
                 "(median log RT per trial position) for the concatenated sessions, S1 and S2, R² per scale; D, G, J: log "
                 "RTs; E, H, K: predicted values; F, I, L: residuals.")
show_figure(0, "f0_2", "Figure 0.2. RT, RT variance and response entropy by scale: session-level and sliding-window",
            FIG_DIR / "supplement" / "fig_feature_distributions.png",
            note="make_replication_figures from the LVL_0 handoff (raw RT) and the LVL1 notebook's session-feature "
                 "export, on LVL_0's trial set as Table 0.3 (the sliding-window entropy: the analysed stimuli).")
show_figure(0, "f0_3", "Figure 0.3. Age associations of the stimulus-level features before the age adjustment",
            FIG_DIR / "supplement" / "fig_age_by_feature.png",
            note="make_replication_figures from the LVL1 notebook's age-check exports (Tables 0.12–0.13).")

*[cell 5]*

## 1. LVL1 — calibration on all data

The LVL_1 sweep crossed `smoothing_window` with `pelt_jump` for every detector on all sessions (T2 and T3). Variance explained is the **SS share** of each knob in the criterion across the configurations of one detector: SS_knob / SS_total, with the smoothing × jump interaction as the remainder (one value per configuration, so the shares are descriptive and carry no test). `pelt_jump` acts only on the PELT detectors; for the others smoothing is the only active knob and its share is 1 by construction, so their range across smoothing values is reported instead.

In [ ]:
# [cell 6]
# ---- 1. LVL1 calibration: SS shares of the knobs and the selection frequencies -------------------------
_ss, _sf = S1("knob_ss"), S1("selection_freq")
_n_pelt = int(_ss.loc[_ss.Method.str.startswith("All PELT"), "Configs"].iloc[0])
_per_det = _ss[~_ss.Method.str.startswith("All PELT")].groupby("Method").Configs.first()
_n_t2 = int(S1("best_per_detector").n_obs.max())
_nv = S1("configs").n_validatable
_CRIT_NOTE = {"η²_H": f"η²_H = sweep mean over each detector's segmented sessions (T2 and T3; split into ≥ 2 blocks: "
                      f"{int(_nv.min())}–{int(_nv.max())} of the {_n_t2} sessions, by detector and configuration).",
              "Corr%": f"Corr% = sweep mean over all {_n_t2} sessions, T2 and T3 (an unsplit session scores 0)."}
for lab in ("η²_H", "Corr%"):
    show(1, f"1a_{lab}", f"Table 1.{1 if lab == 'η²_H' else 2}. Variance in the {lab} criterion explained by the "
         f"LVL_1 knobs, per detector (T2+T3 calibration sweep)", _ss[_ss.Criterion == lab].drop(columns="Criterion"),
         note=f"SS share = SS_knob / SS_total over the detector's configurations ({_per_det.max()} for PELT detectors, "
              f"{_per_det.min()} for the others); interaction = 1 − the two shares. Non-PELT detectors have one active knob "
              f"(smoothing), so their share is 1 by construction and only the criterion's range is shown. The pooled row "
              f"sums the within-detector SS of the PELT detectors. {_CRIT_NOTE[lab]} Descriptive.",
         nd=4)
show(1, "1b", "Table 1.3. Calibration selection frequencies: detectors whose best configuration has each smoothing_window value",
     _sf[_sf.Knob == "smoothing_window"].dropna(axis=1, how="all").drop(columns="Knob"),
     note=f"As Table 1.4, for smoothing_window over all {len(_per_det)} detectors.")
show(1, "1c", "Table 1.4. Calibration selection frequencies: PELT detectors whose best configuration has each pelt_jump value",
     _sf[_sf.Knob == "pelt_jump"].dropna(axis=1, how="all").drop(columns="Knob"),
     note="Best configuration = the configuration with the highest criterion for that detector; a detector whose best "
          "criterion value is reached at more than one value of the knob is counted as tied (exact equality). pelt_jump "
          "counts are over the PELT detectors. The configuration carried forward is the highest η²_H "
          f"({SEL1['detector']}, {SEL1['config']}: {SEL1['LVL_1_CONFIG']}), with Corr% as supporting evidence. Descriptive.")
show_figure(1, "f1_1", "Figure 1.1. LVL1 calibration (T2+T3 sweep) and two segmented sessions",
            FIG_DIR / "main_text" / "fig_lvl1_calibration.png",
            note="make_replication_figures from the LVL1 sweep report's exports, the LVL1 notebook's H1 exports and the "
                 "pinned handoff. A, B: sweep-mean η²_H at each smoothing window (A; the detectors with the highest "
                 "sweep-mean η²_H) and at each PELT jump (B; every PELT detector). C, D: each detector's mean η²_H against "
                 "its PorSig_adj (= Corr%) at the configuration optimised for η²_H (C) or for PorSig_adj (D); red = best, "
                 "orange = not separable from it (BH; Tables 2.4a and 2.5a), blue = significantly lower; numbers = rank in "
                 "C; grey dashed ellipse = the multivariate-outlier boundary, the critical D² of Wilks' test at α = .05 / n "
                 "(Bonferroni; Table 2.8). E–H, I–L: the three detection features and every detector's blocks for one T2 "
                 "and one T3 participant (red dashed line = session boundary).")

*[cell 7]*

## 2. LVL1 — H1 segmentation performance across the datasets

**Measures.** Per participant-session and detector: η²_H = (H − k + 1)/(n − k) of the Kruskal–Wallis test of each LVL_1 feature across the session's blocks (Cohen, 2008; the k − 1 term penalises over-segmentation), averaged over the three features; defined only for sessions split into ≥ 2 blocks. Corr% = share of the three features significant after BH across the detector's validatable sessions of the dataset; an unsplit session counts as not significant, so Corr% is over all sessions. Corr% is a descriptive detection rate, not a test of the session.

**Datasets.** T2+T3 calibration = each detector at its own η²_H-optimal configuration in the sweep on all data (in the Corr% test, at its own Corr%-optimal configuration). T3 only and the pooled datasets use the LVL_1 handoff (every detector at the selected configuration).

**Tests.** Four tests, each of all 120 pairs of the 16 detectors: η²_H and Corr% in T2+T3 calibration, and η²_H and Corr% in T3 only. Each pair: a two-sided participant-cluster paired bootstrap of the difference in means. Participants are resampled with replacement with all their sessions (B = 10,000; one set of resamples per dataset, shared by every detector), and each detector's mean is over its own sessions with a value (η²_H: the sessions it split; Corr%: all sessions), so every detector keeps all the sessions it segmented while the comparison stays paired within the same participants. 95% percentile interval; p = twice the smaller tail, (count + 1)/(B + 1), a tie counting in both; BH-FDR over the 120 pairs of each test.

In [ ]:
# [cell 8]
# ---- 2. H1: the LVL1 notebook's exports ----------------------------------------------------------------------
_d1, _dw, _dd, _tt, _sm, _rk = (H1X(k).loc[lambda d: d.Dataset.isin(H1_DS)].reset_index(drop=True)   # the H1 datasets
                                for k in ("desc_overall", "desc_winner", "desc_detector", "tests", "summary", "ranks"))
_nd = len(_dd.groupby("Detector(s)"))
_note = ("Units: session (detector × session values pooled for multi-detector rows). η²_H over segmented sessions only; "
         "Corr% = 100 × mean session score over all sessions; blocks over all sessions (1 = unsplit). M (SD) = mean and SD over "
         "the same values (SD with n − 1; Corr% in percentage points). † = the selected "
         "detector. T2+T3 calibration = each detector at its own η²_H-optimal sweep configuration, in every column "
         "(Corr% included; the Corr% test of Table 2.5a uses each detector's Corr%-optimal configuration); "
         + ("the other datasets" if POOLED_DS else "T3 only") + " = every detector at the selected configuration (the "
         "LVL_1 handoff)." + (" T3 only and Pooled, T3 share η²_H, segmentation and blocks; they differ in Corr% (BH "
                              "family)." if "Pooled, T3" in DS else ""))
show(2, "2a", "Table 2.1. H1 descriptives: all detectors pooled, per dataset", _d1, _note, nd=1)
show(2, "2b", "Table 2.2. H1 descriptives: selected detector vs the remaining detectors", _dw, _note, nd=1)
for i, ds in enumerate(H1_DS):
    show(2, f"2c_{i}", f"Table 2.3{'abcde'[i]}. H1 descriptives per detector — {ds} (ordered by mean η²_H)",
         _dd[_dd.Dataset == ds].drop(columns="Dataset"), _note, nd=1)
# the tests: every pair of the detectors in four tests (the LVL1 notebook's participant-cluster paired bootstrap)
from statsmodels.stats.multitest import fdrcorrection as _fdr
_bcr, _bet = S1("best_corr").set_index("method"), S1("best_eta").set_index("method")
_t2c = _dd[_dd.Dataset == "T2+T3 calibration"].assign(m=lambda d: d["Detector(s)"].str.removesuffix(" †")).set_index("m")
_dev0 = float(np.abs(_t2c["Corr% (all sessions)"] - _bet["avg_corr_sig%"].reindex(_t2c.index)).max(skipna=False))
check("H1 descriptives, T2+T3 calibration: each detector at its η²_H-optimal configuration (Corr% = the LVL1 sweep report's "
      "best-η²_H table)", len(_t2c) == len(_bet) and _dev0 < 1e-9, f"{len(_t2c)} detectors, max |diff| {_dev0:.1e}")
_t2 = _rk[_rk.Dataset == "T2+T3 calibration"].set_index(["Criterion", "Detector"]).Mean
_dev = max(float(np.abs(_t2["η²_H"] - _bet.avg_effsize.reindex(_t2["η²_H"].index)).max()),
           float(np.abs(_t2["Corr%"] - _bcr["avg_corr_sig%"].reindex(_t2["Corr%"].index)).max()))
check("H1 T2+T3 calibration tests: each detector's mean = the LVL1 sweep report's best-η²_H and best-Corr% tables", bool(_dev < 1e-9),
      f"max |diff| {_dev:.1e}")
_qd = max(float(np.abs(g.q.to_numpy() - _fdr(g.p.to_numpy())[1]).max()) for _, g in _tt.groupby(["Dataset", "Criterion"]))
check("H1 tests: every pair of the detectors in each test, q = BH over the test's pairs",
      bool(_tt.groupby(["Dataset", "Criterion"]).size().eq(_nd * (_nd - 1) // 2).all() and _qd < 1e-12), f"max |Δq| {_qd:.1e}")
_mk = lambda m: f"{m} †" if m == WINNER_LVL1 else m
for lab, num in (("η²_H", 4), ("Corr%", 5)):
    _unit = "η²_H" if lab == "η²_H" else "Corr% in percentage points"
    _own = "the sessions it split" if lab == "η²_H" else "all sessions, an unsplit session scoring 0"
    for j, ds in enumerate(("T2+T3 calibration", "T3 only")):
        k = _rk[(_rk.Dataset == ds) & (_rk.Criterion == lab)]
        t = _tt[(_tt.Dataset == ds) & (_tt.Criterion == lab)]
        _cfg = k.Configurations.iloc[0]
        show(2, f"2d_{lab}_{j}", f"Table 2.{num}{'ac'[j]}. H1, {lab}: each detector — {ds}, {_cfg}",
             k.assign(Detector=k.Detector.map(_mk))[["Rank", "Detector", "Mean", "SD", "Sessions", "Significantly above (n)",
                                                     "Significantly below (n)", "Not separable from (q > .05)", "Group"]],
             note=f"{_unit}; mean and SD (n − 1) over the detector's own sessions with a value ({_own}). Significantly "
                  f"above/below = the number of detectors it differs from at q ≤ .05 in Table 2.{num}{'bd'[j]}; group = best, "
                  "not separable from the best (q > .05), or lower" + (" (the colours of Figure 1.1C–D)." if j == 0 else "."),
             nd=4 if lab == "η²_H" else 2,
             nd_cols={"Rank": 0, "Sessions": 0, "Significantly above (n)": 0, "Significantly below (n)": 0})
        show(2, f"2d_{lab}_{j}_pairs", f"Table 2.{num}{'bd'[j]}. H1, {lab}: all {len(t)} detector pairs — {ds}, {_cfg}",
             t.assign(A=t.A.map(_mk), B=t.B.map(_mk))[["A", "B", "Mean A", "Mean B", "Difference (A − B)", "CI 2.5%",
                                                       "CI 97.5%", "p", "q"]],
             note=f"A = the detector with the higher mean; {_unit}. Two-sided participant-cluster paired bootstrap of the "
                  "difference in means: participants resampled with replacement with all their sessions (B = "
                  f"{int(t.Replicates.max()):,}; one set of resamples per dataset, shared by every detector), each detector's "
                  "mean over its own sessions with a value; 95% percentile interval; p = twice the smaller tail, (count + 1)/(B "
                  f"+ 1), a tie counting in both; q = BH-FDR over the {len(t)} pairs of this test.",
             nd=4 if lab == "η²_H" else 2)
# the selected LVL_1 configuration vs each contender's own optimum (LVL1 sweep), and the winner's margin
_cf = S1("configs"); _sel1 = np.ones(len(_cf), bool)
for _k1, _v1 in SEL1["LVL_1_CONFIG"].items():
    _sel1 &= (_cf[f"cfg_{_k1}"].isna() | (_cf[f"cfg_{_k1}"] == _v1)).to_numpy()
_best1 = _cf.groupby("method").avg_effsize.max(); _at1 = _cf[_sel1].groupby("method").avg_effsize.max()
_cont1 = _best1[_best1 >= _best1.max() - .01].index
H1_CONTENDER_LOSS = float((_best1[_cont1] - _at1[_cont1]).max())
_b1 = np.sort(_best1.to_numpy())[::-1]; H1_T2_MARGIN = float(_b1[0] - _b1[1])
show(2, "2e", "Table 2.6. H1 across the datasets: numerical winner and statistically supported superiority per test",
     _sm.drop(columns="Configurations"), nd=4, nd_cols={"Participants": 0, "Sessions": 0, "† rank": 0, "Significantly below †": 0},
     note="One row per test (Tables 2.4–2.5). Means: η²_H over segmented sessions; Corr% in percentage points over all "
          "sessions. 'Not separable' = q > .05; 'significantly below/above †' = q ≤ .05 with the selected detector "
          "higher/lower. Numerical rank is not statistical superiority: the selected detector is the η²_H winner wherever no "
          "detector is significantly above it. T2+T3 calibration rows use each detector's own η²_H-optimal (η²_H) or "
          "Corr%-optimal (Corr%) configuration in the LVL1 sweep; T3 only runs every detector at the selected configuration (the "
          f"LVL_1 handoff's), where the {len(_cont1)} detectors within .01 of the winner's η²_H lose at most "
          f"{H1_CONTENDER_LOSS:.4f} of their own optimum (the winner's margin over the runner-up: {H1_T2_MARGIN:.4f}). η²_H "
          "and Corr% measure how well the blocks separate on the features the detectors segment, so they rank the detectors "
          "on those features rather than validate them against an outside criterion.")
# Method level, per analysis: the detectors' mean η²_H and mean Corr% (one point per detector), their Pearson r, and the
# Mahalanobis D² of (mean η²_H, mean Corr%) from the detectors' centroid (sample covariance). T2+T3 calibration = each
# detector at its η²_H-optimal configuration (Figure 1.1C); T2+T3 calibration, Corr%-optimal = at its Corr%-optimal
# configuration (Figure 1.1D, post hoc); both from the LVL1 sweep report. The other datasets = the LVL_1 handoff
# (η²_H and blocks from its per-session results, Corr% from the LVL1 notebook's per-detector table). Outlier
# test: the exact distribution of a sample Mahalanobis distance, n D² / (n − 1)² ~ Beta(p / 2, (n − p − 1) / 2) with
# p = 2 (Wilks, 1963), BH over the detectors of each analysis.
_EFF = ["adj_rt_level_stim_effsize", "adj_rt_var_win_effsize", "entropy_win_effsize"]
_res = pd.read_csv(f"{HANDOFF}_results.csv")
_res["condition"] = _res["id"].map(pd.read_csv(f"{HANDOFF}_features.csv", usecols=["id", "condition"])
                                   .drop_duplicates().set_index("id")["condition"])
_res["eta"] = _res[_EFF].mean(axis=1, skipna=False)
_mark = lambda m: f"{m} †" if m == WINNER_LVL1 else m
_ddm = _dd.assign(method=_dd["Detector(s)"].str.removesuffix(" †"))
MLA = {}
for _lab, _f in (("T2+T3 calibration", "best_eta"), ("T2+T3 calibration, Corr%-optimal", "best_corr")):
    _b = pd.read_csv(LVL_OUTS / f"{LVL1_SWEEP}_report_{_f}.csv").set_index("method")
    MLA[_lab] = pd.DataFrame({"eta": _b.avg_effsize, "corr": _b["avg_corr_sig%"], "blocks": _b.blocks_mean})
for _ds, _cs in {ds: d[1] for ds, d in DATASETS.items() if ds != "T2 only"}.items():
    _g = _res[_res.condition.isin(_cs)].groupby("method")
    MLA[_ds] = pd.DataFrame({"eta": _g.eta.mean(), "blocks": _g.n_blocks.mean(),
                             "corr": _ddm[_ddm.Dataset == _ds].set_index("method")["Corr% (all sessions)"]})


def _maha(X):
    X = np.asarray(X, float); dX = X - X.mean(axis=0)
    return np.einsum("ij,jk,ik->i", dX, np.linalg.inv(np.cov(X.T)), dX)


_mao, _mho = H1X("method_association").set_index("Dataset"), H1X("mahalanobis")
_mao = _mao[_mao.index.isin(H1_DS)]
_mho = _mho.set_index(_mho.Detector.str.removesuffix(" †"))
_dev = max(max(abs(stats.pearsonr(MLA[d].eta, MLA[d]["corr"])[0] - _mao.loc[d, "r (mean η²_H, mean Corr%)"]),
               float(np.abs(_maha(MLA[d][["eta", "blocks"]]) - _mho[d].reindex(MLA[d].index)).max())) for d in _mao.index)
check("H1 method-level means = the LVL1 notebook's (its r and its D² of mean η²_H and mean blocks reproduced)", _dev < 1e-9,
      f"max |diff| {_dev:.1e}")
_ma, _md = [], {}
for _lab, _m in MLA.items():
    _n = len(_m); _r, _p = stats.pearsonr(_m.eta, _m["corr"]); _D = _maha(_m[["eta", "corr"]])
    _q = _fdr(stats.beta.sf(_n * _D / (_n - 1) ** 2, 1.0, (_n - 3) / 2.0))[1]; _i = int(np.argmax(_D))
    _ma.append({"Analysis": _lab, "Detectors": _n, "r (mean η²_H, mean Corr%)": _r, "df": _n - 2, "p": _p,
                "Max D² (mean η²_H, mean Corr%)": _D[_i], "Detector with max D²": _mark(_m.index[_i]),
                "Critical D², α = .05 (Wilks)": (_n - 1) ** 2 / _n * stats.beta.isf(.05 / _n, 1.0, (_n - 3) / 2.0),
                "Max D²: q (exact, BH)": _q[_i], "Detectors with q < .05": int((_q < .05).sum()),
                "Largest attainable D²": (_n - 1) ** 2 / _n})
    _md[_lab] = pd.Series(_D, index=[_mark(x) for x in _m.index])
show(2, "2g", "Table 2.8. H1 method-level association: mean η²_H vs mean Corr%, and multivariate outliers",
     pd.DataFrame(_ma), nd_cols={"Detectors": 0, "df": 0, "Detectors with q < .05": 0},
     note="One point per detector: its mean η²_H (segmented sessions) and mean Corr% (all sessions). T2+T3 calibration: each "
          "detector at its η²_H-optimal configuration (Figure 1.1C; as Table 2.6); Corr%-optimal: at its Corr%-optimal "
          "configuration (Figure 1.1D, post hoc); T3 only" + (" and pooled" if POOLED_DS else "") + ": every detector at the "
          "selected configuration. r = "
          "Pearson correlation across the detectors (one test per analysis). D² = Mahalanobis distance of (mean η²_H, mean "
          "Corr%) from the detectors' centroid with the sample covariance; with n detectors it cannot exceed (n − 1)² / n. "
          "Outlier test: each detector's p from the exact distribution of a sample Mahalanobis distance, n D² / (n − 1)² ~ "
          "Beta(1, (n − 3) / 2) (Wilks, 1963), BH over the detectors of the analysis; q is the BH q of the largest D² "
          "(the smallest p), at most n p, its Bonferroni p. Critical D² = (n − 1)² / n times the Beta quantile at "
          "1 − .05 / n, the D² whose p is .05 / n: the Bonferroni boundary, which is BH's threshold for the most extreme "
          "detector (BH can also flag a detector inside it when several p values are small); it is drawn as the grey "
          "dashed ellipse of Figure 1.1C–D. This report computes the analysis; the LVL1 notebook's Mahalanobis "
          "export (D² of mean η²_H and mean blocks) is only reproduced as a check (Table 7.3). Table 2.9 gives every "
          "detector's D².")
show(2, "2h", "Table 2.9. Mahalanobis D² of each detector (mean η²_H, mean Corr%), per analysis",
     pd.DataFrame(_md).rename_axis("Detector").reset_index(), nd=2, note="As Table 2.8.")
# Tables 2.10a–d: every detector pair of each test (Tables 2.4b, 2.4d, 2.5b, 2.5d) as a lower-triangle matrix, the
# supplementary H1 tables ([cell 35] writes them as LaTeX). Cell = Δ (q): Δ = the column detector's mean minus the row
# detector's, with the stars of the pair's q (printed by h1_q: three decimals, more where three would round up to .050
# or .010). Detector 1 = the selected detector (η²_H) or the best detector (Corr%); the others in descending p against
# it (ties: the smaller absolute difference in means first). Names in the figures' notation (make_replication_figures
# [cell 3]); bold = the test's best detector and the detectors not separable from it (Tables 2.4a, 2.4c, 2.5a, 2.5c).
FEATURE_TEX = {"Adj_RT_Level_Stim": "$AdjRT_{stim}$", "Adj_RT_Var_win": "$AdjVar_{stim}$",
               "Entropy_win": "$Ent_{stim}$"}


def method_label(m):
    """PELT_Adj_RT_Var_win -> 'PELT $AdjVar_{stim}$'; HMM_Attention_comp_3 -> 'HMM-3$_{comp}$'."""
    fam, rest = m.split("_", 1)
    states = ""
    if fam == "HMM":
        _m = re.match(r"(\d+)_(.*)", rest) or re.match(r"(.*)_(\d+)$", rest)
        if _m and _m.group(1).isdigit():
            states, rest = _m.group(1), _m.group(2)
        elif _m:
            rest, states = _m.group(1), _m.group(2)
    name = f"{fam}{'-' + states if states else ''}"
    return f"{name}$_{{comp}}$" if rest == "Attention_comp" else f"{name} {FEATURE_TEX.get(rest, rest)}"


H1_BOLD = ("best", "not separable from the best")           # the groups printed in bold
h1_stars = lambda q: "***" if q < .001 else "**" if q < .01 else "*" if q < .05 else ""


def h1_delta(v, crit, minus="-"):
    """Δ as printed: η²_H to 3 decimals without the leading zero, Corr% (percentage points) to 1 decimal; a value that
    rounds to zero carries no sign."""
    s = f"{abs(v):.3f}".lstrip("0") if crit == "η²_H" else f"{abs(v):.1f}"
    return (minus if v < 0 and float(s) != 0 else "") + s


def h1_q(v):
    """q as the H1 matrices print it: fmt_p, except where three decimals round a q below .050, .010 or .001 up to the
    boundary: then four decimals, or more until the rounded value is below it, so the printed q and its stars agree."""
    s = fmt_p(v)
    if s in (".050", ".010", ".001") and v < float(s):
        nd = 4
        while float(f"{v:.{nd}f}") >= float(s):
            nd += 1
        s = f"{v:.{nd}f}".replace("0.", ".", 1)
    return s


H1_MATS = {}       # letter -> (criterion, dataset, configurations, detector order, q matrix, Δ matrix, groups, string table, B)
for _x, (lab, num, j, ds) in zip("abcd", [(l, n, j, d) for l, n in (("η²_H", 4), ("Corr%", 5))
                                          for j, d in enumerate(("T2+T3 calibration", "T3 only"))]):
    t = _tt[(_tt.Dataset == ds) & (_tt.Criterion == lab)]
    _grp = _rk[(_rk.Dataset == ds) & (_rk.Criterion == lab)].set_index("Detector").Group
    _anc = WINNER_LVL1 if lab == "η²_H" else _grp[_grp == "best"].index[0]      # detector 1: selected (η²_H), best (Corr%)
    _vs = t[(t.A == _anc) | (t.B == _anc)]
    _order = [_anc] + (pd.DataFrame({"m": np.where(_vs.A == _anc, _vs.B, _vs.A), "p": _vs.p,
                                     "d": _vs["Difference (A − B)"].abs()})
                       .sort_values(["p", "d", "m"], ascending=[False, True, True]).m.tolist())
    _pos = {m: i for i, m in enumerate(_order)}
    _n = len(_order)
    _M = pd.DataFrame(np.nan, index=_order, columns=_order)
    _D = _M.copy()
    for _a, _b, _dif, _q in t[["A", "B", "Difference (A − B)", "q"]].itertuples(index=False):
        _r, _c = max(_a, _b, key=_pos.get), min(_a, _b, key=_pos.get)
        _M.loc[_r, _c] = _q
        _D.loc[_r, _c] = _dif if _a == _c else -_dif          # the column detector's mean minus the row detector's
    _tab = pd.DataFrame({"Detector": [f"{i + 1}. **{method_label(m)}**" if _grp[m] in H1_BOLD else
                                      f"{i + 1}. {method_label(m)}" for i, m in enumerate(_order)],
                         **{str(c + 1): ["—" if r == c else " " if r < c else
                                         f"{h1_delta(_D.iat[r, c], lab)}{h1_stars(_M.iat[r, c])} ({h1_q(_M.iat[r, c])})"
                                         for r in range(_n)] for c in range(_n)}})
    _names = lambda g: ", ".join(method_label(m) for m in _order if _grp[m] == g) or "none"
    _B = int(t.Replicates.max())
    _sdf = show(2, f"2i_{_x}", f"Table 2.10{_x}. H1, {lab}: difference in means and q of all {len(t)} detector pairs as a "
                f"matrix — {ds}, {t.Configurations.iloc[0]}", _tab,
                note=f"Cells: Δ (q) of the row and the column detector: Δ = the column detector's mean {lab} minus the row "
                     f"detector's" + (" (percentage points)" if lab == "Corr%" else "") + "; "
                     f"q = its BH-adjusted p (Table 2.{num}{'bd'[j]}: two-sided participant-cluster paired bootstrap of the "
                     f"difference in means, B = {_B:,}; BH-FDR over the {len(t)} pairs of the {_n} detectors of this test). "
                     "\\* q < .05, \\*\\* q < .01, \\*\\*\\* q < .001; — = the detector itself. q to three decimals; "
                     "four where three would round up to .050 or .010. "
                     + ("Detector 1 = the selected detector; the others are ordered by descending p against it (ties: "
                        "the smaller absolute difference in means first)." if lab == "η²_H" else
                        f"Detector 1 = the best detector (highest mean {lab}); the others are ordered by descending p "
                        "against it (ties: the smaller absolute difference in means first); the selected detector, "
                        f"{method_label(WINNER_LVL1)}, is detector {_order.index(WINNER_LVL1) + 1}.")
                     + f" Bold = the best detector of Table 2.{num}{'ac'[j]} ({_names('best')}) and the detectors "
                     f"not separable from it, q > .05 ({_names('not separable from the best')}).")
    H1_MATS[_x] = (lab, ds, t.Configurations.iloc[0], _order, _M, _D, _grp, _sdf, _B)
# each matrix against the exports, read again: every pair once in the lower triangle, its printed Δ (column minus row
# mean, read back from the cell) within rounding of the export's signed difference, its stars those of q and of the
# printed q, its q as h1_q prints it (numeric matrices included); the diagonal and the upper triangle empty; bold = the
# best and the not-separable group; detector 1 = the selected detector (η²_H) or the best detector (Corr%) and the
# others in descending p against it
_tt0, _rk0 = H1X("tests"), H1X("ranks")
_h1rx = re.compile(r"^(-?\d*\.\d+)(\**) \((< \.001|\.\d{3,}|1\.000)\)$")
_h1nst = lambda s: 3 if s == "< .001" else int(float(s) < .05) + int(float(s) < .01) + int(float(s) < .001)   # stars of a printed q
for _x, (lab, ds, _, _order, _M, _D, _grp, _sdf, _) in H1_MATS.items():
    _t0 = _tt0[(_tt0.Dataset == ds) & (_tt0.Criterion == lab)]
    _g0 = _rk0[(_rk0.Dataset == ds) & (_rk0.Criterion == lab)].set_index("Detector").Group
    _anc = WINNER_LVL1 if lab == "η²_H" else _g0[_g0 == "best"].index[0]
    _pos = {m: i for i, m in enumerate(_order)}
    _n, _dp = len(_order), 3 if lab == "η²_H" else 1
    _S = _sdf.drop(columns="Detector").to_numpy()
    _low = np.tril(np.ones((_n, _n), bool), -1)
    _rc, _cells = [], []
    for a, b, dif, q in _t0[["A", "B", "Difference (A − B)", "q"]].itertuples(index=False):
        r, c = max(_pos[a], _pos[b]), min(_pos[a], _pos[b])
        dl = dif if _pos[a] < _pos[b] else -dif
        _rc.append((r, c))
        _mt = _h1rx.match(_S[r, c])
        _cells.append(bool(_mt) and abs(float(_mt.group(1)) - dl) <= .5 * 10 ** -_dp + 1e-12
                      and len(_mt.group(2)) == int(q < .05) + int(q < .01) + int(q < .001) == _h1nst(_mt.group(3))
                      and _mt.group(3) == h1_q(q)
                      and _M.iat[r, c] == q and _D.iat[r, c] == dl)
    _pv = [float(_t0.p[((_t0.A == _anc) & (_t0.B == m)) | ((_t0.B == _anc) & (_t0.A == m))].iloc[0])
           for m in _order[1:]]
    _ok = (len(set(_rc)) == len(_t0) == int(_low.sum()) == int(np.isfinite(_M.to_numpy()).sum())
           == int(np.isfinite(_D.to_numpy()).sum()) and all(_cells)
           and bool((_S[~_low] == np.where(np.eye(_n, dtype=bool), "—", " ")[~_low]).all())
           and sorted(_order) == sorted(_g0.index)
           and all(s == (f"{i + 1}. **{method_label(m)}**" if _g0[m] in ("best", "not separable from the best")
                         else f"{i + 1}. {method_label(m)}") for i, (m, s) in enumerate(zip(_order, _sdf.Detector)))
           and _order[0] == _anc and bool(np.all(np.diff(_pv) <= 0)))
    _dl = _D.to_numpy()[_low]
    check(f"H1 Table 2.10{_x} ({lab}, {ds}): its lower triangle = the export's Δ (column minus row mean) and q of each of "
          f"the {len(_t0)} pairs, once, with the stars of q, the printed q agreeing with them (three decimals; four where "
          "three would round up to .050 or .010); bold = the best and the not-separable detectors; detector 1 = the "
          + ("selected" if lab == "η²_H" else "best") + " detector, the others in descending p against it", _ok,
          f"{_n} detectors, {sum(_cells)}/{len(_cells)} cells; {int((_g0.isin(('best', 'not separable from the best'))).sum())} "
          f"bold; Δ {h1_delta(_dl.min(), lab)} to {h1_delta(_dl.max(), lab)}; p against detector 1 {fmt_p(max(_pv))} to "
          f"{fmt_p(min(_pv))}")

In [ ]:
# [cell 9]
# ---- the exact-zero folded units: read-only rebuild of every run's block matrices, and their diagnostics -------------
# The fold of the RT-variance and entropy features (lvl2.py add_exp_features) is |MAD-z of the Box-Cox value| ^ exponent,
# with the median and MAD taken over every unit of the matrix. With an odd number of units the median is one unit's own
# value, so that unit's |MAD-z| is exactly 0 and its folded value is 0 for any exponent; an exponent below 1 maps every
# other unit close to 1 (|z|^p ≈ 1 + p·ln|z|), so after standardisation the median unit lies far below all others. The runs
# export only the dynamic matrix (_units.csv). The control and session matrices, and refits of the carried labellers with
# the exact-zero value replaced by the feature's next-smallest |MAD-z|, need the matrices themselves: a subprocess executes
# lvl2.py on the run's own inputs through Step 5 (read-only: without --export it writes no file), executes Step 5 again on
# copies of the matrices with the value(s) replaced, recomputes the first refits of the stability stage, and passes the
# matrices and flags back through a pipe. CPU: taskset -c 0-30, 31 workers, one thread each. Tables 4.30-4.32.
import pickle, subprocess, sys
_P1_RUNNER = r'''
import os, sys, pickle
_out = os.fdopen(os.dup(1), "wb"); os.dup2(2, 1)          # lvl2.py prints to stderr; the result goes to stdout
script, arms, n_chk = sys.argv[1], sys.argv[2].split(","), int(sys.argv[3])
sys.argv = ["lvl2.py"] + sys.argv[4:]
src = open(script).read()
C42 = "# ============================ NOTEBOOK CELL 42 ============================"
CUT = "block_matrix = block_matrices.get(ANALYSIS_METHOD, block_matrix)"
assert src.count(C42) == 1 and src.count(CUT) == 1, "lvl2.py changed: Step 5 cut points not found"
i42, icut = src.index(C42), src.index(CUT)
G = {"__name__": "__main__", "__file__": script}
exec(compile(src[:i42], script, "exec"), G)                 # Steps 1-4: the block matrices, folded
STEP5 = compile("\n" * src[:i42].count("\n") + src[i42:icut], script, "exec")   # Step 5 as the run executes it
import numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
cfg = G["LVL_2_CONFIG"]
_rule = "speed" if "speed" in G["WCI_RULES"] else G["WCI_RULES"][0]
ARMS = {a: G["rule_arm"](a, _rule) for a in arms}
EXP = {"rt_var": ("kmeans_exp_rt_var_exponent", "mixture_gmm_exp_rt_var_exponent"),
       "entropy": ("kmeans_exp_entropy_exponent", "mixture_gmm_exp_entropy_exponent")}
BASE = {m: bm.copy() for m, bm in G["block_matrices"].items()}
COLS = ["id", "session", "block", "adj_rt_level_block"] + [f"{a}_{s}" for a in EXP for s in ("block", "centered", "abs")] \
       + [f"exp_{a}_block{g}" for a in EXP for g in ("", "_gmm")]


def power(a, arm):
    p, pg = float(cfg[EXP[arm][0]]), cfg[EXP[arm][1]]
    return a ** p, a ** (p if pg is None else float(pg))


def replaced(bm, feats):
    """The matrix with each exact-zero |MAD-z| of `feats` set to that feature's smallest non-zero |MAD-z|."""
    bm = bm.copy()
    for arm in feats:
        a = bm[f"{arm}_abs"].to_numpy(dtype=float, copy=True)
        a[a == 0] = a[np.isfinite(a) & (a > 0)].min()
        bm[f"{arm}_abs"] = a
        bm[f"exp_{arm}_block"], bm[f"exp_{arm}_block_gmm"] = power(bm[f"{arm}_abs"], arm)
    return bm


def flags():
    out = {}
    for m in G["block_matrices"]:
        for a, arm in list(ARMS.items()) + [("inattent_behav", "inattent_behav")]:
            bb = G["bad_blocks_by_method"].get(m, {}).get(arm)
            r = {"flagged": [] if bb is None or not len(bb) else bb[["id", "session", "block"]].astype(int).values.tolist()}
            g = G["gmm_results_by_method"].get(m, {}).get(arm)
            if g is not None:
                s = g["selection_df"]
                r.update(weights=g["gmm"].weights_.tolist(), inattent_idx=int(g["inattent_idx"]),
                         component=s[["id", "session", "block", "_component"]].astype(int).values.tolist())
            out[(m, a)] = r
    return out


# the fold formula, as add_exp_features applies it
fold_ok = all(np.array_equal(v, bm[f"exp_{arm}_block{g}"].to_numpy())
              for bm in BASE.values() for arm in EXP for g, v in zip(("", "_gmm"), power(bm[f"{arm}_abs"], arm)))
exec(STEP5, G)
RES = {"as run": flags()}
for var, feats in (("none", ()), ("entropy", ("entropy",)), ("rt_var", ("rt_var",)), ("both", ("rt_var", "entropy"))):
    G["block_matrices"] = {m: replaced(bm, feats) for m, bm in BASE.items()}
    G["cluster_results"] = None                      # every k-means refitted on the matrix it labels
    exec(STEP5, G)
    RES[var] = flags()

# the first refits of the stability stage (lvl2.py Step 5e: seeds, resamples and settings)
seed = int(cfg["random_seed"]); GA = (cfg["mixture_gmm_covariance_type"], int(cfg["mixture_gmm_n_init"]),
                                      int(cfg["mixture_gmm_max_iter"]))
bm = BASE[G["ANALYSIS_METHOD"]]


def _refit(fam, k, Xs, ref, b):
    idx = np.random.RandomState(seed + b).choice(len(Xs), size=len(Xs), replace=True)
    if fam == "cluster":
        lab = KMeans(n_clusters=k, random_state=seed + b, n_init=10).fit_predict(Xs[idx])
    else:
        lab = GaussianMixture(n_components=k, covariance_type=GA[0], random_state=seed + b, n_init=GA[1],
                              max_iter=GA[2]).fit(Xs[idx]).predict(Xs[idx])
    return adjusted_rand_score(ref[idx], lab)


REFITS = {}
for a in arms:
    fam, k = a.rsplit("_", 1); k = int(k)
    if fam == "cluster":
        Xs = StandardScaler().fit_transform(bm[G["CLUSTER_FEATURE_COLS"]].dropna())
        ref = KMeans(n_clusters=k, random_state=seed, n_init=10).fit_predict(Xs)
    else:
        X = bm[G["GMM_FEATURE_COLS"]].dropna().to_numpy(float); sd = X.std(axis=0, ddof=1); sd[sd == 0] = 1.0
        Xs = (X - X.mean(axis=0)) / sd
        ref = GaussianMixture(n_components=k, covariance_type=GA[0], random_state=seed, n_init=GA[1],
                              max_iter=GA[2]).fit(Xs).predict(Xs)
    REFITS[a] = Parallel(n_jobs=G["N_JOBS"])(delayed(_refit)(fam, k, Xs, ref, b) for b in range(n_chk))
pickle.dump({"matrices": {m: bm_[[c for c in COLS if c in bm_.columns]].copy() for m, bm_ in BASE.items()},
             "fits": RES, "refits": REFITS, "fold_formula": fold_ok, "config": dict(cfg),
             "methods": (G["ANALYSIS_METHOD"], G["CONTROL_METHOD"])}, _out)
_out.close()
'''
P1_ARMS = {lab: arm for lab, (_, arm) in PICKS.items()}               # carried solution -> arm
P1_FEAT = {"entropy": "Entropy", "rt_var": "RT variance"}
P1_MIN = int(RUNLOG["T2"]["parameters"]["irr_min_units"])             # the H3 eligibility rule (minority profile)
P1_NCHK = 62                                                          # stability refits recomputed per carried fit
P1_VARS = {"entropy": "Entropy zero replaced", "rt_var": "RT-variance zero replaced", "both": "Both replaced"}
_P1_ENV = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
           "OPENBLAS_NUM_THREADS": "1", "MPLBACKEND": "Agg"}
P1_RB = {}
for _run in RUN:
    _t0 = time.time()
    _pr = subprocess.run(["taskset", "-c", "0-30", sys.executable, "-B", "-c", _P1_RUNNER, str(CODE_DIR / "lvl2.py"),
                          ",".join(P1_ARMS.values()), str(P1_NCHK), "--lvl1-input", HANDOFF.name, "--condition", _run,
                          "--conditions", str(list(RUNLOG[_run]["parameters"]["conditions"])), "--n-jobs", "31",
                          "--kmeans_bootstrap_reps", "0"], cwd=str(CODE_DIR), env=_P1_ENV, capture_output=True)
    if _pr.returncode != 0:
        raise SystemExit(f"read-only rebuild of the {_run} run failed:\n{_pr.stderr.decode(errors='replace')[-3000:]}")
    P1_RB[_run] = pickle.loads(_pr.stdout)
    print(f"rebuild of the {_run} run (lvl2.py to its Step 5 cut, read-only): {time.time() - _t0:.0f} s")

_p1_src = lambda run: dict(zip(("dynamic", "control", "session"), (*P1_RB[run]["methods"], "participant")))
_p1_key = lambda des, i, s, b: f"{int(i)}/{int(s)}" if des == "session" else f"{int(i)}/{int(s)}/{int(b)}"
_p1_keys = lambda des, recs: {_p1_key(des, *r[:3]) for r in recs}
_p1_gmm = lambda arm: arm.startswith("mixture_gmm")
_p1_unit = {"dynamic": "block", "control": "block", "session": "session", "participant": "participant"}


def _p1_exp(run, lab, feat):
    """The exponent of `feat` in solution `lab` (the run's configuration; GMM None = the k-means exponent)."""
    c = RUNLOG[run]["parameters"]; pk = float(c[f"kmeans_exp_{feat}_exponent"])
    pg = c.get(f"mixture_gmm_exp_{feat}_exponent")
    return (pk if pg is None else float(pg)) if _p1_gmm(P1_ARMS[lab]) else pk


def _p1_z(bm, lab, feat):
    """The feature as the solution's labeller standardises it: k-means StandardScaler (SD ddof 0), GMM SD ddof 1."""
    v = bm[f"exp_{feat}_block" + ("_gmm" if _p1_gmm(P1_ARMS[lab]) else "")].to_numpy(float)
    sd = v.std(ddof=1 if _p1_gmm(P1_ARMS[lab]) else 0)
    return (v - v.mean()) / (sd if sd > 0 else 1.0)


P1_M = {(run, des): R["matrices"][src].reset_index(drop=True) for run, R in P1_RB.items() for des, src in _p1_src(run).items()}
P1_ZI = {(run, des, f): np.flatnonzero(bm[f"{f}_abs"].to_numpy() == 0) for (run, des), bm in P1_M.items() for f in P1_FEAT}
P1_ZK = {k: [_p1_key(k[1], *P1_M[k[:2]].loc[i, ["id", "session", "block"]]) for i in v] for k, v in P1_ZI.items()}

# Table 4.30: every exact-zero unit and its standardised position in each carried labeller's input
rows = []
for (run, des), bm in P1_M.items():
    n = len(bm)
    for f, flab in P1_FEAT.items():
        zi = P1_ZI[(run, des, f)]
        for lab in P1_ARMS:
            z = _p1_z(bm, lab, f); zo = np.delete(z, zi)
            rows.append({"Dataset": RUN_DATASET[run], "Design": des, "Units": n, "Feature": flab, "Solution": lab,
                         "Exponent": _p1_exp(run, lab, f),
                         "Exact-zero unit": ", ".join(P1_ZK[(run, des, f)]) or f"none ({n} units, {'odd' if n % 2 else 'even'})",
                         "z of the zero unit": z[zi].max() if len(zi) else np.nan,
                         "z of the next unit": zo.min() if len(zi) else np.nan,
                         "Gap (SD)": zo.min() - z[zi].max() if len(zi) else np.nan})
P1_ZERO = pd.DataFrame(rows)
_p1g = P1_ZERO[(P1_ZERO.Exponent > 1) & P1_ZERO["Gap (SD)"].notna()]["Gap (SD)"]
P1_GAP_HI = float(_p1g.max()) if len(_p1g) else np.nan          # largest gap at an exponent above 1

# Table 4.31: flags of every carried fit as run and with the exact-zero value(s) replaced (lvl2.py's own Step 5)
_p1_dsc = lambda a, b: 2 * len(a & b) / (len(a) + len(b)) if (a or b) else np.nan
rows, P1_SETS = [], {}
for run, R in P1_RB.items():
    F, ds = R["fits"], RUN_DATASET[run]
    for des, src in _p1_src(run).items():
        bm = P1_M[(run, des)]
        for lab, arm in P1_ARMS.items():
            S = {v: _p1_keys(des, F[v][(src, arm)]["flagged"]) for v in F}
            H = _p1_keys(des, F["as run"][(src, "inattent_behav")]["flagged"])
            designs = [(des, len(bm), S, H)]
            if des == "session":                          # the participant design: participants with a flagged session
                pid = lambda s: {k.split("/")[0] for k in s}
                designs.append(("participant", bm.id.nunique(), {v: pid(s) for v, s in S.items()}, pid(H)))
            for d_, n, S_, H_ in designs:
                P1_SETS[(run, d_, lab)] = S_
                r = {"Dataset": ds, "Design": d_, "Solution": lab, "Units": n, "Flagged (as run)": len(S_["as run"]),
                     "Minority (as run)": min(len(S_["as run"]), n - len(S_["as run"]))}
                r.update({P1_VARS[v]: len(S_[v]) for v in P1_VARS})
                r["Minority (both replaced)"] = min(len(S_["both"]), n - len(S_["both"]))
                r["DSC with the heuristic (as run)"] = _p1_dsc(S_["as run"], H_)
                r["DSC with the heuristic (both replaced)"] = _p1_dsc(S_["both"], H_)
                r["DSC as run vs both replaced"] = _p1_dsc(S_["as run"], S_["both"])
                r["_H"] = len(H_); r["_Hboth"] = len(S_["as run"] & H_)
                rows.append(r)
P1_CF = pd.DataFrame(rows)


def _p1_lone(run, des, lab):
    """The minority side of a carried fit: its units, which exact-zero unit each is, and its component weight (GMM)."""
    bm, src, arm = P1_M[(run, des)], _p1_src(run)[des], P1_ARMS[lab]
    keys = [_p1_key(des, *r) for r in bm[["id", "session", "block"]].values]
    fl = P1_SETS[(run, des, lab)]["as run"]
    side = fl if len(fl) <= len(keys) - len(fl) else set(keys) - fl
    fit = P1_RB[run]["fits"]["as run"][(src, arm)]
    comp = {_p1_key(des, *c[:3]): c[3] for c in fit.get("component", [])}
    out = []
    for k in sorted(side):
        i = keys.index(k)
        zero = [P1_FEAT[f] for f in P1_FEAT if i in P1_ZI[(run, des, f)]]
        zs = {f: _p1_z(bm, lab, f) for f in P1_FEAT}
        out.append(dict(unit=k, zero=zero, z={f: zs[f][i] for f in P1_FEAT},
                        next={f: np.delete(zs[f], i).min() for f in P1_FEAT},
                        top={f: np.delete(zs[f], i).max() for f in P1_FEAT},
                        weight=fit["weights"][comp[k]] if k in comp else np.nan))
    return out


_P1_FL = {"entropy": "entropy", "rt_var": "RT variance"}


def _p1_fit_text(r):
    """One degenerate carried fit (minority < P1_MIN), described from the diagnostics."""
    run = next(k for k, v in RUN_DATASET.items() if v == r.Dataset)
    n, fl, u = int(r.Units), int(r["Flagged (as run)"]), _p1_unit[r.Design]
    odd = len(P1_ZI[(run, "session" if r.Design == "participant" else r.Design, "entropy")]) > 0
    head = f"{r.Dataset} {r.Solution}, {r.Design} design: {fl} of {n} {u}s flagged"
    cf = (f"; with the exact-zero entropy value replaced by the next-smallest deviation the same fit flags "
          f"{int(r[P1_VARS['entropy']])} {u}s, with the RT-variance value replaced {int(r[P1_VARS['rt_var']])}, with both "
          f"{int(r[P1_VARS['both']])} (minority {int(r['Minority (both replaced)'])})") if odd else ""
    if r.Design == "participant":
        return head + f" (the participants with a flagged session in the {r.Solution} session fit)" + cf
    lone = _p1_lone(run, r.Design, r.Solution)
    if not lone:
        return head + " (no unit in the other profile)" + cf
    parts = []
    for L in lone:
        w = f", component weight {L['weight']:.4f}" if np.isfinite(L["weight"]) else ""
        if L["zero"]:
            f = next(k for k, v in P1_FEAT.items() if v == L["zero"][0])
            parts.append(f"{L['unit']}, the median {u} on {_P1_FL[f]}: its folded value is exactly 0, "
                         f"{abs(L['z'][f]):.1f} SD below the mean as the labeller standardises it (next {u} "
                         f"{abs(L['next'][f]):.1f} SD below){w}")
        else:
            f = max(P1_FEAT, key=lambda k: abs(L["z"][k]))
            zf = L["z"][f]
            tail = (f", the largest of the {n} (next {L['top'][f]:.1f} SD above)" if zf > L["top"][f] else
                    f", the smallest of the {n} (next {abs(L['next'][f]):.1f} SD below)" if zf < L["next"][f] else "")
            parts.append(f"{L['unit']}, not an exact-zero unit: its power-transformed {_P1_FL[f]} (exponent "
                         f"{_p1_exp(run, r.Solution, f):g}) lies {abs(zf):.1f} SD {'above' if zf > 0 else 'below'} the "
                         f"mean{tail}{w}")
    par = f"{n} units, odd" if n % 2 else f"{n} units, even, so no value is exactly 0"
    lab_ = ("this unit is the labelled (fastest) component" if all(L["unit"] in P1_SETS[(run, r.Design, r.Solution)]["as run"]
                                                                for L in lone)
            else f"the fastest component, the labelled one, holds the other {n - len(lone)}")
    return head + f" ({par}); the other profile holds " + "; ".join(parts) + f"; {lab_}" + cf


P1_DEG = P1_CF[P1_CF["Minority (as run)"] < P1_MIN].reset_index(drop=True)
P1_DEG_TXT = {(r.Dataset, r.Design, r.Solution): _p1_fit_text(r) for _, r in P1_DEG.iterrows()}
_p1z = P1_ZERO[(P1_ZERO.Exponent < 1) & P1_ZERO["z of the zero unit"].notna()]
P1_FOLD_TXT = ("The fold centres each feature on the median unit of its matrix: with an odd number of units that unit's "
               "|MAD-z| is exactly 0, so its power-transformed value is 0 for any exponent, while an exponent below 1 "
               "compresses every other unit towards 1 (|z|^p ≈ 1 + p·ln|z|). After the labellers' standardisation the "
               f"exact-zero unit lies {-_p1z['z of the zero unit'].max():.1f}–{-_p1z['z of the zero unit'].min():.1f} SD "
               f"below the mean and {_p1z['Gap (SD)'].min():.1f}–{_p1z['Gap (SD)'].max():.1f} SD below the next unit "
               f"(exponents < 1, Table 4.30). With an even number of units no value is 0.") if len(_p1z) else ""


def p1_deg_text(datasets=None, solution=None):
    """The degenerate carried fits (minority profile < P1_MIN) of these datasets / this solution, described."""
    t = [v for (ds, des, sol), v in P1_DEG_TXT.items() if (datasets is None or ds in datasets)
         and (solution is None or sol == solution)]
    return ("Degenerate carried fits (minority profile < " + f"{P1_MIN} units): " + ". ".join(t) + ". " + P1_FOLD_TXT
            + " Tables 4.30–4.31.") if t else ""


# Table 4.32: stability refits split by the copies of the exact-zero units each refit drew (the stability stage's seeds)
_P1_DC = {}


def _p1_draws(run, n_ref, n):
    """Copies of the exact-zero entropy and RT-variance blocks in each stability refit: refit b resamples the n dynamic
    blocks with RandomState(random_seed + b).choice(n, n, replace=True), as the runs' and the sweep's stability stages do."""
    if (run, n_ref, n) not in _P1_DC:
        seed = int(RUNLOG[run]["parameters"]["random_seed"])
        assert P1_M[(run, "dynamic")][["adj_rt_level_block", "exp_rt_var_block", "exp_entropy_block", "exp_rt_var_block_gmm",
                                      "exp_entropy_block_gmm"]].notna().all().all(), \
            "dynamic blocks with missing features: the resample positions are not the matrix rows"
        zi = np.r_[P1_ZI[(run, "dynamic", "entropy")], P1_ZI[(run, "dynamic", "rt_var")]].astype(int)
        cnt = np.zeros((n_ref, len(zi)), int)
        for b in range(n_ref):
            cnt[b] = np.bincount(np.random.RandomState(seed + b).choice(n, size=n, replace=True), minlength=n)[zi]
        ne = len(P1_ZI[(run, "dynamic", "entropy")])
        _P1_DC[(run, n_ref, n)] = (cnt[:, :ne].sum(1), cnt[:, ne:].sum(1))
    return _P1_DC[(run, n_ref, n)]


rows, P1_DRAWS = [], {}
for run in RUN:
    n = len(P1_M[(run, "dynamic")])
    if not len(P1_ZI[(run, "dynamic", "entropy")]):
        continue
    for lab, arm in P1_ARMS.items():
        a = REPL[run][f"ari|{arm}"]
        if run not in P1_DRAWS:
            P1_DRAWS[run] = _p1_draws(run, len(a), n)
        ce, cr = P1_DRAWS[run]
        for cat, m in (("all refits", np.ones(len(a), bool)), ("entropy zero: 0 copies", ce == 0),
                       ("entropy zero: 1 copy", ce == 1), ("entropy zero: 2 copies", ce == 2),
                       ("entropy zero: ≥ 3 copies", ce >= 3), ("neither exact-zero block drawn", (ce == 0) & (cr == 0))):
            rows.append({"Dataset": RUN_DATASET[run], "Solution": lab, "Blocks": n,
                         "Exact-zero blocks": f"entropy {', '.join(P1_ZK[(run, 'dynamic', 'entropy')])}; RT variance "
                                              f"{', '.join(P1_ZK[(run, 'dynamic', 'rt_var')]) or '—'}",
                         "Refits drawn": cat, "Refits": int(m.sum()),
                         "Median ARI": np.median(a[m]) if m.any() else np.nan,
                         "Refits ARI < .50 (%)": 100 * np.mean(a[m] < .5) if m.any() else np.nan})
P1_STAB = pd.DataFrame(rows)
_p1_even = [f"{RUN_DATASET[r]} ({len(P1_M[(r, 'dynamic')])} blocks)" for r in RUN if not len(P1_ZI[(r, "dynamic", "entropy")])]


def p1_stab_text(datasets):
    """% of refits with ARI < .50 by copies of the exact-zero entropy block drawn, per dataset and carried solution."""
    out = []
    for ds in datasets:
        g = P1_STAB[P1_STAB.Dataset == ds]
        if not len(g):
            continue
        for lab in P1_ARMS:
            v = g[g.Solution == lab].set_index("Refits drawn")["Refits ARI < .50 (%)"]
            out.append(f"{ds} {lab} {v['all refits']:.1f}% overall; 0 / 1 / 2 / ≥ 3 copies "
                       + " / ".join(f"{v[c]:.1f}" for c in ("entropy zero: 0 copies", "entropy zero: 1 copy",
                                                            "entropy zero: 2 copies", "entropy zero: ≥ 3 copies"))
                       + f"%; neither exact-zero block drawn {v['neither exact-zero block drawn']:.1f}%")
    return ("Refits with ARI < .50 by the copies of the exact-zero folded-entropy block each refit drew (the stability "
            "stage's own resamples; Table 4.32): " + "; ".join(out) + ".") if out else ""


# the sweep's stability tie-breaks (Table 3.4): the tied configurations among the refits that draw neither exact-zero unit
_st4 = S2("stability")
_st4[["Arm", "Config"]] = _st4.Solution.str.extract(r"^(\S+) \((cluster\d+)\)$")
_n2 = len(P1_M[("both", "dynamic")])
rows = []
for arm_, g in _st4.groupby("Arm", sort=False):
    if len(g) < 2:
        continue
    for r in g.itertuples(index=False):
        a = np.load(SWEEPS_DIR / LVL2_SWEEP / "stability" / f"{r.Config}.npz")[arm_]
        ce, cr = _p1_draws("both", len(a), _n2)
        m = (ce == 0) & (cr == 0)
        rows.append({"Arm": arm_, "Config": r.Config, "Status": r.Status, "Median": np.median(a), "lt50": 100 * np.mean(a < .5),
                     "n0": int(m.sum()), "Median0": np.median(a[m]), "lt50_0": 100 * np.mean(a[m] < .5)})
P1_TIE = pd.DataFrame(rows)
P1_TIE_TXT = " ".join(
    f"Tie-break of {FAM_LABEL[a_.rsplit('_', 1)[0]]} ({a_.rsplit('_', 1)[1]}): "
    + " vs ".join(f"{r.Config} median {r.Median:.3f} ({r.lt50:.1f}% < .50)" for r in g.itertuples(index=False))
    + f" over all refits; among the {g.n0.iloc[0]:,} refits that draw neither exact-zero block of the pooled run's dynamic matrix, "
    + " vs ".join(f"{r.Config} {r.Median0:.3f} ({r.lt50_0:.1f}% < .50)" for r in g.itertuples(index=False))
    + (". The order of the medians reverses among the refits that draw neither exact-zero block: the tie-break follows "
       "the refits that draw them." if list(g.sort_values("Median").Config) != list(g.sort_values("Median0").Config)
       else ". The order of the medians is the same among the refits that draw neither exact-zero block.")
    for a_, g in P1_TIE.groupby("Arm", sort=False)) if len(P1_TIE) else ""
if len(P1_TIE) and not (len(P1_ZI[("both", "dynamic", "entropy")]) or len(P1_ZI[("both", "dynamic", "rt_var")])):
    P1_TIE_TXT = " ".join(        # the pooled run's dynamic matrix has no exact-zero block
        f"Tie-break of {FAM_LABEL[a_.rsplit('_', 1)[0]]} ({a_.rsplit('_', 1)[1]}): "
        + " vs ".join(f"{r.Config} median {r.Median:.3f} ({r.lt50:.1f}% < .50)" for r in g.itertuples(index=False))
        + " over all refits (the pooled run's dynamic matrix has no exact-zero block)."
        for a_, g in P1_TIE.groupby("Arm", sort=False))

# the LVL2 sweep's removals that leave a single unit of their design unflagged (the pooled run's matrices; Tables 3.5-3.6)
_rule2 = RUNLOG["T2"]["parameters"]["worst_cluster_idx"]
_p1_T2keys = {des: [_p1_key(des, *r) for r in P1_M[("both", des)][["id", "session", "block"]].values] for des in _p1_src("both")}
rows, _p1_unknown = [], 0
for f_ in sorted((SWEEPS_DIR / LVL2_SWEEP).glob("cluster[0-9][0-9][0-9][0-9][0-9][0-9]_flagged.csv")):
    d_ = pd.read_csv(f_)
    d_ = d_[d_.Rule == _rule2]
    for (des, arm_), g in d_.groupby(["Design", "Removal_Method"]):
        if des not in _p1_T2keys:
            continue
        ks = set(_p1_keys(des, g[["id", "session", "block"]].values))
        _p1_unknown += len(ks - set(_p1_T2keys[des]))
        if len(ks) == len(_p1_T2keys[des]) - 1:
            (u_,) = set(_p1_T2keys[des]) - ks
            rows.append({"Config": f_.name.removesuffix("_flagged.csv"), "Design": des, "Arm": arm_, "Unflagged": u_,
                         "Zero": [P1_FEAT[f] for f in P1_FEAT if u_ in P1_ZK[("both", des, f)]]})
P1_NT = pd.DataFrame(rows, columns=["Config", "Design", "Arm", "Unflagged", "Zero"])
_armlab = lambda a: f"{FAM_LABEL[a.rsplit('_', 1)[0]]} ({a.rsplit('_', 1)[1]})"
_cfgspan = lambda c: (f"{c.min()}–{c.max()}" if int(c.max()[-6:]) - int(c.min()[-6:]) + 1 == len(c)
                      else f"between {c.min()} and {c.max()}")
if len(P1_NT):
    _ex = S2("dose_fits"); _exc = next(c for c in _ex.columns if c.startswith("Removals excluded"))
    _ex = _ex[_ex.Family == "overall"].set_index(["Arm", "Design"])[_exc]
    _ntz = P1_NT[P1_NT.Zero.str.len() > 0]
    P1_M2_TXT = (f"{len(P1_NT)} removals of the sweep flag all but one unit of their design ("
                 + "; ".join(f"{des} {_armlab(a)}: {len(g)} ({_cfgspan(g.Config)}; Table 3.5 excludes "
                             f"{int(_ex.get((a, des), 0))} {des} removals of this arm)"
                             for (des, a), g in P1_NT.groupby(["Design", "Arm"]))
                 + f"). In {len(_ntz)} of them the one unflagged unit is the design's exact-zero folded unit ("
                 + "; ".join(f"{u} {des}, {'/'.join(z).lower()}" for (u, des), z in
                             _ntz.groupby(["Unflagged", "Design"]).Zero.first().items())
                 + "; Table 4.30): it is the only unit outside the labelled component.")
    P1_M2_SHORT = (f"Of the {len(P1_NT)} removals that flag all but one unit of their design, {len(_ntz)} leave the "
                   "design's exact-zero folded unit as the one unflagged unit (Table 3.5).")
else:
    P1_M2_TXT = P1_M2_SHORT = ""

# ---- checks: the rebuild reproduces the runs' exports -----------------------------------------------------------------
for run, R in P1_RB.items():
    _par = RUNLOG[run]["parameters"]
    _cd = {k: (v, R["config"].get(k)) for k, v in _par.items()
           if k not in ("n_jobs", "kmeans_bootstrap_reps") and cfg_val(v) != cfg_val(R["config"].get(k))}
    check(f"Exact-zero diagnostics, {run} run: the rebuild's LVL_2_CONFIG = the run's (every key it records but the "
          "worker count and the k-means bootstrap, which do not enter the fits)", not _cd, f"{len(_par)} keys; differing: {_cd or 'none'}")
    check(f"Exact-zero diagnostics, {run} run: |MAD-z| ^ exponent reproduces every power-transformed column of the rebuilt "
          "matrices (lvl2.py add_exp_features)", R["fold_formula"])
    _u = pd.read_csv(f"{RUN[run]}_units.csv", float_precision="round_trip")
    _bm = P1_M[(run, "dynamic")]
    _cols = [c for c in _bm.columns if c in _u.columns]
    check(f"Exact-zero diagnostics, {run} run: the rebuilt dynamic matrix = _units.csv (same rows in the same order, "
          f"{len(_cols)} columns bitwise)", len(_u) == len(_bm) and all(np.array_equal(_u[c].to_numpy(), _bm[c].to_numpy())
                                                                        for c in _cols), f"{len(_bm)} blocks; {_cols}")
    _dd = P("dsc_designs", [RUN_DATASET[run]]).set_index(["Solution", "Design"])
    _cf = P1_CF[P1_CF.Dataset == RUN_DATASET[run]].set_index(["Solution", "Design"])
    _bad = [i for i in _cf.index if not (_cf.loc[i, "Units"] == _dd.loc[i, "Units"] and _cf.loc[i, "Flagged (as run)"]
                                         == _dd.loc[i, "Flagged"] and _cf.loc[i, "_H"] == _dd.loc[i, "Heuristic flagged"]
                                         and _cf.loc[i, "_Hboth"] == _dd.loc[i, "Both"])]
    check(f"Exact-zero diagnostics, {run} run: the rebuilt Step 5 reproduces the run's units, flagged units, heuristic "
          "flags and their overlap in every design of both carried solutions (Table 4.7b)", len(_cf) > 0 and not _bad,
          f"{len(_cf)} fits; differing: {_bad or 'none'}")
    _nb = [k for k in R["fits"]["as run"] if sorted(map(tuple, R["fits"]["none"][k]["flagged"]))
           != sorted(map(tuple, R["fits"]["as run"][k]["flagged"]))]
    check(f"Exact-zero diagnostics, {run} run: Step 5 executed again on the unaltered matrices (every k-means refitted) "
          "gives the run's flags", not _nb, f"{len(R['fits']['as run'])} flag sets; differing: {_nb or 'none'}")
    _cp, _bad = P("components", [RUN_DATASET[run]]), []
    for (lab, des), g in _cp.groupby(["Solution", "Design"]):
        src, arm = _p1_src(run)[des], P1_ARMS[lab]
        fit = R["fits"]["as run"][(src, arm)]
        if _p1_gmm(arm):
            mine = sorted(np.bincount([c[3] for c in fit["component"]]).tolist())
            ok = mine == sorted(g.Units.astype(int).tolist())
        else:
            ok = int(g[g.Labelled.astype(bool)].Units.iloc[0]) == len(fit["flagged"]) and int(g.Units.sum()) == len(P1_M[(run, des)])
        _bad += [] if ok else [(lab, des)]
    check(f"Exact-zero diagnostics, {run} run: the rebuilt fits' component sizes = Table 4.26", len(_cp) > 0 and not _bad,
          f"differing: {_bad or 'none'}")
    _rd = {a: float(np.max(np.abs(np.asarray(v) - REPL[run][f"ari|{a}"][:len(v)]))) for a, v in R["refits"].items()}
    check(f"Exact-zero diagnostics, {run} run: the first {P1_NCHK} stability refits recomputed from the seeds = the run's "
          "exported ARIs (the resamples that Table 4.32 counts)", max(_rd.values()) < 1e-12, str(_rd))
for lab, (cfg_, arm) in PICKS.items():
    _sa = np.load(SWEEPS_DIR / LVL2_SWEEP / "stability" / f"{cfg_}.npz")[arm]
    check(f"Exact-zero diagnostics: the sweep's stability refits of {lab} ({cfg_}) = the pooled run's (same resamples and order)",
          np.array_equal(_sa, REPL["both"][f"ari|{arm}"]), f"{len(_sa)} refits")
check("Exact-zero diagnostics: every unit in the sweep's flagged-unit files is a unit of the pooled run's rebuilt matrices",
      _p1_unknown == 0, f"{_p1_unknown} unknown")
print(f"exact-zero units: {sum(len(v) for v in P1_ZI.values())} in {len(P1_M)} matrices; degenerate carried fits "
      f"(minority < {P1_MIN}): {len(P1_DEG)}; sweep removals leaving one unit unflagged: {len(P1_NT)}")


def p1_dsc_text(datasets):
    """DSC with the behavioural standard of the degenerate fits of these datasets, as run and with the value(s) replaced."""
    t = [f"{r.Dataset} {r.Solution} {r.Design} {r['DSC with the heuristic (both replaced)']:.3f} (as run "
         f"{r['DSC with the heuristic (as run)']:.3f})" for _, r in P1_DEG.iterrows() if r.Dataset in datasets]
    return (" DSC with the behavioural standard with the exact-zero values replaced (Table 4.31): " + "; ".join(t)
            + ".") if t else ""

*[cell 12]*

## 3. LVL2 — classification calibration on all data

The LVL_2 sweep crossed the entropy and RT-variance exponents on the dynamic blocks (T2 and T3) of the selected detector; every configuration is fitted as k-means (k = 2, 3) and GMM (k = 2, 3), labelled with the speed rule (the fastest component is the inattentive one). Calibration target: the Sørensen–Dice coefficient (DSC) between an arm's flagged units and the a-priori behavioural standard's flagged units of the same design. A configuration is **degenerate** when its DSC < 0.20; "no overlap" means DSC = 0.

In [ ]:
# [cell 13]
# ---- 3. LVL2 calibration: the LVL2 sweep report's exports; the carried solutions' blocks (T2 + T3 run) -----------------
_units = S2("dsc_units").set_index("Design")
_dsum = S2("dsc_summary")
_n_cfg = int(_dsum.Configs.max())
_grid = sorted(S2("dsc_configs").exp_entropy.unique())
show(3, "3a", f"Table 3.1. DSC with the a-priori standard across the {_n_cfg} sweep configurations, by arm and design",
     _dsum,
     note="DSC = 2|A ∩ G| / (|A| + |G|) between the arm's flagged units and the a-priori standard's flagged units of the "
          f"same design (G: {_units.loc['dynamic', 'A-priori flagged']} dynamic blocks, "
          f"{_units.loc['control', 'A-priori flagged']} control blocks, {_units.loc['session', 'A-priori flagged']} "
          "sessions). Degenerate = DSC < .20 (these configurations flag few units, i.e. under-flag); configurations "
          "flagging ≥ 99% of units are not degenerate under this definition (count shown). Pick = the arm's calibrated "
          "configuration. The participant design is not in the sweep's flagged-unit files (see Table 3.3). Chance-expected "
          "DSC = 2nG / (N(n + G)), the expected overlap if the arm's n flagged units were drawn at random from the design's "
          f"N units (N = {_units.loc['dynamic', 'Units']:,} blocks or {_units.loc['session', 'Units']} sessions; G = the "
          "standard's flagged units). All arms = the four arms' configurations together.")
show(3, "3a2", "Table 3.1b. DSC with the a-priori standard in qualitative bands, by arm and design", S2("dsc_bands"),
     nd_cols={"Configs": 0, "Configs (%)": 1, "Units flagged (%): min": 1, "Units flagged (%): median": 1,
              "Units flagged (%): max": 1},
     note="Bands of Savio et al. (2017): < .20 poor, .20–.40 fair, .40–.60 moderate, .60–.80 good, > .80 excellent "
          "(poor = the degenerate configurations of Table 3.1). Units flagged = the share of the design's units the "
          "configurations of the band flag (min, median, max over them). All arms = the four arms' configurations together.")
_dv = S2("dsc_variance")
show(3, "3b", "Table 3.2. Variance in DSC explained by each exponent, by arm and design", _dv,
     note="SS share = Type II sum of squares of the exponent (additive model DSC ~ entropy exponent + RT-variance exponent, "
          "both categorical) / total SS; the remainder is the exponent interaction (all configurations: a balanced grid, one "
          "value per cell, so the shares are orthogonal and add to 1) or interaction plus imbalance (non-degenerate subset, "
          f"where the shares need not add to 1). Exponent grid (both exponents, {len(_grid)} values): "
          + ", ".join(f"{v:g}" for v in _grid) + ". Descriptive.")
# DSC at the calibrated configurations; the participant design of the carried solutions from the pooled run
_dp = S2("dsc_picks")
_dsg = P("dsc_designs", ["Pooled, overall"])
_arm2lab = {a: lab for lab, (_, a) in PICKS.items()}
_part = _dsg[_dsg.Design == "participant"].set_index("Solution")
_dp["participant DSC"] = [(_part.loc[_arm2lab[a], "DSC"] if a in _arm2lab else np.nan) for a in _dp.Arm]
_dp["participant n flagged"] = [(_part.loc[_arm2lab[a], "Flagged"] if a in _arm2lab else np.nan) for a in _dp.Arm]
_hp = _part["Heuristic flagged"].iloc[0] if len(_part) else np.nan
_dc = S2("dsc_configs"); _near = {}
for _fam, _f in SEL2["families"].items():                    # other configurations within .01 of each pick's dynamic DSC
    _g = _dc[(_dc.arm == _f["arm"]) & (_dc.design == "dynamic")]
    _pdsc = _g.loc[_g.config == _f["config"], "DSC"].iloc[0]
    _near[_fam] = (int(((_g.DSC >= _pdsc - .01) & (_g.config != _f["config"])).sum()), len(_g) - 1)
show(3, "3c", "Table 3.3. DSC with the a-priori standard at each arm's calibrated configuration, by design (Pooled, overall)", _dp,
     note=f"A-priori standard: dynamic {_units.loc['dynamic', 'A-priori flagged']}, control "
          f"{_units.loc['control', 'A-priori flagged']}, sessions {_units.loc['session', 'A-priori flagged']}, "
          f"participants {_hp}. The participant design exists only for the two carried solutions (T2 + T3 run); the sweep does "
          "not export it (— for the other arms). The picks are weakly identified: "
          + "; ".join(f"{n_} other {('k-means' if fam_ == 'cluster' else 'GMM')} configurations (of {tot_}) come within .01 "
                      "of the pick's dynamic DSC" for fam_, (n_, tot_) in _near.items()) + ".",
     nd_cols={**{c: 0 for c in ["dynamic n flagged", "control n flagged", "session n flagged", "participant n flagged"]},
              "Entropy exponent": 2, "RT-variance exponent": 2})
show(3, "3c2", "Table 3.3b. Maximum DSC of every arm and design across the sweep, with its exponents", S2("dsc_maxima"),
     nd_cols={"Configs at the maximum": 0},
     note=f"Exponents of the configuration(s) with the arm's highest DSC in each design (the grid's ends are {_grid[0]:g} "
          f"and {_grid[-1]:g}). The calibrated configuration of each arm is chosen on the dynamic design (Table 3.3); ties "
          "are broken by stability (Table 3.4).")
_ARI_PANELS = {"robust summary": ["Refits", "Median", "Q1", "Q3", "sMAD"],
               "distribution": ["Mean", "SD", "2.5%", "97.5%", "Refits ARI < .50 (%)"]}
_st = S2("stability")
show_panels(3, "3d", "Table 3.4. Cluster stability (bootstrap ARI) of the carried solutions and the tied candidates, T2 and T3",
            _st, ["Solution", "Status", "Blocks"], _ARI_PANELS, nd_cols={"Refits ARI < .50 (%)": 1, "Refits": 0},
            note=f"{int(_st.Refits.max()):,} refits on blocks resampled with replacement (the LVL2 sweep's stability "
                 "stage); ARI between each refit and the full-data partition on the resampled blocks. Candidates: each "
                 "arm's highest-DSC configuration and every exact tie; within an arm the more stable tied configuration "
                 "is the arm pick, and between the arms of a family the higher median ARI was carried. Distributions are "
                 "often bimodal: compare median, IQR, the 2.5–97.5% range and the share of refits with ARI < .50, not the "
                 "mean alone. The carried solutions' refits are the pooled run's (Table 7.3). "
                 + p1_stab_text(["Pooled, overall"]) + " " + P1_TIE_TXT)
show(3, "3e", "Table 3.5. Dose–response of the alpha gain across the sweep: saturating fits and AUC, by arm and design",
     S2("dose_fits"),
     note="Per configuration and design: alpha gain beyond matched random removal (the sweep's Step 8, canonical alpha; "
          "block designs remove exactly the flagged number of trials from as many random sessions as hold flagged blocks, "
          "and a replicate whose sessions hold fewer trials is invalid; session and participant designs remove as many "
          "random whole units), mean over the scales, against % of trials removed. Removals whose post-removal coverage falls below the runs' floor (h2_min_coverage, given in the 'Removals excluded' header) in any scale are left out of the fits, the common dose range and both bootstraps; 'Removals excluded' counts them. Fit: excess = a(1 − e^(−b·v)), soft-L1 "
          "robust least squares. AUC = mean height of the fitted curve over the common dose range of the arm's designs; "
          "per-trial = alpha gain per trial removed (× 10⁻⁶). R² = fit of the curve to that design's configurations on the "
          "panel's scale; R² < 0 means the curve fits worse than a flat line and its AUC is least trustworthy. Differs "
          "from every other design (q < .05) = q < .05 in each of the design's contrasts with the arm's other designs in "
          "Table 3.6 (the distinct-solution test; the * of Figure 3.1). " + P1_M2_TXT,
     nd=4, nd_cols={"b (rate)": 4, "R²": 3})
_dc, _dcc = S2("dose_contrasts"), S2("dose_contrasts_per_config")
_NSOL = {"n_solutions", "n_configs"} <= set(_dc.columns)
_nsol = _dc.groupby("Arm", sort=False)[["n_solutions", "n_configs"]].first() if _NSOL else pd.DataFrame()
check("Table 3.6: the distinct-solution test (solutions and configurations per arm; solutions ≤ configurations)",
      _NSOL and len(_nsol) > 0 and bool((_nsol.n_solutions <= _nsol.n_configs).all()),
      _nsol.to_dict("index") if _NSOL else "the export has no n_solutions / n_configs (rerun LVL2_sweep_report)")
_kc = ["Family", "Arm", "Comparison"]
_mc = _dc.merge(_dcc, on=_kc, suffixes=("", " (per configuration)"))
_dmc = max(float(np.abs(_mc[c] - _mc[f"{c} (per configuration)"]).max()) for c in ("AUC A", "AUC B", "Diff"))
check("Tables 3.6 and 3.6b: the same contrasts and point estimates (the primary test keeps every configuration)",
      len(_mc) == len(_dc) == len(_dcc) and _dmc < 1e-12, f"{len(_mc)} contrasts, max |diff| {_dmc:.1e}")
_qdc = max(float(np.nanmax(np.abs(bh(g.p.to_numpy()) - g.q.to_numpy()))) for _, g in _dc.groupby("Family"))
check("Table 3.6: q = BH within each family (overall; per-trial)", _qdc < 1e-12, f"max |diff| {_qdc:.1e}")
_dfit = S2("dose_fits")
_smis = [(r["Arm"], r["Family"], r["Design"]) for _, r in _dfit.iterrows()
         if bool((_dc[(_dc.Family == r["Family"]) & (_dc.Arm == r["Arm"]) & _dc.Comparison.str.contains(r["Design"])].q
                  < ALPHA).all()) != bool(r["Differs from every other design (q < .05)"])]
check("Table 3.5 / Figure 3.1: 'differs from every other design' = the Table 3.6 test", not _smis,
      f"{len(_dfit)} designs; mismatches {_smis[:6]}")
show(3, "3f", "Table 3.6. Dose–response AUC: pairwise design contrasts (paired bootstrap over distinct solutions)", _dc,
     note="Primary test, on the eligible removals of Table 3.5. Resampling unit: the arm's distinct solutions (configurations that flag the same units in every "
          "design count as one solution; solutions / configurations per arm: "
          + ("; ".join(f"{a} {int(r.n_solutions)} / {int(r.n_configs)}" for a, r in _nsol.iterrows()) if _NSOL else "—")
          + f"). Each of the {int(_dc.n_boot.max()):,} resamples draws the arm's solutions with replacement and includes "
          "every configuration of each drawn solution, the same resample for every design, so the point estimates are the "
          "all-configuration values (as Table 3.6b); two-sided percentile p with the +1 correction; BH-FDR within each "
          "family (overall; per-trial) across all arms' contrasts. " + P1_M2_SHORT,
     nd=4, nd_cols={"n_boot": 0, "n_solutions": 0, "n_configs": 0})
show(3, "3f2", "Table 3.6b. Dose–response AUC: pairwise design contrasts, configurations resampled (sensitivity)", _dcc,
     note=f"Sensitivity analysis of Table 3.6: the same contrasts with the arm's configurations as the resampling unit "
          f"({int(_dcc.n_boot.max()):,} resamples, the same resample for every design; two-sided percentile p with the +1 "
          "correction; BH-FDR within each family across all arms' contrasts). The configurations are not independent (many "
          "give the same solution; Table 3.6), so these p and q are descriptive.",
     nd=4, nd_cols={"n_boot": 0})
show(3, "3g", "Table 3.7. Where each calibrated configuration sits on its dynamic dose–response curve (T2 and T3)", S2("dose_picks"),
     note="The calibrated configuration (open circle of the dose figure): its removal volume and alpha gain beyond matched "
          "random removal, and the fitted curve's value at that dose.", nd=4)
show_figure(3, "f3_1", "Figure 3.1. Dose–response of the alpha gain across the sweep, by arm and design",
            FIG_DIR / "supplement" / "fig_dose_response.png",
            note="make_replication_figures from the LVL2 sweep report's dose exports (the fits and integration range of "
                 "Tables 3.5 and 3.7). Points = configurations; lines = saturating fits over each design's observed doses; "
                 "shading = the area integrated for the AUC (dotted line = its upper bound); open circle = the calibrated "
                 "configuration; box = each design's AUC and R²; * = AUC differs from every other design in that panel "
                 "(q < .05 in each of its contrasts in Table 3.6, the distinct-solution test; BH within the overall and "
                 "per-trial families).")

# block counts and features of the carried solutions (Pooled, overall here; the other datasets in Tables 4.10-4.16)
FEAT_LABELS = ["RT level", "RT variance", "Entropy", "RT level per scale", "RT variance per scale", "Entropy per scale"]
_ex = "; ".join(f"{lab} {e[0]:g} / {e[1]:g}" for lab, e in EXPONENTS.items())
_FEAT_NOTE = ("Unit: dynamic block. Cells: median [Q1, Q3] (sMAD). Composite RT level = mean over stimuli of the mean over "
              "scales of rt_input (log-RT drift residual, age-adjusted at LVL_0); mean log RT = the same from log RT (RT in "
              "seconds). RT variance and entropy: per-scale block variance of rt_input / normalised Shannon entropy of "
              "the responses; composite = mean over scales after per-scale age adjustment (the clustering input); log-RT "
              "variance = per-scale block variance of log RT, mean over scales; Box-Cox transformed = after Box-Cox across "
              "blocks; power-transformed = |MAD-z of the Box-Cox value| ^ exponent "
              f"(entropy / RT variance: {_ex}); raw entropy = unadjusted mean over scales; per-scale values: before LVL_2's "
              "per-scale age adjustment (per-scale RT level = the block mean of the scale's rt_input, age-adjusted at "
              "LVL_0). Fits and transforms are estimated within each dataset's run.")
# the runs' RT columns as computed: LVL_1's per-scale RT is log RT, so their "Raw RT" columns hold log RT and their "Log
# RT" columns log(log RT) (defined only where log RT > 0); shown relabelled, the latter left out
check("Block-feature tables: LVL_1's per-scale RT is log RT (the runs' 'Raw RT' columns are shown as log RT)",
      cfg_val(META["lvl1_config"]["feature_rt_transform"]) == "log", str(META["lvl1_config"]["feature_rt_transform"]))
FEAT_SHOW = lambda d: (d.drop(columns=["Log RT (log s)", "Log RT variance"], errors="ignore")
                       .rename(columns={"Raw RT (s)": "Mean log RT (log s)", "Raw RT variance (s²)": "Log-RT variance",
                                        "Extremized": "Power-transformed"}))
show(3, "3h", "Table 3.8. Flagged and unflagged dynamic blocks of the carried solutions, Pooled, overall",
     P("block_counts", ["Pooled, overall"]), nd_cols={"Trials in flagged blocks (%)": 1})
for i, fam in enumerate(FEAT_LABELS):
    show(3, f"3i_{i}", f"Table 3.{9 + i}. Block features of the carried solutions, Pooled, overall — {fam}",
         FEAT_SHOW(P(f"block_features_{i}", ["Pooled, overall"]).drop(columns="Dataset")), _FEAT_NOTE)

*[cell 17]*

## 4. LVL2 — H2 flagged vs unflagged blocks

For every dataset × carried solution, three primary tests:

- **Size (minority profile).** Flagged share of the dynamic blocks (block-weighted: Σ flagged blocks / Σ blocks) against the equal-profile share p0 = 1/k; H1: share < p0, one-sided. Participant-cluster bootstrap (participants resampled with all their blocks).
- **Speed.** Difference in block-weighted mean `adj_rt_level_block` (flagged − unflagged); H1: < 0, one-sided; participant-cluster bootstrap. The labelling rule makes the flagged component the fastest of its fit, so in every dataset that is its own fit the flagged blocks are faster by construction: there the contrast reports the size of the speed difference and is not a test that can fail.
- **IRR.** Krippendorff's α (canonical) of the responses in flagged blocks vs those in unflagged blocks, per scale × condition, averaged over the row's cells; two-sided; stimulus-paired bootstrap (one stimulus resample per condition and replicate, shared by both matrices and all scales; raters fixed). Eligibility: in every cell, both the flagged and the unflagged matrix must reach the coverage floor; otherwise the result is excluded from inference (also in robustness A and the GMM posterior-threshold tests; in robustness B the rule applies to each split, and only the splits that meet it enter).

BH-FDR within each row (the three tests); BH over all primary tests as a sensitivity column. One-sided tests report a one-sided 95% bound; the two-sided IRR test a 95% CI.

**Secondary: the heuristic as a classifier (Tables 4.21–4.23).** The same three primary tests applied to the blocks the a-priori behavioural heuristic flags. Size has no equal-profile share for a fixed rule, so the minority claim is tested against a half (Cohen's h as the effect size); its own BH family.

In [ ]:
# [cell 18]
# ---- 4a. H2 primary tests: size, speed, IRR (the LVL_2 runs' Step 5e); BH across every dataset here -------------
CFG2 = RUNLOG["T2"]["parameters"]
for _k in ("h2_boot_n", "h2_min_coverage", "h2_robustness_a_min_stimuli", "h2_robustness_b_splits",
           "h2_robustness_b_min_raters", "stability_refits", "gmm_posterior_thresholds"):
    check(f"LVL_2 runs share {_k}", len({json.dumps(RUNLOG[r]["parameters"].get(_k)) for r in RUN}) == 1,
          str({r: RUNLOG[r]["parameters"].get(_k) for r in RUN}))
B, MIN_COVERAGE = int(CFG2["h2_boot_n"]), float(CFG2["h2_min_coverage"])
H2_SIZE, H2_SPEED, H2_IRR = P("h2_size"), P("h2_speed"), P("h2_irr")
_nall = 3 * len(H2_SIZE)
_q = bh(np.r_[H2_SIZE.p, H2_SPEED.p, H2_IRR.p]); n_ = len(H2_SIZE)
_QA = f"q (all {_nall})"
H2_SIZE[_QA], H2_SPEED[_QA], H2_IRR[_QA] = _q[:n_], _q[n_:2 * n_], _q[2 * n_:]
_pc = ["p", "q (row)", _QA]
show_panels(4, "4a", "Table 4.1. H2 size: flagged share of the dynamic blocks vs the equal-profile share (one-sided)", H2_SIZE,
     ["Dataset", "Solution"], {"counts and share": ["p0", "Participants", "Blocks", "Flagged blocks", "Share (blocks)", "95% upper bound"],
                               "test": ["Share − p0", "Share / p0", "p", "q (row)", _QA, "Direction"]},
     note=f"Unit of resampling: participant (cluster bootstrap, B = {B:,}). Share = Σ flagged blocks / Σ blocks (block-weighted; "
          "Table 4.6 gives the trial-weighted and participant-weighted shares). p0 = 1/2 (k = 2) or 1/3 (k = 3); "
          "H1: share < p0; p = (#{share* ≥ p0} + 1)/(B + 1); bound = 95th percentile of share*. q (row) = BH over the row's "
          f"three tests; {_QA} = BH over all primary H2 tests. The size test is a weak criterion (any solution whose flagged "
          "profile is the minority passes it); the speed direction follows from the labelling rule (Table 4.2). "
          "Tables 4.1–4.3 are manipulation checks; their estimates describe magnitudes.", p_cols=_pc,
     nd_cols={"p0": 3, "Participants": 0})
show_panels(4, "4b", "Table 4.2. H2 speed: flagged vs unflagged block RT level (one-sided)", H2_SPEED,
     ["Dataset", "Solution"], {"descriptives": ["Participants", "Flagged Mdn [Q1, Q3] (sMAD)", "Unflagged Mdn [Q1, Q3] (sMAD)",
                                                "Flagged M (SD)", "Unflagged M (SD)"],
                               "test": ["Δ (flagged − unflagged)", "95% upper bound", "exp(Δ) (RT ratio)", "d", "d 95% upper bound",
                                        "p", "q (row)", _QA, "Direction"]},
     note="Speed = adj_rt_level_block (log-RT drift residual, age-adjusted; lower = faster). Δ = difference of block-weighted "
          "means; exp(Δ) = ratio of RT to drift-expected RT; d = Δ / pooled within-group SD of block values. Participant-cluster "
          "bootstrap; H1: Δ < 0; bounds = 95th percentiles. Descriptives over blocks. The labelling rule makes the "
          "flagged component the fastest of its fit, so in every dataset that is its own fit the flagged blocks are faster "
          "by construction (Δ < 0): there this contrast reports the size of the speed difference and is not a test that "
          "can fail." + (" The pooled-derived rows carry the pooled fit's labels." if POOLED_DS else ""), p_cols=_pc)
show_panels(4, "4c", "Table 4.3. H2 IRR: Krippendorff's α of flagged vs unflagged responses (two-sided)", H2_IRR,
     ["Dataset", "Solution"], {"descriptives": ["Cells", "α flagged across cells Mdn [Q1, Q3] (sMAD)",
                                                "α unflagged across cells Mdn [Q1, Q3] (sMAD)", "α flagged across cells M (SD)",
                                                "α unflagged across cells M (SD)", "Min coverage flagged",
                                                "Min coverage unflagged"],
                               "test": ["α flagged (mean of cells)", "α unflagged (mean of cells)", "Δα", "95% CI",
                                        "α ratio (flagged / unflagged)", "p", "q (row)", _QA, "Direction"]},
     note=f"Canonical Krippendorff's α (ordinal; nominal for emotion) per scale × condition on the condition's full stimulus "
          f"set; row value = mean over the row's cells ({len(SCALES)} per condition). Stimulus-paired bootstrap (B = {B:,}): "
          "one stimulus resample per condition and replicate, applied to both matrices and all scales; raters fixed. "
          "Two-sided p; 95% percentile CI. Coverage = share of the condition's stimuli with ≥ 2 ratings in the flagged / "
          f"unflagged matrix (minimum over the row's cells); a row below {MIN_COVERAGE:.0%} in either matrix of any cell is "
          "excluded from inference (no CI, p or q; Table 6.3).", p_cols=_pc)

In [ ]:
# [cell 19]
# ---- 4b. H2 robustness A, B, C (the LVL_2 runs' Step 5e) ------------------------------------------------------------
MIN_STIM_A = int(CFG2["h2_robustness_a_min_stimuli"])
show(4, "4d", f"Table 4.4. H2 robustness A: participants with ≥ {MIN_STIM_A} flagged and ≥ {MIN_STIM_A} unflagged stimuli",
     P("h2_robustness_a"), p_cols=["p", "q (row)"],
     note="Same estimands, tests and bootstraps as Tables 4.1–4.3, restricted to participants contributing both profiles "
          "(always-attentive participants excluded), so nearly the same raters enter both IRR matrices. The size share rises "
          "by construction (participants without flagged blocks are removed); speed and IRR show whether the full-sample "
          "contrasts depend on the composition of raters. q (row) = BH over the row's three tests.")
_ns, _mu = int(CFG2["h2_robustness_b_splits"]), int(CFG2["h2_robustness_b_min_raters"])
show(4, "4e", "Table 4.5. H2 robustness B: rater-disjoint IRR contrast within the flagged participants", P("h2_robustness_b"),
     p_cols=["p", "q (row)"], nd_cols={"Splits meeting the coverage floor (%)": 1},
     note=f"Participants with ≥ 1 flagged block are split at random into halves ({_ns} splits): one half contributes "
          "only its unflagged (attentive) responses, the other only its flagged responses, so no rater is in both "
          f"matrices. Δα = α(flagged half) − α(attentive half), canonical α, mean over cells; each split carries "
          f"{B // _ns} stimulus resamples (B = {B:,} in all). A split enters only when ≥ {MIN_COVERAGE:.0%} of the "
          "stimuli have ≥ 2 ratings in both matrices of every cell (the coverage floor); Δα is the mean over the splits "
          "that meet it, and the two-sided p and 95% CI come from their draws pooled; 'Splits meeting the coverage floor "
          f"(%)' gives their share, and a row with no such split has no Δα, CI or p (Table 6.3). Minimum: ≥ {_mu} raters "
          "per half. One test per row, so q = p.")
show(4, "4f", "Table 4.6. H2 robustness C: trial-weighted prevalence, and the participant-weighted sensitivity",
     P("h2_robustness_c"), p_cols=["p", "q (row)"],
     note="Trial-weighted share = Σ trials in flagged blocks / Σ trials; participant-weighted = mean over participants of "
          "their flagged block share. Same participant-cluster bootstrap replicates as Table 4.1; one-sided vs p0; one test "
          "per row and estimand (q = p).")

In [ ]:
# [cell 20]
# ---- 4c. H2 classification across datasets: DSC, ARI, counts, features; T2 vs T3 ---------------------
H2_DSC = P("dsc_dynamic")
show(4, "4g", "Table 4.7. Overlap (DSC) of the flagged dynamic blocks with the behavioural standard of the same run",
     H2_DSC, note=f"Standard: the a-priori heuristic of the T2+T3 sweep (Pooled, overall) or the run's own inattent_behav arm (same "
                  f"definition). 95% CI: participant bootstrap (B = {B:,}) within each condition.")
show(4, "4g2", "Table 4.7b. Overlap (DSC) of the flagged units with the behavioural standard of the same run, by design",
     P("dsc_designs", H3_DS).assign(
         Degenerate=lambda d: np.where(np.minimum(d.Flagged, d.Units - d.Flagged) < 3, "yes", "")),
     nd_cols={"Units": 0, "Flagged": 0, "Heuristic flagged": 0, "Both": 0},
     note="Every design of the carried solutions: dynamic blocks, fixed control blocks, sessions and participants. DSC = "
          "2|A ∩ G| / (|A| + |G|) between the units the solution flags (A) and the units the behavioural standard flags (G), "
          "both in the design's units; Both = |A ∩ G|. Degenerate = fewer than three flagged or unflagged units (the H3 "
          "eligibility rule); its DSC is the overlap of flagging (nearly) every unit, not an agreement result. Dynamic: "
          "Table 4.7. " + p1_deg_text(H3_DS) + p1_dsc_text(H3_DS))
_DSCB = {k: v for r in RUN for k, v in REPL[r].items() if k.startswith("dsc|")}
_dpt = H2_DSC.set_index(["Dataset", "Solution"]).DSC
rows = []
for sol in PICKS:
    for a_, b_, lab in (("T3 only", "T2 only", "T3 only − T2 only (independent samples)"),
                        ("Pooled, T2", "T2 only", "Pooled T2 − T2 only (same participants)"),
                        ("Pooled, T3", "T3 only", "Pooled T3 − T3 only (same participants)")):
        if not {a_, b_} <= set(DS):                 # the contrasts of the datasets present
            continue
        d = _DSCB[f"dsc|{a_}|{sol}"] - _DSCB[f"dsc|{b_}|{sol}"]; lo, hi = ci(d)
        rows.append({"Solution": sol, "Contrast": lab, "Δ DSC": _dpt[(a_, sol)] - _dpt[(b_, sol)],
                     "95% CI": ci_str(lo, hi), "p": p_two(d) if lab.startswith("T3 only") else np.nan})
H2_DSC_D = pd.DataFrame(rows)
_m = H2_DSC_D.Contrast.str.startswith("T3 only"); H2_DSC_D.loc[_m, "q"] = bh(H2_DSC_D.loc[_m, "p"].to_numpy())
show(4, "4h", "Table 4.8. Change in DSC between the datasets", H2_DSC_D,
     note="T3 only vs T2 only: different participants, so the two bootstraps are independent; two-sided p, BH over the "
          "two solutions." + (" Pooled-derived vs independent: the same participants enter both, so only the difference "
                              "and a descriptive interval (independent bootstraps; conservative) are shown, without a "
                              "test." if POOLED_DS else "") + " Replicates: the runs' participant bootstraps of Table 4.7.")
_ARI_PANELS = {"robust summary": ["Refits", "Median", "Q1", "Q3", "sMAD"],
               "distribution": ["Mean", "SD", "2.5%", "97.5%", "Refits ARI < .50 (%)"]}
H2_ARI = P("stability")
show_panels(4, "4i", "Table 4.9. Cluster stability (bootstrap ARI) of the carried solutions across datasets", H2_ARI,
     ["Dataset", "Solution", "Blocks"], _ARI_PANELS, nd_cols={"Refits ARI < .50 (%)": 1, "Refits": 0},
     note=f"As Table 3.4 ({int(CFG2['stability_refits']):,} refits, refitted in each run). "
          + ("Pooled T2 / T3: within each refit of the pooled solution, the ARI on that condition's resampled blocks "
             "only. " if POOLED_DS else "")
          + "Bimodal distributions: read the 2.5–97.5% range and the share of refits with ARI < .50 alongside the median. "
          + p1_stab_text(H3_DS)
          + (f" No exact-zero block in {', '.join(_p1_even)}: an even number of blocks." if _p1_even else ""))
_COUNT_COLS = ["Dataset", "Solution", "Blocks", "Flagged n (%)", "Unflagged n (%)", "Participants",
               "Participants with ≥ 1 flagged block", "Trials in flagged blocks n", "Trials, all blocks",
               "Trials in flagged blocks (%)"]
_counts = P("block_counts")
show(4, "4j", "Table 4.10. Flagged and unflagged dynamic blocks of the carried solutions, all datasets",
     _counts[_COUNT_COLS], nd_cols={"Trials in flagged blocks (%)": 1})
for i, fam in enumerate(FEAT_LABELS):
    show(4, f"4k_{i}", f"Table 4.{11 + i}. Block features of the carried solutions, "
         f"{'T2 only, T3 only and the pooled-derived datasets' if POOLED_DS else 'T2 only and T3 only'} — {fam}",
         FEAT_SHOW(P(f"block_features_{i}", [ds for ds in DS if ds != "Pooled, overall"])),
         _FEAT_NOTE + " Pooled, overall: Tables 3.9–3.14.")

# T2 only vs T3 only: direction and significance of the three primary tests, and independent vs pooled-derived change
rows = []
for test, t, est in (("size", H2_SIZE, "Share (blocks)"), ("speed", H2_SPEED, "Δ (flagged − unflagged)"), ("IRR", H2_IRR, "Δα")):
    for sol in PICKS:
        tt = t[t.Solution == sol].set_index("Dataset")
        r = {"Test": test, "Solution": sol}
        for ds in DS:
            r[ds] = f"{tt.loc[ds, est]:.3f} ({fmt_p(tt.loc[ds, 'q (row)'])})"
        sig = (tt["q (row)"] <= ALPHA)
        r["Same direction, all datasets"] = bool(tt.Direction.nunique() == 1)
        r["Significant (q ≤ .05), all datasets"] = bool(sig.all())
        r["Holds in T2 and T3 (direction and q ≤ .05)"] = bool(tt.loc["T3 only", "Direction"] == tt.loc["T2 only", "Direction"]
                                                             and sig["T3 only"] and sig["T2 only"])
        rows.append(r)
H2_REP = pd.DataFrame(rows)
show(4, "4l", "Table 4.17. H2 across the datasets: estimate (q, BH within row) of each primary test", H2_REP,
     note="Cells: estimate (q). Size: block-weighted flagged share; speed: Δ of block RT level; IRR: Δα. Every dataset is "
          "in-sample (the configuration was chosen on all data, T2 and T3)"
          + ("; the pooled rows share their participants with T2 only and T3 only."
             if POOLED_DS else "."))
if {"Pooled, T2", "Pooled, T3"} <= set(DS):          # Table 4.18: the pooled-derived datasets present
    _cnt = _counts.set_index(["Dataset", "Solution"])
    _flag_pct = lambda ds, sol: 100 * int(str(_cnt.loc[(ds, sol), "Flagged n (%)"]).split()[0]) / _cnt.loc[(ds, sol), "Blocks"]
    rows = []
    for sol in PICKS:
        for ind, pool in (("T2 only", "Pooled, T2"), ("T3 only", "Pooled, T3")):
            vals = lambda ds: {
                "Flagged blocks (%)": _flag_pct(ds, sol),
                "Speed Δ": H2_SPEED.set_index(["Dataset", "Solution"]).loc[(ds, sol), "Δ (flagged − unflagged)"],
                "IRR Δα": H2_IRR.set_index(["Dataset", "Solution"]).loc[(ds, sol), "Δα"],
                "DSC": _dpt[(ds, sol)],
                "ARI median": H2_ARI.set_index(["Dataset", "Solution"]).loc[(ds, sol), "Median"],
                **{c: _cnt.loc[(ds, sol), c] for c in ("Flagged median RT level (composite)",
                                                       "Flagged median RT variance (composite)",
                                                       "Flagged median entropy (composite)",
                                                       "Unflagged median RT level (composite)")}}
            vi, vp = vals(ind), vals(pool)
            for k in vi:
                rows.append({"Solution": sol, "Condition": ind.split()[0], "Measure": k, "Independent": vi[k],
                             "Pooled-derived": vp[k], "Pooled − independent": vp[k] - vi[k]})
    show(4, "4m", "Table 4.18. Independent vs pooled-derived solutions: change in the H2 summaries", pd.DataFrame(rows),
         note="Independent = T2 only or T3 only (fitted on that condition); pooled-derived = the same condition's blocks "
              "under the pooled T2 + T3 fit (assignments kept). Descriptive.")

H2H_SIZE, H2H_SPEED, H2H_IRR = P("heuristic_size"), P("heuristic_speed"), P("heuristic_irr")
_H2H = (H2H_SIZE, H2H_SPEED, H2H_IRR)
_hf = [t.Dataset.isin(IND_DS).to_numpy() & np.isfinite(t.p.to_numpy(float)) for t in _H2H]   # BH family: the T2
_nhq = int(sum(m.sum() for m in _hf))                                 # only and T3 only tests, each once
_qa = bh(np.concatenate([t.p.to_numpy(float)[m] for t, m in zip(_H2H, _hf)]))
_QH = f"q ({_nhq} tests)"
_i = 0
for t, m in zip(_H2H, _hf):
    t[_QH] = np.nan
    t.loc[m, _QH] = _qa[_i:_i + m.sum()]
    _i += m.sum()
_hq = pd.concat(_H2H, ignore_index=True)
_hin = (_hq.Dataset.isin(IND_DS) & _hq.p.notna()).to_numpy()
check(f"Tables 4.21–4.23: {_QH} = BH over the tests of the T2 only and T3 only rows, each test once"
      + ("; the pooled rows carry no q" if POOLED_DS else ""), _hin.sum() == _nhq and _hq.loc[~_hin, _QH].isna().all()
      and np.allclose(_hq.loc[_hin, _QH], multipletests(_hq.loc[_hin, "p"].astype(float), method="fdr_bh")[1], rtol=0,
                      atol=1e-12), f"{int(_hin.sum())} tests" + (f"; pooled rows {int((~_hin).sum())}" if POOLED_DS else ""))
_hpc = ["p", "q (row)", _QH]
_hdrop = ["Solution"]
_HNOTE = ("Unit: dynamic block, flagged by the behavioural heuristic (the a-priori standard of the T2+T3 sweep; the same "
          "definition in every run, checked in [cell 3]). Estimators, resamples and seeds are those of Tables 4.1–4.3, "
          "so these rows are directly comparable with the carried solutions. "
          + ("The rule does not depend on the classification, so the pooled-derived rows repeat the independent ones. "
             if POOLED_DS else "")
          + f"{_QH} = BH over the {_nhq} tests of the T2 only and T3 only rows of Tables 4.21–4.23, each test once "
          "(this analysis's own family)"
          + ("; the pooled rows repeat or combine those datasets' blocks and are described without it. " if POOLED_DS
             else ". "))
show(4, "4p", "Table 4.21. The behavioural heuristic as a classifier — size: flagged share of the dynamic blocks (one-sided)",
     H2H_SIZE.drop(columns=_hdrop), p_cols=_hpc, nd_cols={"p0": 3, "Participants": 0},
     note=_HNOTE + f"Share = Σ flagged blocks / Σ blocks; participant-cluster bootstrap (B = {B:,}). The rule fixes no "
          "number of profiles, so the minority claim is tested against p0 = 1/2: H1: share < 1/2; p = (#{share* ≥ 1/2} + "
          "1)/(B + 1). Cohen's h = 2·arcsin√share − 2·arcsin√(1/2). q (row) = BH over the row's three tests.")
show(4, "4q", "Table 4.22. The behavioural heuristic as a classifier — speed: flagged vs unflagged block RT level (one-sided)",
     H2H_SPEED.drop(columns=_hdrop), p_cols=_hpc,
     note=_HNOTE + "Speed = adj_rt_level_block (log-RT drift residual, age-adjusted; lower = faster). Δ = difference of "
          "block-weighted means; exp(Δ) = ratio of RT to drift-expected RT; d = Δ / pooled within-group SD of block values; "
          "participant-cluster bootstrap, H1: Δ < 0. The heuristic's own rule includes a speed term, so this direction is "
          "expected by design; the test shows it is significant, not merely present. Descriptives over blocks.")
show(4, "4r", "Table 4.23. The behavioural heuristic as a classifier — IRR: Krippendorff's α of flagged vs unflagged "
     "responses (two-sided)", H2H_IRR.drop(columns=_hdrop), p_cols=_hpc,
     note=_HNOTE + f"Canonical α (ordinal; nominal for emotion) per scale × condition; row value = mean over the row's "
          f"cells. Stimulus-paired bootstrap (B = {B:,}), the resamples of Table 4.3; two-sided p, 95% percentile CI. A row "
          f"below {MIN_COVERAGE:.0%} coverage in either matrix of any cell is excluded from inference (Table 6.3).")
_lc = P("labelled_components")
show(4, "4s", "Table 4.24. The labelled component of every carried fit: speed ranking, margin and profile", _lc,
     nd_cols={"Units": 0, "Components": 0},
     note="Speed = mean adj_rt_level_block of the component's units (lower = faster); the fastest component is the "
          "labelled (inattentive) one. Margin = next-fastest − labelled. P(labelled stays fastest) = share of "
          f"{B:,} participant-cluster resamples (partition held fixed) in which the labelled component's mean speed is "
          "below every other component's. Δ = the labelled component's mean minus the grand mean of the fit's units, on "
          "the power-transformed features each labeller was fitted on (entropy / RT variance exponents: "
          + "; ".join(f"{lab} {e[0]:g} / {e[1]:g}" for lab, e in EXPONENTS.items()) + "). "
          + ("Pooled T2 / T3 share the pooled fit and are not listed. " if POOLED_DS else "") + p1_deg_text())
show(4, "4t", "Table 4.25. Stability refits of the dynamic fits: share of blocks in each refit's fastest component",
     P("stability_fastest"), nd=1, nd_cols={"Refits": 0},
     note=f"The {int(CFG2['stability_refits']):,} refits of Table 4.9 (blocks resampled with replacement, the labeller "
          "refitted with the run's settings and features). Share = blocks in the refit's fastest component / resampled "
          "blocks; the last column describes the fastest component's profile on the other two features. A wide or "
          "two-peaked share distribution means the labelled component is not well determined.")
show(4, "4u", "Table 4.26. Components of every carried fit: size and mean (SD) block features, by speed rank",
     P("components"), nd_cols={"Component": 0, "Speed rank": 0, "Units": 0},
     note="Every component of the fits of Table 4.24, ranked by mean adj_rt_level_block (rank 1 = fastest = labelled). "
          "Features as fitted (power-transformed RT variance and entropy). " + p1_deg_text())

# the exact-zero folded units ([cell 9]): where they sit, the carried fits with them replaced, the stability refits
show(4, "4y1", "Table 4.30. Exact-zero folded units: the median unit of every odd-sized matrix and its standardised "
     "position in each carried labeller's input", P1_ZERO, nd=2, nd_cols={"Units": 0},
     note=P1_FOLD_TXT + " Unit: id/session/block (sessions: id/session). One row per matrix, feature and carried solution "
          "(the solution's exponent): z = the power-transformed feature as the solution's labeller standardises it "
          "(k-means: SD with ddof 0; GMM: ddof 1); next unit = the lowest z among the other units; gap = next − zero (in SD). "
          + (f"At an exponent above 1 the exact-zero unit stays next to its neighbours (largest gap {P1_GAP_HI:.1e} SD). "
             if np.isfinite(P1_GAP_HI) else "")
          + "The matrices are rebuilt read-only from lvl2.py; the dynamic matrices equal the runs' _units.csv (Table 7.3).")
show(4, "4y2", "Table 4.31. Units flagged by every carried fit, as run and with the exact-zero folded values replaced by "
     "the next-smallest deviation", P1_CF.drop(columns=["_H", "_Hboth"]),
     nd_cols={c: 0 for c in ["Units", "Flagged (as run)", "Minority (as run)", *P1_VARS.values(), "Minority (both replaced)"]},
     note="Descriptive counterfactual. Replaced: the exact-zero |MAD-z| of the feature set to the smallest non-zero |MAD-z| "
          "of that feature in the same matrix, the power transform recomputed with the run's exponents, and lvl2.py's own "
          "Step 5 executed again on the altered matrices (the run's seeds, settings and labelling rule: the fastest "
          "component is the labelled one; every k-means refitted). As run = the run's flags (Table 4.7b; reproduced, Table "
          "7.3). A matrix with an even number of units has no exact-zero unit, so its counts do not change. Participant "
          "design: the participants with a flagged session in the session fit. Minority = the smaller of flagged and "
          f"unflagged units (H3 eligibility: ≥ {P1_MIN}). DSC with the heuristic = with the behavioural standard of the "
          "same run and design; as run vs both replaced = the overlap of the two flagged sets. " + p1_deg_text())
show(4, "4y3", "Table 4.32. Stability refits of the dynamic fits by the copies of the exact-zero blocks each refit drew",
     P1_STAB, nd_cols={"Blocks": 0, "Refits": 0, "Refits ARI < .50 (%)": 1},
     note=f"The refits of Table 4.9 (exported ARIs), split by the copies of the exact-zero folded-entropy block in refit "
          "b's resample, RandomState(random_seed + b).choice(n, n, replace=True), the stability stage's own (the first "
          f"{P1_NCHK} refits recomputed from the seeds: Table 7.3); the last row of each fit: refits that draw neither the "
          "exact-zero entropy block nor the exact-zero RT-variance block. "
          + (f"No exact-zero block in {', '.join(_p1_even)}: an even number of blocks. " if _p1_even else "")
          + P1_FOLD_TXT)

*[cell 21]*

## 5. LVL2 — H3 interrater reliability

**Primary measure:** canonical Krippendorff's α (ordinal; nominal for emotion), per scale × condition; dataset rows average their cells.

**Removal comparisons.** For each carried solution and design (dynamic blocks, fixed control blocks, sessions, participants), with all quantities computed per cell and averaged over the row's cells:

| Contrast | Definition |
|---|---|
| Overall vs baseline | α after removing the design's flagged units − α of the minimally screened data |
| Per-trial vs baseline | the same per stimulus-trial removed (× 10⁻⁶) |
| Overall vs null | mean D, D = α after the flagged removal − α after a matched random removal (within the same condition: block designs remove as many trials from as many random sessions as hold flagged blocks; session / participant designs remove as many random whole sessions / participants) |
| Per-trial vs null | D per stimulus-trial removed (× 10⁻⁶), each removal by its own number of trials removed |

**Inference.**
- **Vs baseline:** a stimulus-paired bootstrap, two-sided. One stimulus resample per condition and replicate is applied to the baseline and post-removal matrices and to all scales; raters are fixed.
- **Vs null (sanity check, not a hypothesis test):** the empirical 2.5–97.5 percentile interval of D over the random removals (the null given these data, not a confidence interval) and the two-sided Monte Carlo p, 2 × min(#{D ≤ 0} + 1, #{D ≥ 0} + 1)/(R + 1). Reported per scale and overall.
- **Design contrasts (dynamic vs control, session, participant; the LVL_2 runs' Step 9):** mean over the shared scale × condition cells of dynamic − other in each quantity, stimulus-paired bootstrap, two-sided; q = BH over the ≤ 9 tests of one quantity (3 comparisons × 3 removal methods). A design needs enough units in its minority profile to be tested. The gain per trial removed beyond the null decides the H3 design claims, with the totals and removal volumes beside it.
- **Null D-difference contrasts (separate tables):** every pair of eligible designs, D of one against D of the other by independent resampling with replacement of each design's D replicates (no pairing of replicate numbers): mean difference, empirical 95% interval, two-sided p.
- **FDR:** BH within each dataset × solution × contrast across the eligible designs.
- **Eligibility:** a comparison is excluded when its minority profile has too few units (blocks / control blocks / sessions / participants), or when a cell's post-removal matrix rates too few stimuli at least twice (Table 7.1).

In [ ]:
# [cell 22]
# ---- 5a. descriptive alpha: all, flagged and unflagged responses (the LVL_2 runs' Step 5e) ----------------------
H3_DESC = P("alpha_cells")
show_panels(5, "5a1", "Table 5.1a. Krippendorff's α of all, flagged and unflagged responses, per scale × condition", H3_DESC,
     ["Dataset", "Solution", "Condition", "Scale"], {"α": ["α all", "α flagged", "α unflagged", "Coverage flagged", "Coverage unflagged"],
                                                     "counts": ["Responses flagged", "Responses unflagged", "Raters flagged",
                                                                "Raters unflagged", "Stimuli rated (flagged)"]},
     note="Canonical α (ordinal; nominal for emotion) on the condition's full stimulus set. Responses = rater × stimulus "
          "ratings of that scale; raters = participants with ≥ 1 response in the matrix; coverage = share of stimuli with "
          "≥ 2 ratings. The same matrices as Table 4.3.")
show(5, "5a2", "Table 5.1b. Krippendorff's α across scales (and conditions), per dataset and solution", P("alpha_summary"),
     note=f"Median [Q1, Q3] (sMAD) over the row's cells ({len(SCALES)} scales per condition"
          + ("; both conditions for pooled overall)." if POOLED_DS else ")."))

In [ ]:
# [cell 24]
# ---- 5b. H3 removal comparisons (STEP 10's engine, every participant) and the designs compared (Step 9) -------------
H3_ARMS = {lab: arm for lab, (_, arm) in PICKS.items()}
RM_LAB = {**{arm: lab for lab, arm in H3_ARMS.items()}, "inattent_behav": "Heuristic"}
MIN_REMOVAL_UNITS = int(CFG2["irr_min_units"])
H3_ALL = P("h3_primary", H3_DS)
H3T = H3_ALL[H3_ALL.Outcome == "overall"].reset_index(drop=True)
_lab = {"imp": "Overall vs baseline", "imp_t": "Per-trial vs baseline (× 10⁻⁶)", "exc": "Overall vs null (D)",
        "exc_t": "Per-trial vs null (D per trial, × 10⁻⁶)"}


def h3_long(T):
    rows = []
    for _, r in T.iterrows():
        for k in ("imp", "imp_t", "exc", "exc_t"):
            rows.append({"Solution": r["Solution"], "Design": r["Design"], "% responses removed": r["% responses removed"],
                         "Contrast": _lab[k], "Δα": r[f"{k} est"], "95% CI / interval": r[f"{k} CI"], "p": r[f"{k} p"],
                         "q": r[f"{k} q"]})
    return pd.DataFrame(rows)


_cells_h3 = H3_ALL[H3_ALL.Outcome != "overall"]
ALPHA_NULL_GAP = float((_cells_h3["Reference α"] - _cells_h3["Random α (mean)"]).abs().max())
_h3note = ("Mean over the row's scale × condition cells. Vs baseline: α after removal − α of the minimally screened data; per trial "
           "= that change ÷ the trials the design removed (× 10⁻⁶); 95% CI and p from the stimulus-paired bootstrap "
           f"(B = {B:,}), two-sided. Vs null: D = (α_ref − α_random) − (α_ref − α_flagged) = α_flagged − α_random over the "
           "matched random removals (block designs: as many trials from as many random sessions as hold flagged blocks; "
           "session / participant designs: as many random whole sessions / participants), averaged over the cells within "
           "each removal; Δα = mean D (per trial: each removal by its own number of trials removed, × 10⁻⁶); the interval is "
           "the empirical 2.5–97.5 percentile range of D, which describes the random-removal null given these data, not a "
           "confidence interval; p = two-sided Monte Carlo p, 2 × min(#{D ≤ 0} + 1, #{D ≥ 0} + 1)/(R + 1). q = BH within "
           "dataset × solution × contrast across the eligible designs. Excluded comparisons: Table 5.5; per scale and the "
           "removal volumes: Tables 5.2f–j. "
           + ("The pooled datasets share their participants with T2 only and T3 only. " if POOLED_DS else "")
           + "For α the matched random removals leave α unchanged on average (their mean is within "
           f"{ALPHA_NULL_GAP:.4f} of the reference α in every cell), so for α 'vs null' adds only the null spread to 'vs "
           "baseline'; the null matters for the measures that change with missingness (ICC(2,k), split-half; Tables 5.6).")
_LET = {"T2 only": ("a", "f"), "T3 only": ("b", "g"), "Pooled, overall": ("c", "h"), "Pooled, T2": ("d", "i"),
        "Pooled, T3": ("e", "j")}
for ds in H3_DS:
    show(5, f"5b_{ds}", f"Table 5.2{_LET[ds][0]}. H3 removal comparisons (Krippendorff's α) — {ds}",
         h3_long(H3T[H3T.Dataset == ds]), _h3note, nd=4, nd_cols={"% responses removed": 2})
_ren = {"exc est": "Mean D", "exc CI": "95% interval of D", "exc p": "p (MC)", "exc q": "q (MC)",
        "exc_t est": "Mean D per trial (× 10⁻⁶)", "exc_t CI": "95% interval of D per trial", "exc_t p": "p (MC) per trial",
        "exc_t q": "q (MC) per trial"}
_p1 = ["Reference α", "Post α", "Δα flagged (ref − post)", "Random α (mean)", "Δα random (ref − random, mean)",
       "Mean D", "95% interval of D", "p (MC)", "q (MC)"]
_p2 = ["Mean D per trial (× 10⁻⁶)", "95% interval of D per trial", "p (MC) per trial", "q (MC) per trial",
       "Units flagged / total", "Units sampled per random removal", "Trials removed (flagged)",
       "Trials removed (random, mean)", "Valid iterations", "Invalid iterations"]
for ds in H3_DS:
    show_panels(5, f"5b2_{ds}", f"Table 5.2{_LET[ds][1]}. H3 flagged vs matched random removal, per scale and overall — {ds}",
                H3_ALL[H3_ALL.Dataset == ds].rename(columns=_ren), ["Solution", "Design", "Outcome"],
                {"flagged vs random": _p1, "per trial and removal volumes": _p2},
                note="Reference α = the minimally screened data (one per scale × condition, shared by the four designs). Δα = α_ref − "
                     "α after removal (positive = α fell). D, interval and p as in Table 5.2; each scale's row averages its "
                     "conditions, overall averages every cell. q = BH within dataset × solution × outcome across the "
                     "eligible designs. Units sampled per random removal: sessions for the block designs (those holding "
                     "flagged blocks), sessions or participants otherwise; Trials removed (random) is exact for the block "
                     "designs and varies with the units drawn otherwise. Invalid iterations: random removals whose α was "
                     "undefined or whose sampled sessions held too few trials (counted, never redrawn).",
                nd=4, p_cols=["p (MC)", "q (MC)", "p (MC) per trial", "q (MC) per trial"],
                nd_cols={"Mean D per trial (× 10⁻⁶)": 2, "Units sampled per random removal": 0,
                         "Trials removed (flagged)": 0, "Trials removed (random, mean)": 0, "Valid iterations": 0,
                         "Invalid iterations": 0})

# the designs compared: the LVL_2 runs' Step 9 tables, per dataset
DS_SCOPE = {"T2 only": ("T2", "T2"), "T3 only": ("T3", "T3"), "Pooled, overall": ("both", "all"),
            "Pooled, T2": ("both", "T2"), "Pooled, T3": ("both", "T3")}
PANEL = {"Excess per trial": "1. D per trial removed (vs null, × 10⁻⁶)", "Excess": "2. D (vs null)",
         "Per trial": "3. Gain per trial removed (vs baseline, × 10⁻⁶)", "Improvement": "4. Gain (vs baseline)"}
_PER_TRIAL_Q = ("Excess per trial", "Per trial")


def step9_pull(src, metrics, outcome=None):
    """A Step 9 table (TESTS9, NULL9 or CELLS9) of the H3 datasets for `metrics`; per-trial quantities x 10^6; outcome
    filters the null table (overall / per scale)."""
    out = []
    for ds, (run, sc) in DS_SCOPE.items():
        t = src[run]
        t = t[t.Metric.isin(metrics) & (t.Condition == sc)]
        if outcome is not None:
            t = t[(t.Outcome == "overall") if outcome == "overall" else (t.Outcome != "overall")]
        out.append(t.assign(Dataset=ds))
    T = pd.concat(out, ignore_index=True)
    f = np.where(T.Quantity.isin(_PER_TRIAL_Q), 1e6, 1.0)
    for c in ("mean_a", "mean_b", "diff", "ci_low", "ci_high"):
        if c in T.columns:
            T[c] = T[c] * f
    T["Solution"] = T.Removal_Method.map(RM_LAB)
    T["95% CI / interval"] = [ci_str(a, b, 5) for a, b in zip(T.ci_low, T.ci_high)]
    if "Comparison" in T.columns and src is TESTS9:
        T["Panel"] = T.Quantity.map(PANEL)
    return T


H3FT = step9_pull(TESTS9, ["Alpha"])
H3NC = step9_pull(NULL9, ["Alpha"], "overall")
H3NC_S = step9_pull(NULL9, ["Alpha"], "scale")
_n9 = H3FT.groupby(["Dataset", "Quantity"]).size()
check("Step 9 figure tests (alpha): 9 per dataset x panel", len(_n9) == len(H3_DS) * 4 and (_n9 == 9).all(), _n9.to_dict())
_dq = [np.nanmax(np.abs(bh(g.p.to_numpy()) - g.q.to_numpy())) for _, g in H3FT.groupby(["Dataset", "Quantity"]) if g.p.notna().any()]
check("Step 9 figure tests: q = BH over the panel's eligible tests", len(_dq) > 0 and max(_dq) < 1e-12, f"max |diff| {max(_dq):.1e}")
_el = set(map(tuple, H3T[["Dataset", "Solution", "Design"]].values))
_ft = H3FT[H3FT.Removal_Method.isin(H3_ARMS.values())]
_mis = [(r.Dataset, r.Solution, r.Comparison) for r in _ft.itertuples()
        if bool(r.Eligible) != ((r.Dataset, r.Solution, "dynamic") in _el and (r.Dataset, r.Solution, r.Comparison.split(" vs ")[1]) in _el)]
check("Step 9 eligibility = STEP 10's eligible designs (k-means, GMM)", not _mis, f"{len(_ft)} tests; mismatches {_mis[:4]}")
_e = H3T.set_index(["Dataset", "Solution", "Design"])["exc est"]
_dm = [max(abs(r.mean_a - _e[(r.Dataset, r.Solution, "dynamic")]), abs(r.mean_b - _e[(r.Dataset, r.Solution, r.Comparison.split(" vs ")[1])]))
       for r in _ft[(_ft.Quantity == "Excess") & _ft.Eligible].itertuples()
       if (r.Dataset, r.Solution, "dynamic") in _e.index and (r.Dataset, r.Solution, r.Comparison.split(" vs ")[1]) in _e.index]
check("Step 9 design means (D) = Table 5.2 mean D", len(_dm) > 0 and max(_dm) < 1e-9,
      f"{len(_dm)} tests, max |diff| {max(_dm) if _dm else np.nan:.1e}")
_h3ftnote = ("From the LVL_2 runs' Step 9 (the figures' brackets). Δ = mean over the dataset's shared scale × condition "
             "cells of dynamic − other in the panel's quantity; 95% CI and two-sided p from the stimulus-paired bootstrap (every "
             "design scored on the same stimulus resamples; for the vs-null panels each design's mean change under its "
             "matched random removals is held fixed; per trial: the change ÷ the trials the design removed). q = BH over the "
             f"panel's eligible tests (3 comparisons × 3 removal methods). Eligible = both designs have ≥ {MIN_REMOVAL_UNITS} "
             "units in their minority profile (flagged and unflagged, design's own unit); otherwise no test (degenerate "
             "design). Panel 1 (D per trial removed) decides the H3 design claims (Table 6.1), with the totals of panel 2 "
             "beside it.")
_ftcols = ["Panel", "Solution", "Comparison", "Eligible", "mean_a", "mean_b", "diff", "95% CI / interval", "p", "q"]
for ds in H3_DS:
    show(5, f"5d_{ds}", f"Table 5.3{_LET[ds][0]}. H3 design contrasts (Krippendorff's α): dynamic vs control, session and "
         f"participant removal — {ds}", H3FT[H3FT.Dataset == ds][_ftcols]
         .rename(columns={"mean_a": "Dynamic", "mean_b": "Other", "diff": "Δ"}), _h3ftnote, nd=5)
_h3ncnote = ("From the LVL_2 runs' Step 9. Every pair of eligible designs: Δ = mean of D*_A − D*_B over the runs' "
             "independent draws with replacement from each design's D replicates (replicate numbers never paired); the "
             "interval (2.5–97.5 percentiles of the draws) and the two-sided p describe the spread of the two null-relative "
             "distributions, not participant-level sampling uncertainty. q = BH within dataset × solution × outcome × "
             "quantity over the pairs. Per trial: × 10⁻⁶.")
_nccols = ["Dataset", "Solution", "Quantity", "Comparison", "mean_a", "mean_b", "diff", "95% CI / interval", "p", "q"]
show(5, "5d_n", "Table 5.3f. H3 null D-difference contrasts (Krippendorff's α, overall)", H3NC[_nccols]
     .rename(columns={"mean_a": "Mean D (A)", "mean_b": "Mean D (B)", "diff": "Δ"}), _h3ncnote, nd=5)
show(5, "5d_ns", "Table 5.3g. H3 null D-difference contrasts per scale (Krippendorff's α)",
     H3NC_S[["Outcome"] + _nccols].rename(columns={"mean_a": "Mean D (A)", "mean_b": "Mean D (B)", "diff": "Δ"}),
     _h3ncnote + " Per scale within each condition.", nd=5)

In [ ]:
# [cell 25]
# ---- 5c. H3 comparisons excluded from inference (STEP 10) -------------------------------------------------------
_full = H3T.set_index(["Dataset", "Solution", "Design"])          # the primary comparisons by design (Table 5.13)
H3_EXCL = P("h3_primary_excluded", H3_DS)
show(5, "5f", "Table 5.5. H3 comparisons excluded from inference",
     H3_EXCL if len(H3_EXCL) else pd.DataFrame(columns=["Dataset", "Solution", "Design", "Analysis", "Reason"]),
     note=f"A comparison is excluded when its minority profile has < {MIN_REMOVAL_UNITS} units in the design's own unit "
          f"(dynamic / control blocks, sessions, participants), or when any cell's post-removal α matrix rates < "
          f"{MIN_COVERAGE:.0%} of the stimuli at least twice.", nd_cols={"% responses removed": 2})

In [ ]:
# [cell 26]
# ---- 5d. secondary reliability measures (the runs' Step 8, overall rows) and their design contrasts (Step 9) --------
SEC = {"ICC2_1": "ICC(2,1)", "ICC2_k": "ICC(2,k)", "Kappa": "Fleiss κ", "SH_Between": "SH-between", "SH_Within": "SH-within"}
_elig = set(map(tuple, H3T[["Dataset", "Solution", "Design"]].values))
rows = []
for ds, (run, cond) in DS_SCOPE.items():
    o = NULL8[run][(NULL8[run].Condition == cond) & (NULL8[run].Outcome == "overall")]
    m8 = STEP8[run][(STEP8[run].Condition == cond) | (cond == "all")]
    for sol, arm in H3_ARMS.items():
        for des in DESIGNS:
            if (ds, sol, des) not in _elig:
                continue
            r8 = o[(o.Removal_Method == arm) & (o.Design == des)]
            if not len(r8):
                continue
            r8 = r8.iloc[0]
            pct = float(m8[(m8.Removal_Method == arm) & (m8.Design == des)].Pct_Removed.mean())
            for col, met in SEC.items():
                if not np.isfinite(r8.get(f"{col}_Baseline", np.nan)):
                    continue
                g = lambda k: r8.get(f"{col}_{k}", np.nan)
                rows.append({"Dataset": ds, "Solution": sol, "Design": des, "Measure": met, "Cells": int(g("Cells")),
                             "% trials removed": pct, "Baseline": g("Baseline"), "Post": g("Post"),
                             "Random (mean)": g("Null_Value_Mean"), "Improvement": g("Improvement"),
                             "Improvement 95% CI": ci_str(g("Improvement_CI_Low"), g("Improvement_CI_High"), 4),
                             "Improvement p": g("Improvement_P_Boot"),
                             "Per trial (× 10⁻⁶)": 1e6 * g("Improvement_PerTrial"),
                             "Per trial 95% CI": ci_str(1e6 * g("Improvement_PerTrial_CI_Low"),
                                                        1e6 * g("Improvement_PerTrial_CI_High"), 2),
                             "Per trial p": g("Improvement_PerTrial_P_Boot"),
                             "Excess over null": g("Excess"), "D 95% interval": ci_str(g("D_Lo"), g("D_Hi"), 4),
                             "p (MC)": g("P_MC"), "Excess per trial (× 10⁻⁶)": 1e6 * g("Excess_PerTrial"),
                             "Valid iterations": int(g("Null_Valid")), "Invalid iterations": int(g("Null_Invalid"))})
H3S = pd.DataFrame(rows)
for _, g in H3S.groupby(["Dataset", "Solution", "Measure"]):
    H3S.loc[g.index, "q (MC)"] = bh(g["p (MC)"].to_numpy())
    H3S.loc[g.index, "Improvement q"] = bh(g["Improvement p"].to_numpy())
    H3S.loc[g.index, "Per trial q"] = bh(g["Per trial p"].to_numpy())
_desc = ["Cells", "% trials removed", "Baseline", "Post", "Random (mean)"]
_vb = ["Improvement", "Improvement 95% CI", "Improvement p", "Improvement q", "Per trial (× 10⁻⁶)", "Per trial 95% CI",
       "Per trial p", "Per trial q"]
_vn = ["Excess over null", "D 95% interval", "p (MC)", "q (MC)", "Excess per trial (× 10⁻⁶)", "Valid iterations",
       "Invalid iterations"]
for j, met in enumerate(SEC.values()):
    show_panels(5, f"5g_{j}", f"Table 5.6{'abcde'[j]}. Secondary reliability measure: {met}", H3S[H3S.Measure == met],
         ["Dataset", "Solution", "Design"], {"descriptives": _desc, "vs baseline": _vb, "vs matched random removal": _vn},
         note="The LVL_2 runs' Step 8 (overall rows): per cell, then mean over the row's cells (ICC and split-half: the "
              "ordinal scales; κ: all scales). Per trial = the change ÷ the trials the design removed (× 10⁻⁶). Vs "
              "baseline: 95% CI and p from the stimulus-paired bootstrap of the row mean, two-sided. Vs null: D = measure "
              "after the flagged removal − after each matched random removal, averaged over the cells within each removal; "
              "Excess over null = mean D; the interval is the empirical 2.5–97.5 percentile range of D (the random-removal "
              "null given these data, not a confidence interval); p (MC) = two-sided Monte Carlo p. q = BH within dataset × "
              "solution × measure across the eligible designs, per column. SH-within measures the consistency of raters' "
              "mean levels, not agreement: it falls by construction when straight-lined blocks are removed. Sensitivity "
              "analyses.", nd=4, nd_cols={"% trials removed": 2, "Cells": 0, "Valid iterations": 0,
                                          "Invalid iterations": 0, "Per trial (× 10⁻⁶)": 2, "Excess per trial (× 10⁻⁶)": 2})
H3SFT = step9_pull(TESTS9, list(SEC)); H3SFT["Measure"] = H3SFT.Metric.map(SEC)
H3SNC = step9_pull(NULL9, list(SEC), "overall"); H3SNC["Measure"] = H3SNC.Metric.map(SEC)
_n9s = H3SFT.groupby(["Measure", "Dataset", "Quantity"]).size()
check("Step 9 figure tests (secondary measures): 9 per measure x dataset x panel",
      len(_n9s) == len(SEC) * len(H3_DS) * 4 and (_n9s == 9).all(), f"{len(_n9s)} panels")
show(5, "5g_d", "Table 5.6f. Secondary measures: dynamic vs control, session and participant removal", H3SFT[
     ["Measure", "Dataset"] + _ftcols].rename(columns={"mean_a": "Dynamic", "mean_b": "Other", "diff": "Δ"}),
     note=_h3ftnote.split(" Panel 1 (D per trial removed)")[0] + " ICC and split-half: the ordinal scales; κ: all scales.",
     nd=5)
show(5, "5g_n", "Table 5.6g. Secondary measures: null D-difference contrasts (overall)", H3SNC[
     ["Measure"] + _nccols].rename(columns={"mean_a": "Mean D (A)", "mean_b": "Mean D (B)", "diff": "Δ"}),
     note=_h3ncnote, nd=5)

In [ ]:
# [cell 28]
# ---- 5f. H3 across the datasets and agreement with the secondary measures ------------------------------------------
rows = []
for mlab in H3_ARMS:
    for des in DESIGNS:
        for k in ("imp", "imp_t", "exc", "exc_t"):
            r = {"Solution": mlab, "Design": des, "Contrast": _lab[k]}
            sigs, dirs = {}, {}
            for ds in H3_DS:
                t = H3T[(H3T.Dataset == ds) & (H3T.Solution == mlab) & (H3T.Design == des)]
                if not len(t):
                    r[ds] = "excluded"; continue
                t = t.iloc[0]; r[ds] = f"{t[f'{k} est']:.4f} ({fmt_p(t[f'{k} q'])})"
                sigs[ds], dirs[ds] = t[f"{k} q"] <= ALPHA, np.sign(t[f"{k} est"])
            both = "T2 only" in sigs and "T3 only" in sigs
            r["Holds in T2 and T3 (direction and q ≤ .05)"] = (bool(both and sigs["T2 only"] and sigs["T3 only"]
                                                              and dirs["T2 only"] == dirs["T3 only"]) if both else np.nan)
            rows.append(r)
H3_REP = pd.DataFrame(rows)
show(5, "5k", "Table 5.10. H3 across the datasets: Δα (q) of every removal contrast", H3_REP,
     note="Cells: Δα (BH q within dataset × solution × contrast; vs null: Monte Carlo); per trial × 10⁻⁶. Every dataset is "
          "in-sample (calibration on all data); the pooled datasets share their participants with T2 only and "
          "T3 only.")
rows = []
_a = H3T.set_index(["Dataset", "Solution", "Design"])
for (ds, mlab, des, met), g in H3S.set_index(["Dataset", "Solution", "Design", "Measure"]).groupby(level=[0, 1, 2, 3]):
    a = _a.loc[(ds, mlab, des)]; s = g.iloc[0]
    rows.append({"Dataset": ds, "Solution": mlab, "Design": des, "Measure": met,
                 "α excess": a["exc est"], "Measure excess": s["Excess over null"],
                 "Same sign as α": np.sign(a["exc est"]) == np.sign(s["Excess over null"]),
                 "α significant (MC q)": a["exc q"] <= ALPHA, "Measure significant (MC q)": s["q (MC)"] <= ALPHA})
H3_AGREE = pd.DataFrame(rows)
_sum = (H3_AGREE.groupby("Measure").agg(Comparisons=("Same sign as α", "size"), **{"Same sign as α (%)": ("Same sign as α", "mean")})
        .reset_index())
_sum["Same sign as α (%)"] *= 100
_sum["Same MC significance as α (%)"] = H3_AGREE.groupby("Measure").apply(
    lambda g: 100 * (g["α significant (MC q)"] == g["Measure significant (MC q)"]).mean()).values
_ssig = H3_AGREE["α significant (MC q)"] == H3_AGREE["Measure significant (MC q)"]
_ssgn = H3_AGREE["Same sign as α"].astype(bool)
_sum = _sum.merge(pd.DataFrame({"Measure": H3_AGREE.Measure, "Same significance, same sign (%)": _ssig & _ssgn,
                                "Same significance, opposite sign (%)": _ssig & ~_ssgn,
                                "Different significance (%)": ~_ssig}).groupby("Measure").mean() * 100,
                  left_on="Measure", right_index=True, how="left")
show(5, "5l", "Table 5.11. Agreement of the secondary reliability measures with α (gain beyond matched random removal)", _sum,
     note="Over the eligible dataset × solution × design rows (Tables 5.2 and 5.6): share of rows where the measure's excess has "
          "the sign of α's excess, and where its Monte Carlo significance (q ≤ .05) agrees with α's. That share ignores "
          "direction: a measure significant in the direction opposite to α counts as agreeing. The last three columns "
          "split the rows by significance and sign (they add to 100%; the first two add to the significance share). "
          "The pooled rows share their participants with T2 only and T3 only.",
     nd=1)

In [ ]:
# [cell 29]
# ---- 5g. sensitivity: the session-2 data of the participants shown repeated stimuli dropped (STEP 10) ---------------
REPEATS = P("repeat_exposures").drop(columns="Dataset", errors="ignore").drop_duplicates()
_rid = " and ".join(str(i) for i in sorted(REPEATS["id"].unique()))
show(5, "5s1", f"Table 5.12. Sensitivity: H2 IRR with the session-2 data of participants {_rid} dropped",
     P("repeat_exposure_h2"),
     note="Δα = α flagged − α unflagged (mean over the row's scale × condition cells), as in Table 4.3. Primary: the "
          "participants' repeated session-2 responses masked; sensitivity: all their session-2 responses dropped. Same flags "
          f"and stimulus-paired bootstrap (B = {B:,}); two-sided p. Datasets without their condition are identical by "
          "construction.", nd=4, p_cols=["p (primary)", "p (session 2 dropped)"])
_rs = P("repeat_exposure_h3", H3_DS)
_rs = _rs[_rs.Outcome == "overall"]
rows = []
for _, r in _rs.iterrows():
    f_ = _full.loc[(r.Dataset, r.Solution, r.Design)] if (r.Dataset, r.Solution, r.Design) in _full.index else None
    for k in ("imp", "imp_t", "exc", "exc_t"):
        rows.append({"Dataset": r.Dataset, "Solution": r.Solution, "Design": r.Design, "Contrast": _lab[k],
                     "Δα (primary)": np.nan if f_ is None else f_[f"{k} est"], "q (primary)": np.nan if f_ is None else f_[f"{k} q"],
                     "Δα (session 2 dropped)": r[f"{k} est"], "95% CI / interval (session 2 dropped)": r[f"{k} CI"],
                     "q (session 2 dropped)": r[f"{k} q"],
                     "Difference": np.nan if f_ is None else r[f"{k} est"] - f_[f"{k} est"]})
show(5, "5s2", f"Table 5.13. Sensitivity: H3 removal comparisons with the session-2 data of participants {_rid} dropped",
     pd.DataFrame(rows), note="Contrasts, bootstrap, nulls, eligibility and BH families as in Table 5.2, recomputed on the "
                              "data without the participants' session 2 (flags unchanged; their dropped units leave no "
                              "responses). Primary values from Table 5.2. Datasets without their condition are identical by "
                              "construction.", nd=4, p_cols=["q (primary)", "q (session 2 dropped)"])

In [ ]:
# [cell 30]
# ---- 5h. sensitivity: GMM posterior threshold (P(inattentive) > .5 primary; stricter thresholds) (STEP 10) --------
GMM_LAB = next(lab for lab, (_, a) in PICKS.items() if a.startswith("mixture_gmm"))
POSTERIOR_THRESHOLDS = [float(t) for t in CFG2["gmm_posterior_thresholds"]]
_tlab = lambda t: f"> {t:g}".replace("0.", ".")
A = P("gmm_posterior_summary")
for c in [c for c in A.columns if c.startswith("Flagged at") and not c.endswith("(%)")]:
    A[c] = [f"{int(n)} ({p:.1f}%)" for n, p in zip(A[c], A[c + " (%)"])]
show(5, "5p1", f"Table 5.14a. {GMM_LAB}: distribution of the posterior P(inattentive) of every unit",
     A.drop(columns=[c for c in A.columns if c.endswith("(%)")]), nd=3,
     nd_cols={"Units in data": 0, "Units with a posterior": 0},
     note="Posterior probability of the inattentive (fastest) component for every unit of each design's fit; a unit is "
          "flagged when it exceeds the threshold (.5 = the most likely component for two components).")
show(5, "5p2", f"Table 5.14b. {GMM_LAB}: counts of units by P(inattentive), the full range in steps of .1",
     P("gmm_posterior_counts"), note="Counts of units per tenth of the posterior; the last bin includes 1.")
show_panels(5, "5p3", f"Table 5.15. {GMM_LAB}: H2 tests with blocks flagged at stricter posterior thresholds",
     P("gmm_threshold_h2"), ["Dataset", "P(inattentive)"],
     {"size and speed": ["Flagged blocks", "Blocks", "Share (blocks)", "Size p", "Size q", "Speed Δ", "Speed p", "Speed q"],
      "IRR": ["Δα", "Δα 95% CI", "IRR p", "IRR q", "Min coverage flagged", "Min coverage unflagged"]},
     note=f"Same tests, bootstraps (the same participant weights and stimulus resamples at every threshold) and row-wise BH "
          f"as Tables 4.1–4.3; only dynamic blocks with P(inattentive) above the threshold are flagged (> .5 = the primary "
          f"analysis; its rows equal Tables 4.1–4.3). Size: share of flagged blocks vs p0 = .5, one-sided (it falls by "
          f"construction as the threshold rises); speed: flagged − unflagged block RT level, one-sided; IRR: α flagged − α "
          f"unflagged, two-sided; q = BH over the row's three tests. Coverage floor {MIN_COVERAGE:.0%} as Table 4.3.",
     nd=4, p_cols=["Size p", "Size q", "Speed p", "Speed q", "IRR p", "IRR q"],
     nd_cols={"Flagged blocks": 0, "Blocks": 0, "Min coverage flagged": 2, "Min coverage unflagged": 2})
show(5, "5p3b", f"Table 5.15b. {GMM_LAB}: overlap (DSC) with the behavioural standard at each posterior threshold",
     P("gmm_threshold_dsc"), nd_cols={"Flagged blocks": 0, "Heuristic flagged": 0, "Both": 0},
     note="DSC of the dynamic blocks flagged at each threshold with the behavioural heuristic's flagged blocks (Table 4.7 at "
          "> .5).")
_g = P("gmm_threshold_h3", H3_DS)
_g = _g[_g.Outcome == "overall"]
rows = []
for (ds, des), gg in _g.groupby(["Dataset", "Design"], sort=False):
    for k in ("imp", "imp_t", "exc", "exc_t"):
        r = {"Dataset": ds, "Design": des, "Contrast": _lab[k]}
        for t in POSTERIOR_THRESHOLDS:
            x = gg[gg["P(inattentive)"] == _tlab(t)]
            r[_tlab(t)] = (f"{x.iloc[0][f'{k} est']:.4f} {x.iloc[0][f'{k} CI']} (q {fmt_p(x.iloc[0][f'{k} q'])})"
                           if len(x) else "excluded")
        rows.append(r)
show(5, "5p4", f"Table 5.16. {GMM_LAB}: H3 removal comparisons (Krippendorff's α) with units flagged at stricter posterior "
     "thresholds", pd.DataFrame(rows),
     note="Each cell: estimate [95% CI or interval] (q), as in Table 5.2 (same engine, nulls and BH families); per trial × "
          f"10⁻⁶. Columns at > .5 equal Table 5.2. Excluded = minority profile < {MIN_REMOVAL_UNITS} units or post-removal "
          f"coverage < {MIN_COVERAGE:.0%} (Table 5.17).")
show(5, "5p5", f"Table 5.17. {GMM_LAB}: units flagged and eligibility at each posterior threshold",
     P("gmm_threshold_units", H3_DS), nd_cols={"% responses removed": 2}, note=p1_deg_text(H3_DS, GMM_LAB))
_h3p5 = P("gmm_threshold_h3", H3_DS)
_h3p5 = _h3p5[(_h3p5["P(inattentive)"] == _tlab(0.5)) & (_h3p5.Outcome == "overall")].set_index(["Dataset", "Design"])
_h3g = H3T[H3T.Solution == GMM_LAB].set_index(["Dataset", "Design"])
_dv5 = [abs(_h3p5.loc[i, f"{k} est"] - _h3g.loc[i, f"{k} est"]) for i in _h3g.index if i in _h3p5.index
        for k in ("imp", "imp_t", "exc", "exc_t")]
check("GMM posterior sensitivity: H3 at > .5 reproduces Table 5.2 (estimates)", len(_dv5) > 0 and max(_dv5) < 1e-12,
      f"{len(_dv5)} values, max |diff| {max(_dv5) if _dv5 else np.nan:.1e}")
show_figure(5, "f5_1", f"Figure 5.1. {GMM_LAB}: distribution of the posterior P(inattentive) per block",
            FIG_DIR / "supplement" / "fig_gmm_posterior.png",
            note="make_replication_figures from the T2 and T3 runs' replicate exports (_post_replicates.npz: the posterior "
                 "of every unit of each design's fit): histograms of P(inattentive) as % of the panel's blocks, dynamic "
                 "blocks (top row) and control blocks (bottom row), T2 only (left) and T3 only (right); dashed lines "
                 f"at P = {', '.join(f'{t:g}'.replace('0.', '.') for t in POSTERIOR_THRESHOLDS)}, with the number (%) of "
                 "blocks above each (Table 5.14a)." + (" The pooled run is not drawn." if POOLED_DS else ""))

# ---- H3 descriptives across the scale cells (Tables 5.18-5.19), and the supplementary tables 5.20-5.21 -------------
_MEAS = [("α", "Alpha"), ("ICC(2,1)", "ICC2_1"), ("ICC(2,k)", "ICC2_k"), ("Fleiss κ", "Kappa"), ("SH-between", "SH_Between"),
         ("SH-within", "SH_Within")]


def _cells(ds, sol, des, m):
    run, cd = DS_SCOPE[ds]; d_ = STEP8[run]
    g = d_[(d_.Condition == cd) & (d_.Removal_Method == H3_ARMS[sol]) & (d_.Design == des)].dropna(subset=[f"{m}_Excess"])
    return g.assign(nctr_pt=1e6 * g[f"{m}_Excess_PerTrial"], nctr_tot=g[f"{m}_Excess"],
                    bctr_pt=1e6 * g[f"{m}_Improvement"] / g.Trials_Removed, bctr_tot=g[f"{m}_Improvement"])


_msd = lambda x, nd: f"{x.mean():.{nd}f} ({x.std(ddof=1):.{nd}f})" if len(x) > 1 else "—"
_mdq = lambda x, nd: f"{x.median():.{nd}f} [{x.quantile(.25):.{nd}f}, {x.quantile(.75):.{nd}f}]" if len(x) else "—"
_QN = {"nctr_pt": ("per trial (× 10⁻⁶) vs null", 2), "nctr_tot": ("total vs null", 4),
       "bctr_pt": ("per trial (× 10⁻⁶) vs baseline", 2), "bctr_tot": ("total vs baseline", 4)}
for t_no, meas in (("5.18", [("α", "Alpha")]), ("5.19", [x for x in _MEAS if x[1] != "Alpha"])):
    rows = []
    for ds in H3_DS:
        for sol in H3_ARMS:
            for des in DESIGNS:
                for lab, m in meas:
                    g = _cells(ds, sol, des, m)
                    if not len(g):
                        continue
                    r = {"Dataset": ds, "Solution": sol, "Design": des, "Measure": lab, "Cells": len(g),
                         "% trials removed": g.Pct_Removed.mean()}
                    for q_, (ql, nd) in _QN.items():
                        r[f"{ql}: M (SD)"], r[f"{ql}: Mdn [Q1, Q3]"] = _msd(g[q_], nd), _mdq(g[q_], nd)
                    rows.append(r)
    _inel = sorted({(r["Dataset"], r["Solution"], r["Design"]) for r in rows}
                   - set(map(tuple, H3T[["Dataset", "Solution", "Design"]].values)))
    show_panels(5, f"5q_{t_no}", f"Table {t_no}. H3 {'Krippendorff’s α' if t_no == '5.18' else 'secondary measures'}: change from "
                f"each removal design across the scale cells, per trial removed and in total, vs matched random removal and vs baseline",
                pd.DataFrame(rows), ["Dataset", "Solution", "Design", "Measure", "Cells", "% trials removed"],
                {"vs matched random removal (D)": [f"{_QN[k][0]}: {s}" for k in ("nctr_pt", "nctr_tot") for s in ("M (SD)", "Mdn [Q1, Q3]")],
                 "vs baseline": [f"{_QN[k][0]}: {s}" for k in ("bctr_pt", "bctr_tot") for s in ("M (SD)", "Mdn [Q1, Q3]")]},
                nd=1, nd_cols={"Cells": 0},
                note="Per scale × condition cell (Step 8 of the run): D = post-removal value − mean of the matched random removals "
                     "(per trial: each removal by its own trials removed); baseline change = post-removal − minimally screened value; "
                     "per trial = the change / trials removed (× 10⁻⁶). M (SD), Mdn [Q1, Q3] across the cells (α and κ: every "
                     "scale; ICC and split-half: the ordinal scales). "
                     + ("Designs excluded from inference (Table 5.5) have no Step 8 change values and are not listed. " if not _inel
                        else "Designs excluded from inference (Table 5.5) are listed with their Step 8 values for "
                             f"completeness: {'; '.join(' / '.join(x) for x in _inel)}. ")
                     + "Tests: α Tables 5.2a–e (vs baseline), 5.2f–j (vs null) and 5.3a–e (designs); secondary measures "
                       "Tables 5.6a–f. Pooled, overall is not listed: its cells are those of Pooled, T2 and "
                       "Pooled, T3. The pooled rows share their participants with T2 only and T3 only.")
_H3S_i = H3S.set_index(["Dataset", "Solution", "Design", "Measure"])
rows20, rows21, _ntr = [], [], []
_Q9 = {"Excess per trial": ("vs null", "per trial"), "Excess": ("vs null", "total"),
       "Per trial": ("vs baseline", "per trial"), "Improvement": ("vs baseline", "total")}
for ds, (run, cond) in DS_SCOPE.items():
    for sol, arm in H3_ARMS.items():
        for lab, col in _MEAS:
            g = _cells(ds, sol, "dynamic", col)
            if not len(g) or not ((H3T.Dataset == ds) & (H3T.Solution == sol) & (H3T.Design == "dynamic")).any():
                continue
            T_ = float(g.Trials_Removed.iloc[0]); _ntr.append(g.Trials_Removed.nunique() == 1)
            pt = lambda v: 1e6 * v / T_
            qn, qb = bh(g[f"{col}_P_MC"].to_numpy(float)), bh(g[f"{col}_Improvement_P_Boot"].to_numpy(float))
            sc_rows = []
            for i, rr in enumerate(g.itertuples()):
                D, lo, hi = getattr(rr, f"{col}_Excess"), getattr(rr, f"{col}_D_Lo"), getattr(rr, f"{col}_D_Hi")
                I_, blo, bhi = (getattr(rr, f"{col}_Improvement"), getattr(rr, f"{col}_Improvement_CI_Low"),
                                getattr(rr, f"{col}_Improvement_CI_High"))
                sc_rows.append({"Dataset": ds, "Solution": sol, "Measure": lab, "Scale": rr.Scale,
                                "vs null: per trial (× 10⁻⁶)": pt(D), "vs null: per trial 95% interval": ci_str(pt(lo), pt(hi), 2),
                                "vs null: total (D)": D, "vs null: total 95% interval": ci_str(lo, hi, 4), "vs null: q": qn[i],
                                "vs baseline: per trial (× 10⁻⁶)": pt(I_), "vs baseline: per trial 95% CI": ci_str(pt(blo), pt(bhi), 2),
                                "vs baseline: total": I_, "vs baseline: total 95% CI": ci_str(blo, bhi, 4), "vs baseline: q": qb[i]})
            if lab == "α":
                r_ = H3T[(H3T.Dataset == ds) & (H3T.Solution == sol) & (H3T.Design == "dynamic")].iloc[0]
                over = {"vs null: per trial (× 10⁻⁶)": r_["exc_t est"], "vs null: per trial 95% interval": r_["exc_t CI"],
                        "vs null: total (D)": r_["exc est"], "vs null: total 95% interval": r_["exc CI"],
                        "vs null: q": r_["exc q"], "vs baseline: per trial (× 10⁻⁶)": r_["imp_t est"],
                        "vs baseline: per trial 95% CI": r_["imp_t CI"], "vs baseline: total": r_["imp est"],
                        "vs baseline: total 95% CI": r_["imp CI"], "vs baseline: q": r_["imp q"]}
            else:
                r_ = _H3S_i.loc[(ds, sol, "dynamic", lab)]
                o = NULL8[run]; o = o[(o.Condition == cond) & (o.Outcome == "overall") & (o.Removal_Method == arm)
                                      & (o.Design == "dynamic")].iloc[0]
                over = {"vs null: per trial (× 10⁻⁶)": 1e6 * o[f"{col}_Excess_PerTrial"],
                        "vs null: per trial 95% interval": ci_str(1e6 * o[f"{col}_DpT_Lo"], 1e6 * o[f"{col}_DpT_Hi"], 2),
                        "vs null: total (D)": o[f"{col}_Excess"],
                        "vs null: total 95% interval": ci_str(o[f"{col}_D_Lo"], o[f"{col}_D_Hi"], 4),
                        "vs null: q": r_["q (MC)"], "vs baseline: per trial (× 10⁻⁶)": r_["Per trial (× 10⁻⁶)"],
                        "vs baseline: per trial 95% CI": r_["Per trial 95% CI"], "vs baseline: total": r_["Improvement"],
                        "vs baseline: total 95% CI": r_["Improvement 95% CI"], "vs baseline: q": r_["Improvement q"]}
            rows20.append({"Dataset": ds, "Solution": sol, "Measure": lab, "Scale": "Overall", **over})
            rows20 += sc_rows
            # Table 5.21: dynamic - control per scale cell (Step 9's per-cell tests) and over the cells (Step 9's design test)
            c9 = CELLS9[run]; t9 = TESTS9[run]
            c9 = c9[(c9.Metric == col) & (c9.Condition == cond) & (c9.Removal_Method == arm)
                    & (c9.Comparison == "dynamic vs control")]
            t9 = t9[(t9.Metric == col) & (t9.Condition == cond) & (t9.Removal_Method == arm)
                    & (t9.Comparison == "dynamic vs control")]
            res = {}
            for q_, (ref, kind) in _Q9.items():
                f_ = 1e6 if kind == "per trial" else 1.0
                x9 = t9[t9.Quantity == q_]
                if len(x9):
                    x9 = x9.iloc[0]
                    res.setdefault("Overall", {})[q_] = (f_ * x9["diff"], f_ * x9.ci_low, f_ * x9.ci_high, x9.q)
                for x in c9[c9.Quantity == q_].itertuples():
                    res.setdefault(x.Scale, {})[q_] = (f_ * x.diff, f_ * x.ci_low, f_ * x.ci_high, x.q)
            for s, v in res.items():
                r = {"Dataset": ds, "Solution": sol, "Measure": lab, "Scale": s}
                for q_, (ref, kind) in _Q9.items():
                    est, lo, hi, q = v.get(q_, (np.nan, np.nan, np.nan, np.nan))
                    nd_ = 2 if kind == "per trial" else 4
                    r[f"{ref}: Δ {kind}" + (" (× 10⁻⁶)" if kind == "per trial" else "")] = est
                    r[f"{ref}: {kind} 95% CI"] = ci_str(lo, hi, nd_)
                    r[f"{ref}: {kind} q"] = q
                rows21.append(r)
check("Table 5.20: one trials-removed count per dataset x solution (every scale of the dynamic design)", all(_ntr), "")
_T = pd.DataFrame(rows20)
show_panels(5, "5q_5.20", "Table 5.20. H3 per scale and overall: change from dynamic-block removal, every reliability measure", _T,
            ["Dataset", "Solution", "Measure", "Scale"],
            {"vs matched random removal": ["vs null: per trial (× 10⁻⁶)", "vs null: per trial 95% interval", "vs null: total (D)",
                                           "vs null: total 95% interval", "vs null: q"],
             "vs baseline": ["vs baseline: per trial (× 10⁻⁶)", "vs baseline: per trial 95% CI", "vs baseline: total",
                             "vs baseline: total 95% CI", "vs baseline: q"]},
            nd=4, nd_cols={"vs null: per trial (× 10⁻⁶)": 2, "vs baseline: per trial (× 10⁻⁶)": 2},
            note="Dynamic design. Per scale × condition cell (Step 8 of the run); Overall = mean over the measure's cells (α and "
                 "κ: every scale; ICC and split-half: the ordinal scales; emotion is nominal). Vs matched random removal: D = "
                 "the measure after the flagged removal − after a matched random removal (Overall: averaged over the cells "
                 "within each removal); total = mean D; the 95% interval is the 2.5–97.5 percentile range of D across the "
                 "random removals (the random-removal null given these data, not a confidence interval); two-sided Monte Carlo "
                 "p. Vs baseline: post-removal − minimally screened value, with the stimulus-paired bootstrap 95% CI and two-sided p. "
                 "Per trial = total ÷ the trials the dynamic design removed (one count for every scale of the condition) × 10⁶, "
                 "so its interval and q are the total's. q: per scale, BH over the measure's scale cells within dataset × "
                 "solution × reference; Overall, the row tests of Tables 5.2a–e (α) and 5.6a–e (BH across the eligible "
                 "designs). Pooled, overall is not listed: its cells are those of Pooled, T2 and Pooled, T3.")
_T = pd.DataFrame(rows21)
_N9 = {ds: sorted(set(g[g.p.notna()].groupby(["Metric", "Quantity"]).size()))
       for ds, g in pd.concat([H3FT, H3SFT]).groupby("Dataset", sort=False)}       # the BH family sizes of Step 9's panels
_N9_TXT = lambda short: ", ".join(f"{'/'.join(map(str, n))} in {ds.split()[0] if short else ds}" for ds, n in _N9.items()
                                 if (ds in IND_DS if short else ds != "Pooled, overall"))
_cols = lambda ref: [c for c in _T.columns if c.startswith(ref + ":")]
show_panels(5, "5q_5.21", "Table 5.21. H3 per scale and overall: dynamic minus control block removal, every reliability measure", _T,
            ["Dataset", "Solution", "Measure", "Scale"], {"vs matched random removal": _cols("vs null"), "vs baseline": _cols("vs baseline")},
            nd=4, nd_cols={c: 2 for c in _T.columns if "(× 10⁻⁶)" in c}, p_cols=[c for c in _T.columns if c.endswith(" q")],
            note="Δ = dynamic − control (positive = the dynamic design gained more), per scale × condition cell and Overall (mean "
                 "over the cells: α and κ every scale, ICC and split-half the ordinal ones). 95% CI and two-sided p from the "
                 "runs' Step 9 design test (both designs scored on the same stimulus resamples; vs matched random removal, each "
                 "design's mean change under its own matched random removals is held fixed). Per trial: vs null, the "
                 "difference in D per trial removed; vs baseline, the change per trial removed (the change ÷ the trials the "
                 "design removed); × 10⁻⁶. q: per scale, BH over the measure's scale cells within dataset × solution × "
                 f"quantity (Step 9's per-cell tests); Overall, the Step 9 test's own q (BH over the panel's eligible tests, "
                 f"{_N9_TXT(False)}; Tables 5.3a–e, 5.6f). A different per-scale test of the vs-null differences, the null "
                 "D-difference contrast "
                 "(independent draws of each design's D), is in Table 5.3g (α). Pooled, overall is not listed: its cells "
                 "are those of Pooled, T2 and Pooled, T3.")

*[cell 31]*

## 6. Consolidated results

Assembled from the tables above (no statistic is recomputed here).

In [ ]:
# [cell 32]
# ---- 6. consolidated tables ----------------------------------------------------------------
_p = NUM["2e"]
rows = [{"Hypothesis": "H1", "Claim": f"{WINNER_LVL1} has the highest {r['Criterion']} and no detector exceeds it",
         "Dataset": r["Dataset"], "Holds": (r["Numerical best"].replace(" †", "") == WINNER_LVL1
                                             and r["Significantly above † (q ≤ .05)"] == "—"),
         "Detail": f"best = {r['Numerical best']}; † rank {r['† rank']}; above †: {r['Significantly above † (q ≤ .05)']}"}
        for _, r in _p.iterrows()]
for t, test in ((H2_SIZE, "size: flagged blocks are a minority"), (H2_SPEED, "speed: flagged blocks are faster"),
                (H2_IRR, "IRR: flagged responses agree less")):
    for _, r in t.iterrows():
        good = {"size": r.get("Direction") == "minority", "speed": r.get("Direction") == "flagged faster",
                "IRR": r.get("Direction") == "flagged less reliable"}[test.split(":")[0]]
        rows.append({"Hypothesis": "H2", "Claim": f"{r['Solution']} — {test}", "Dataset": r["Dataset"],
                     "Holds": bool(good and r["q (row)"] <= ALPHA),
                     "Detail": "excluded (coverage)" if r.get("Direction") == "excluded (coverage)" else f"q (row) {fmt_p(r['q (row)'])}"})
for _, r in H3T.iterrows():
    rows.append({"Hypothesis": "H3", "Claim": f"{r['Solution']} — {r['Design']} removal raises α over the minimally screened data (overall)",
                 "Dataset": r["Dataset"], "Holds": bool(r["imp est"] > 0 and r["imp q"] <= ALPHA),
                 "Detail": f"Δα {r['imp est']:.4f}, q {fmt_p(r['imp q'])}; per trial {r['imp_t est']:.2f} × 10⁻⁶"})
_tot = H3FT[H3FT.Quantity == "Excess"].set_index(["Dataset", "Solution", "Comparison"])
_vol = H3T.set_index(["Dataset", "Solution", "Design"])["Trials removed (flagged)"]
for _, r in H3FT[(H3FT.Quantity == "Excess per trial") & H3FT.Removal_Method.isin(H3_ARMS.values())].iterrows():
    t_ = _tot.loc[(r["Dataset"], r["Solution"], r["Comparison"])]
    other = r["Comparison"].split(" vs ")[1]
    vols = [_vol.get((r["Dataset"], r["Solution"], d_), np.nan) for d_ in ("dynamic", other)]
    rows.append({"Hypothesis": "H3", "Claim": f"{r['Solution']} — {r['Comparison'].replace(' vs ', ' − ')}: dynamic gains "
                                              "more per trial removed beyond its matched null (D per trial)",
                 "Dataset": r["Dataset"], "Holds": bool(r["diff"] > 0 and r["q"] <= ALPHA) if r["Eligible"] else np.nan,
                 "Detail": (f"Δ {r['diff']:.2f} × 10⁻⁶ per trial, q {fmt_p(r['q'])}; total Δ {t_['diff']:.5f}, q "
                            f"{fmt_p(t_['q'])}; trials removed {vols[0]:,.0f} vs {vols[1]:,.0f}") if r["Eligible"]
                 else "degenerate design"})
CLAIMS = pd.DataFrame(rows)
_w = CLAIMS.pivot_table(index=["Hypothesis", "Claim"], columns="Dataset", values="Holds", aggfunc="first").reindex(columns=["T2+T3 calibration"] + DS)
_w = _w.map(lambda v: "—" if pd.isna(v) else ("yes" if v else "no")).reset_index()
_w["Holds in T2 and T3"] = [("not tested" if "—" in (a, b) else ("yes" if (a == "yes" and b == "yes") else "no"))
                             for a, b in zip(_w["T2 only"], _w["T3 only"])]
_p1d = [k for k in P1_DEG_TXT if k[0] in H3_DS]                  # degenerate designs of the H3 datasets
_p1i = [k for k in _p1d if k[0] in IND_DS]                     # the degenerate designs of T2 only and T3 only
_p1c = lambda c: any(c.startswith(f"{sol} —") and (f" {des} removal" in c or f"− {des}:" in c) for _, des, sol in _p1i)
_p1nt = _w[(_w["Holds in T2 and T3"] == "not tested") & _w.Claim.map(_p1c)]
_p1ok = all(P1_CF.set_index(["Dataset", "Design", "Solution"]).loc[k, "Minority (both replaced)"] >= P1_MIN for k in _p1i)
_P1_61 = ((f" {len(_p1nt)} of the claims 'not tested' rest on degenerate designs ("
           + ", ".join(f"{ds} {sol} {des}" for ds, des, sol in _p1i) + ")"
           + (f"; with the exact-zero folded values replaced each of these designs has a minority profile of at least "
              f"{P1_MIN} units (Table 4.31), so these claims are untested because of the feature construction" if _p1ok else "")
           + ". " + p1_deg_text(IND_DS)) if len(_p1nt) else "")
show(6, "6a", "Table 6.1. Every hypothesis claim across the datasets", _w,
     note="'yes' = the claim holds in that dataset at q ≤ .05 in the stated direction (H1: the selected detector is the numerical "
          "best and no detector is significantly above it). H3 design claims: the difference in D per trial removed (Table "
          "5.3, panel 1), with the totals and removal volumes in CLAIMS' detail. Sources: Tables 2.6, 4.1–4.3, 5.2 and 5.3. "
          "— = not tested (excluded or not applicable). Every dataset is in-sample (calibration on all data); the "
          "pooled datasets share their participants with T2 only and T3 only. H1 is tested in T2+T3 calibration "
          "and in T3 only, so its 'Holds in T2 and T3' is not tested. H2 speed holds by construction in every "
          "dataset that is its own fit, T2 only, T3 only and Pooled, overall included: the labelling rule makes the flagged component the fastest, so there its 'yes' is not a "
          "test that can fail; Table 4.2 gives the size of the speed difference." + _P1_61)
SANITY = pd.DataFrame([{"Dataset": r["Dataset"], "Solution": r["Solution"], "Design": r["Design"],
                        "% responses removed": r["% responses removed"], "Mean D": r["exc est"],
                        "95% interval of D": r["exc CI"], "p (MC)": r["exc p"], "q (MC)": r["exc q"],
                        "Beats random removal": bool(r["exc est"] > 0 and r["exc q"] <= ALPHA)}
                       for _, r in H3T.iterrows()])
show(6, "6a2", "Table 6.1b. Sanity check: removal vs matched random removal (Krippendorff's α, overall)", SANITY,
     note="Not a hypothesis test. D = α after the flagged removal − α after each matched random removal (Table 5.2); the "
          "interval is the empirical 2.5–97.5 percentile range of D (the random-removal null given these data); p (MC) = "
          "two-sided Monte Carlo p; q = BH within dataset × solution across the eligible designs. Beats random removal = mean "
          "D > 0 and q ≤ .05. Per scale: Tables 5.2f–j. The pooled rows share their participants with T2 only and "
          "T3 only.", nd=4, p_cols=["p (MC)", "q (MC)"],
     nd_cols={"% responses removed": 2})
ROB_B = P("h2_robustness_b")
_mu = int(CFG2["h2_robustness_b_min_raters"])
ex = [{"Analysis": f"H3 {r['Analysis']}", "Item": f"{r['Dataset']} / {r['Solution']} / {r['Design']}", "Status": "excluded",
       "Reason": r["Reason"]} for _, r in H3_EXCL.iterrows()]
ex += [{"Analysis": "H3 design contrasts (Step 9)", "Item": f"{r['Dataset']} / {r['Solution']} / {r['Comparison']}",
        "Status": "not tested (degenerate design)",
        "Reason": f"minority profile: dynamic {r['Minority_Units_a']:.0f}, other {r['Minority_Units_b']:.0f} units "
                  f"(< {MIN_REMOVAL_UNITS} in either)"}
       for _, r in H3FT[(H3FT.Quantity == "Excess") & ~H3FT.Eligible.astype(bool)].iterrows()]
ex += [{"Analysis": "H2 robustness B", "Item": f"{r['Dataset']} / {r['Solution']}", "Status": "excluded",
        "Reason": f"raters per half {r['Raters per half (min)']} < {_mu}"} for _, r in ROB_B[~ROB_B.Eligible.astype(bool)].iterrows()]
_rbc = ROB_B["Splits meeting the coverage floor (%)"]
ex += [{"Analysis": "H2 robustness B", "Item": f"{r['Dataset']} / {r['Solution']}", "Status": "excluded",
        "Reason": f"no split meets the coverage floor {MIN_COVERAGE:.2f} in both matrices of every cell"}
       for _, r in ROB_B[ROB_B.Eligible.astype(bool) & (_rbc == 0)].iterrows()]
ex += [{"Analysis": "H2 robustness B", "Item": f"{r['Dataset']} / {r['Solution']}", "Status": "reported with caveat",
        "Reason": f"coverage floor {MIN_COVERAGE:.2f} met in {r['Splits meeting the coverage floor (%)']:.0f}% of splits; "
                  "Δα, CI and p over those splits"}
       for _, r in ROB_B[ROB_B.Eligible.astype(bool) & (_rbc > 0) & (_rbc < 100)].iterrows()]
_A_IRR = P("h2_robustness_a"); _A_IRR = _A_IRR[_A_IRR.Test.astype(str).str.contains("IRR")]
_h2cov = [("H2 IRR", H2_IRR, lambda r: f"{r['Dataset']} / {r['Solution']}")]
_gt = P("gmm_threshold_h2")
ex += [{"Analysis": a, "Item": f(r), "Status": "excluded",
        "Reason": f"coverage (min over cells) flagged {r['Min coverage flagged']:.2f}, unflagged {r['Min coverage unflagged']:.2f}; "
                  f"floor {MIN_COVERAGE:.2f} in both"}
       for a, T, f in _h2cov for _, r in T[T.p.isna()].iterrows()]
ex += [{"Analysis": "H2 robustness A (IRR)", "Item": f"{r['Dataset']} / {r['Solution']}", "Status": "excluded",
        "Reason": f"coverage below the floor {MIN_COVERAGE:.2f}"} for _, r in _A_IRR[_A_IRR.p.isna()].iterrows()]
ex += [{"Analysis": "H2 IRR, GMM posterior", "Item": f"{r['Dataset']} / P {r['P(inattentive)']}", "Status": "excluded",
        "Reason": f"coverage (min over cells) flagged {r['Min coverage flagged']:.2f}, unflagged {r['Min coverage unflagged']:.2f}; "
                  f"floor {MIN_COVERAGE:.2f} in both"} for _, r in _gt[_gt["IRR p"].isna()].iterrows()]
ex += [{"Analysis": a, "Item": i, "Status": s, "Reason": r} for a, i, s, r in (
    ("H1", "T2 and T3 tested separately at each detector's own optimal configuration", "not computed",
     "the sweep's sessions (T2 and T3) are tested together at each detector's own optimum; T3 only at the "
     "selected configuration (the LVL_1 handoff)"),
    ("LVL2 calibration", f"participant-design DSC across the {_n_cfg} sweep configurations", "blocked",
     "the sweep's flagged-unit files do not contain the participant design (picks: Table 3.3)"),
    ("LVL2 calibration", "ROC AUC against the a-priori standard", "not computed (decision F.8)",
     "only the dose–response AUC is reported"),
    )]
_p1r = [e["Item"] for e in ex if any(e["Item"] in (f"{ds} / {sol} / {des}", f"{ds} / {sol} / dynamic vs {des}")
                                        for ds, des, sol in _p1d)]
show(6, "6c", "Table 6.3. Excluded, unavailable and statistically ineligible results", pd.DataFrame(ex),
     note=(f"{len(_p1r)} rows ({len(set(_p1r))} items: " + "; ".join(dict.fromkeys(_p1r)) + ") follow from degenerate designs. "
           + p1_deg_text(H3_DS)) if _p1r else "")

*[cell 33]*

## 7. Statistical notes, checks, provenance — and the exports

In [ ]:
# [cell 34]
# ---- 7. notes, checks and provenance ------------------------------------------------------------------------
import platform, sklearn, scipy, statsmodels
_NR = {k: int(v) for k, v in CFG2["irr_null_reps"].items()}
NOTES = pd.DataFrame([
    ("Scaled MAD", "1.4826 × median absolute deviation from the median; every 'Mdn [Q1, Q3] (sMAD)' cell."),
    ("Minimally screened data", "the analysed responses before any removal: out-of-range codes and RTs above the upper RT limit are already missing; fast responses and practice stimuli are kept. The H3 baseline (reference α)."),
    ("η²_H", "Kruskal–Wallis effect size (H − k + 1)/(n − k) (Cohen, 2008); kept because the k − 1 term penalises over-segmentation."),
    ("Corr%", "share of a session's three features significant after BH across the detector's validatable sessions of the dataset; unsplit = not significant; descriptive."),
    ("H1 tests", "every pair of the 16 detectors (120) in four tests, η²_H and Corr% in T2+T3 calibration (every session, each detector at its own η²_H-optimal or Corr%-optimal configuration) and in T3 only (the selected configuration): two-sided participant-cluster paired bootstrap of the difference in means, each detector's mean over its own sessions with a value, 95% percentile interval; a tie counts in both tails."),
    ("Participant-cluster bootstrap", f"participants resampled with replacement with all their blocks (multinomial weights), B = {B:,}; H2 size and speed; DSC CIs; H1 detector pairs (participants with all their sessions)."),
    ("Stimulus-paired bootstrap", f"stimuli resampled with replacement, one resample per condition and replicate shared by every matrix and scale; raters fixed; B = {B:,}; H2 IRR, H3 gains vs the minimally screened data, the H3 design contrasts. Secondary measures use the first n of the same resamples; ICC and split-half are refitted on every resample, each resampled stimulus copy entering as a separate stimulus. Split-half: every score of a scale × condition cell (baseline, post-removal, each matched random removal and each bootstrap replicate) uses the cell's one split set of {int(CFG2['splithalf_bootstrap_reps']):,} random splits, seeded from splithalf_seed ({int(CFG2['splithalf_seed'])}) with the scale and condition."),
    ("Bootstrap p", "one-sided (#{θ* ≥ null} + 1)/(B + 1); two-sided 2 × the smaller tail, each with + 1, capped at 1."),
    ("Matched random-removal null", f"per flagged removal, {_NR['alpha']:,} random removals within the same condition (κ {_NR['kappa']:,}; ICC and split-half the first {_NR['icc']:,} / {_NR['splithalf']:,}): the block designs sample as many sessions as hold flagged blocks and remove exactly as many trials from them (shuffled, so unevenly across the sampled sessions); the session / participant designs remove as many random whole sessions / participants. One trial-level removal per replicate applies to every scale (trials with missing responses stay in the trial index). D = (α_ref − α_random) − (α_ref − α_flagged) = α_flagged − α_random, with α_ref the minimally screened data (one per scale × condition, shared by the four designs). Reported: mean D, the empirical 2.5–97.5 percentile interval of D (the null given these data, not a confidence interval) and the two-sided Monte Carlo p = 2 × min(#{{D ≤ 0}} + 1, #{{D ≥ 0}} + 1)/(R + 1). Invalid replicates (undefined α, too few trials) are counted, never redrawn or zero-filled. The removal-vs-null comparisons are a sanity check, not a hypothesis test."),
    ("Per trial removed", "every per-trial value is an absolute change in the measure divided by the stimulus-trials the design removed (vs null: each random removal by its own count), shown × 10⁻⁶. It puts every dataset, design and measure on one scale; % of trials removed is reported beside it as the removal volume."),
    ("Design contrasts", f"dynamic vs control, session and participant removal (the LVL_2 runs' Step 9): mean over the shared scale × condition cells of dynamic − other in each of the four quantities (D per trial, D, gain per trial, gain), stimulus-paired bootstrap (each design's mean change under its matched random removals held fixed for D), two-sided; q = BH over the ≤ 9 tests of one panel (3 comparisons × 3 removal methods); only designs with ≥ {MIN_REMOVAL_UNITS} units in the minority profile are tested. Panel 1 (D per trial removed) decides the H3 design claims, with the totals (panel 2) and the removal volumes beside it. The null D-difference contrasts (every pair of eligible designs, independent draws with replacement from each design's D replicates, replicate numbers never paired) are reported separately; their interval and p describe the spread of the two null-relative distributions, not participant-level sampling uncertainty."),
    ("Krippendorff's α", "canonical estimator (Krippendorff, 2011): coincidences weight each stimulus's pairs by 1/(m_u − 1); ordinal distances from pairable marginals; nominal for emotion."),
    ("FDR families", "H1: the 120 pairs of each test; H2: the 3 primary tests per row (all rows as sensitivity); the heuristic as a classifier (Tables 4.21–4.23): the 3 tests per row, and across rows the T2 only and T3 only rows, each test once" + (" (pooled rows without it)" if POOLED_DS else "") + "; robustness: per row; H3: per dataset × solution × contrast across designs; design contrasts: per dataset × panel over the ≤ 9 dynamic-vs-other tests; null D-difference contrasts: per dataset × solution × outcome × quantity over the pairs; secondary IRRs: per dataset × solution × measure; dose-response AUC: per family (overall, per-trial), in the distinct-solution test (Table 3.6) and in its per-configuration sensitivity analysis (Table 3.6b, descriptive)."),
    ("Eligibility", f"H3 comparisons need ≥ {MIN_REMOVAL_UNITS} units in the minority profile (design's own unit) and ≥ {MIN_COVERAGE:.0%} of stimuli rated at least twice after removal. H2 IRR (primary, robustness A, GMM posterior thresholds) needs ≥ {MIN_COVERAGE:.0%} of stimuli rated at least twice in both the flagged and the unflagged matrix of every cell; robustness B applies the same floor to each split and uses only the splits that meet it."),
    ("Selection", "all data (T2 and T3) chose the LVL_1 detector and configuration, the LVL_2 exponents and k, and the heuristic thresholds (the all-data 99th-percentile markers of LVL_0): every result is in-sample (LVL_0 drift models were fitted on T2 + T3)."),
    *([("Pooled T2 + T3", "the pooled run (one joint fit, on the calibration data) and its T2 / T3 subsets are reported for H1 (descriptives), H2, H3 and the classification tables; they share their participants with T2 only and T3 only.")] if POOLED_DS else []),
    ("Repeat exposures (data issue)", f"participants {_rid} were shown in session 2 stimuli they had already rated in session 1 (a session-assignment error); all other participants saw each of their stimuli once. Their behaviour is kept in every feature, block, size and speed analysis; for α and every other IRR the session-2 response to a repeated stimulus is masked (the first exposure is kept), in the LVL_2 runs. Removal volumes count every response. Tables 5.12–5.13 repeat H2 IRR and H3 with these participants' session-2 data dropped."),
    ("GMM posterior thresholds", "a unit is flagged when the GMM's posterior probability of the inattentive component exceeds .5 (for two components, the most likely component). Tables 5.14–5.17 and Figure 5.1 give the full distribution of the posteriors and repeat H2 and H3 with only the units above the stricter thresholds flagged."),
    ("Exact-zero folded units", P1_FOLD_TXT + " No step of LVL_2 treats the exact-zero unit differently from the other "
     "units; Tables 4.30–4.32 give where it sits, the carried fits with it replaced by the next-smallest deviation, and "
     "the stability refits by the copies of it they draw."),
    ("Speed test", "the labelling rule makes the flagged component the fastest of its fit, so the speed contrast holds by construction in every dataset that is its own fit; there it reports the size of the speed difference (Tables 4.2, 6.1) and is not a test that can fail."),
], columns=["Item", "Definition / note"])
show(7, "7a", "Table 7.1. Statistical definitions and conventions", NOTES)
show(7, "7r", "Table 7.2. Repeat exposures masked in the reliability analyses (session-assignment error)",
     REPEATS.rename(columns={"condition": "Condition", "id": "Participant", "sessions": "Session of the masked responses",
                             "masked_stimuli": "Stimuli repeated (masked per scale)"}))
_fp = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]
_sw = lambda d: json.load(open(SWEEPS_DIR / d / "_provenance.json"))
_prov = [("LVL_0 handoff", Path(f"{LV0}.csv").name, f"raw input sha256 {json.load(open(f'{LV0}_parameters.json'))['outputs'].get('input_long_sha256', '—')[:16]}"),
         ("LVL_1 handoff", HANDOFF.name, f"built from {META.get('lvl0_input', '—')}"),
         ("LVL_1 sweep", LVL1_SWEEP, f"lvl1.py {_sw(LVL1_SWEEP)['pipeline_fingerprint']}; selection {SEL1['detector']} {SEL1['config']}"),
         ("LVL_2 sweep", LVL2_SWEEP, f"lvl2.py {_sw(LVL2_SWEEP)['pipeline_fingerprint']}; selection " + ", ".join(
             f"{lab} {cfg}" for lab, (cfg, _) in PICKS.items()))]
_prov += [(f"LVL_2 run ({k})", v, f"{RUNLOG[k].get('timestamp', '')}") for k, v in RUN_TAGS.items()]
_prov += [("Pipeline version", PIPELINE_VERSION, "development series; the literal of this report's [cell 1] and of the "
           "level notebooks' [cell 2]; the run logs' records: Table 7.3")]
_prov += [("Software", f"Python {platform.python_version()}", f"numpy {np.__version__}, pandas {pd.__version__}, scipy {scipy.__version__}, "
                                                             f"scikit-learn {sklearn.__version__}, statsmodels {statsmodels.__version__}")]
# every figure's provenance sidecar (FIG_DIR, recursively, linked or not): its pins = this report's, its PNG present, and
# every input it lists = the current file (bytes and SHA-256; names resolved in LVL_OUTS and SWEEPS_DIR)
_FILES = {}
for _root in (LVL_OUTS, SWEEPS_DIR):
    for _file in _root.rglob("*"):
        if _file.is_file():
            _FILES.setdefault(_file.name, []).append(_file)
_PINNED = (LVL_OUTS, SWEEPS_DIR / LVL1_SWEEP, SWEEPS_DIR / LVL2_SWEEP)


def _sidecar_files(name):
    """The current files a sidecar input may name: a path (absolute, or relative to the dissertation folder, LVL_OUTS or
    SWEEPS_DIR), else the files of that name in LVL_OUTS and the pinned sweeps' folders (else anywhere in SWEEPS_DIR)."""
    p = Path(name)
    if p.is_absolute() or len(p.parts) > 1:
        return [c for c in ([p] if p.is_absolute() else [Path("/home/sjsuss626/EMST_files/sjs_dissertation") / p,
                                                         LVL_OUTS / p, SWEEPS_DIR / p]) if c.is_file()][:1]
    locs = _FILES.get(name, [])
    return [f for f in locs if any(f.is_relative_to(r) for r in _PINNED)] or locs


_PINS_ME = {**_pins, "LVL0_HANDOFF": LVL0_HANDOFF.removesuffix(".csv")}
for _side in sorted(FIG_DIR.rglob("*.json")):
    _s, _prob = json.load(open(_side)), []
    _sp = dict(_s.get("pins") or {})
    _sp["LVL0_HANDOFF"] = str(_sp.get("LVL0_HANDOFF", "")).removesuffix(".csv")
    if _sp != _PINS_ME:
        _prob.append(f"pins {_s.get('pins')}")
    _png = _side.parent / str(_s.get("figure") or "")
    if not (_s.get("figure") and _png.is_file()):
        _prob.append(f"PNG {_s.get('figure')} missing")
    _ins = _s.get("inputs") or []
    for _inp in _ins:
        _cands = _sidecar_files(str(_inp.get("file", "")))
        if not _cands:
            _prob.append(f"{_inp.get('file')}: not found")
        elif not any(sha16(c) == (_inp.get("bytes"), _inp.get("sha256_16")) for c in _cands):
            _prob.append(f"{_inp.get('file')}: current {[sha16(c) for c in _cands]}, sidecar {(_inp.get('bytes'), _inp.get('sha256_16'))}")
    check(f"Figure {_side.relative_to(FIG_DIR).with_suffix('.png')}: sidecar pins = this report's, PNG present, its "
          f"{len(_ins)} inputs = the current files (bytes, SHA-256)", not _prob and len(_ins) > 0,
          "; ".join(_prob) if _prob else f"drawn {_s.get('drawn')}; PNG sha256_16 {sha16(_png)[1]}")
_chk = pd.DataFrame(CHECKS, columns=["Check", "Passed", "Detail"])
show(7, "7b", "Table 7.3. Validation checks run by this notebook", _chk)
show(7, "7c", "Table 7.4. Sources and provenance", pd.DataFrame(_prov, columns=["Input", "Name", "Fingerprint / detail"]))

In [ ]:
# [cell 35]
# ---- Markdown export (every registered table, in order; the appendix after section 7) and the LaTeX tables --------
def md_table(sdf):
    esc = lambda v: str(v).replace("|", "\\|").replace("\n", " ")
    head = "| " + " | ".join(esc(c) for c in sdf.columns) + " |"
    sep = "|" + "|".join("---" for _ in sdf.columns) + "|"
    return "\n".join([head, sep] + ["| " + " | ".join(esc(v) for v in r) + " |" for r in sdf.itertuples(index=False)])


def _md_items(sec, appendix):
    out = []
    for s_, key, title, sdf, note in TABLES:
        if s_ == sec and in_appendix(title) == appendix:
            out += [f"**{title}**", "", md_table(sdf), ""] + ([f"*Note.* {note}", ""] if note else [])
    for s_, key, title, path, note in FIGURES:
        if s_ == sec and in_appendix(title) == appendix:
            out += [f"**{title}**", "", f"![{title}]({os.path.relpath(path, REPORT_MD.parent)})", ""] + (
                [f"*Note.* {note}", ""] if note else [])
    return out


lines = [f"# Final report, practical application (calibration on all data) — {ANALYSIS_METHOD}", "",
         f"Generated by `Replication_and_final_report.ipynb` (pipeline version {PIPELINE_VERSION}) on "
         f"{time.strftime('%Y-%m-%d %H:%M')} from the pipeline's "
         "exports (besides the cross-dataset comparisons it computes Tables 2.8–2.9 and 4.30–4.32, the BH families stated under its "
         "tables and the summaries of Tables 5.11 and 5.18–5.19). The manuscript calls the two conditions cohorts: T2 = "
         "Cohort A (CA), T3 = Cohort B (CB); the tables keep T2 and T3, the figures print CA and CB. Every table is the "
         "notebook's own table object; numbers "
         "are not transcribed. Conventions: median [Q1, Q3] (sMAD), sMAD = 1.4826 × MAD; p-values below .001 are shown as "
         "< .001; † = the selected method; per trial = per trial removed × 10⁻⁶.", ""]
for sec in sorted(SECTION_TITLES):
    lines += [f"## {sec}. {SECTION_TITLES[sec]}", ""] + _md_items(sec, False)
lines += ["## Appendix. Supporting analyses", "",
          "Analyses the manuscript does not cite, kept for completeness, under their table numbers.", ""]
for sec in sorted(SECTION_TITLES):
    lines += _md_items(sec, True)
REPORT_MD.write_text("\n".join(lines))
_md = REPORT_MD.read_text()
_missing_tables = [k for _, k, t, sdf, _ in TABLES if f"**{t}**" not in _md or md_table(sdf) not in _md]
print(f"wrote {REPORT_MD} ({len(TABLES)} tables, {len(_md):,} characters); tables missing or altered: {_missing_tables or 'none'}")
assert not _missing_tables

# the four supplementary H3 tables (the manuscript's LaTeX layout), from Tables 5.20 and 5.21
ID = ["Dataset", "Solution", "Measure", "Scale"]
t20 = pd.concat([NUM["5q_5.20_a"], NUM["5q_5.20_b"].drop(columns=ID)], axis=1)
t21 = pd.concat([NUM["5q_5.21_a"], NUM["5q_5.21_b"].drop(columns=ID)], axis=1)
COL = {"dyn_nctr": {"per trial": ("vs null: per trial (× 10⁻⁶)", "vs null: per trial 95% interval", "vs null: q"),
                    "total": ("vs null: total (D)", "vs null: total 95% interval", "vs null: q")},
       "dyn_bctr": {"per trial": ("vs baseline: per trial (× 10⁻⁶)", "vs baseline: per trial 95% CI", "vs baseline: q"),
                    "total": ("vs baseline: total", "vs baseline: total 95% CI", "vs baseline: q")},
       "dmc_nctr": {"per trial": ("vs null: Δ per trial (× 10⁻⁶)", "vs null: per trial 95% CI", "vs null: per trial q"),
                    "total": ("vs null: Δ total", "vs null: total 95% CI", "vs null: total q")},
       "dmc_bctr": {"per trial": ("vs baseline: Δ per trial (× 10⁻⁶)", "vs baseline: per trial 95% CI", "vs baseline: per trial q"),
                    "total": ("vs baseline: Δ total", "vs baseline: total 95% CI", "vs baseline: total q")}}


def _tex_num(x, kind, neg=None):
    """Manuscript number format: per trial 2 decimals; totals 4 decimals without the leading zero; LaTeX minus. neg:
    the sign as printed in the report (an interval bound that rounds to zero keeps its minus: it lies below zero)."""
    if not np.isfinite(x):
        return "--"
    s = f"{abs(x):.2f}" if kind == "per trial" else f"{abs(x):.4f}".lstrip("0")
    neg = (x < 0 and float(s) != 0) if neg is None else neg
    return ("$-$" if neg else "") + s


def _tex_cell(t, tab, ds, sol, meas, scale, kind):
    g = t[(t.Dataset == ds) & (t.Solution == sol) & (t.Measure == meas) & (t.Scale == scale)]
    if len(g) != 1:
        return "--", ""
    vc, ic, qc = COL[tab][kind]
    r = g.iloc[0]
    v = float(r[vc]); m = re.match(r"\[(-?[\d.]+), (-?[\d.]+)\]", str(r[ic]))
    lo, hi = (float(m.group(1)), float(m.group(2))) if m else (np.nan, np.nan)
    nlo, nhi = (m.group(1).startswith("-"), m.group(2).startswith("-")) if m else (None, None)
    q = float(r[qc])
    st = "" if not np.isfinite(q) else "$^{***}$" if q < .001 else "$^{**}$" if q < .01 else "$^{*}$" if q < .05 else ""
    kap = "$^{\\kappa}$" if meas == "Fleiss κ" else ""
    return f"{_tex_num(v, kind)}{st}{kap}", f"[{_tex_num(lo, kind, nlo)}, {_tex_num(hi, kind, nhi)}]"


TITLES = {"dyn_nctr": ("Scale-wise reliability change from dynamic block removal: per trial and total ($nctr$)", "tab:supp_h3_dyn_nctr"),
          "dyn_bctr": ("Scale-wise reliability change from dynamic block removal: per trial and total ($bctr$)", "tab:supp_h3_dyn_bctr"),
          "dmc_nctr": ("Scale-wise difference in reliability change, dynamic minus control block removal: per trial and total ($nctr$)", "tab:supp_h3_dmc_nctr"),
          "dmc_bctr": ("Scale-wise difference in reliability change, dynamic minus control block removal: per trial and total ($bctr$)", "tab:supp_h3_dmc_bctr")}
_COMMON = ("The ICC(2,1)/$\\kappa$ column reports ICC(2,1) for the ordinal scales and Overall rows (mean of the ordinal "
           "scales) and Fleiss' $\\kappa$ for emotion ($^{\\kappa}$); -- = not computed (emotion is nominal). Overall rows "
           "average the cells (Krippendorff's $\\alpha$: every scale; ICC and split-half [SH$_{b}$ = between, SH$_{w}$ = "
           "within]: the ordinal scales).")
_ndes = H3T[H3T.Dataset.isin(IND_DS)].groupby(["Dataset", "Solution"], sort=False).size()       # the eligible designs behind each Overall row's q
_ND_TXT = str(_ndes.mode().iloc[0]) + "".join(f", except {ds.split()[0]} {sol}: {n}" for (ds, sol), n in _ndes.items()
                                            if n != _ndes.mode().iloc[0])
NOTES_TEX = {
    "dyn_nctr": "Cells: change for the dynamic-block design beyond matched random removal of equal volume [95\\% interval]. "
                "The change is the mean of $D$, the measure after the flagged removal minus after a random removal of the same "
                f"number of trials from as many sessions, over {_NR['alpha']:,} random removals ($\\alpha$), {_NR['kappa']:,} "
                f"($\\kappa$), {_NR['icc']:,} (ICC) or {_NR['splithalf']:,} (split-half); the interval is the 2.5--97.5 percentile range of $D$ across those removals (the spread of "
                "random removal, not a confidence interval). Per trial = total divided by the trials removed ($\\times "
                "10^{-6}$; one count for every scale), so it has the total's interval and $q$. " + _COMMON + " Two-sided Monte "
                "Carlo $p$, BH-FDR $q$ across the scales within dataset $\\times$ solution $\\times$ measure (Overall rows: "
                f"across the eligible removal designs, {_ND_TXT}).",
    "dyn_bctr": "Cells: change for the dynamic-block design from the minimally screened data [95\\% CI from the stimulus-paired "
                "bootstrap]. Per trial = total divided by the trials removed ($\\times 10^{-6}$; one count for every scale), "
                "so it has the total's interval and $q$. " + _COMMON + " Two-sided bootstrap $p$, BH-FDR $q$ across the scales "
                f"within dataset $\\times$ solution $\\times$ measure (Overall rows: across the eligible removal designs, {_ND_TXT}).",
    "dmc_nctr": "Cells: dynamic minus control difference in the change beyond matched random removal (positive = dynamic gained "
                "more) [95\\% CI from the stimulus-paired bootstrap: both designs scored on the same stimulus resamples, each "
                "design's mean change under its own random removals held fixed]. Per trial: per trial removed ($\\times "
                "10^{-6}$). " + _COMMON + " Two-sided bootstrap $p$; BH-FDR $q$ across the scales within dataset $\\times$ "
                f"solution $\\times$ measure (Overall rows: across the panel's eligible design tests, {_N9_TXT(True)}).",
    "dmc_bctr": "Cells: dynamic minus control difference in the change from the minimally screened data (positive = dynamic gained "
                "more) [95\\% CI from the stimulus-paired bootstrap: both designs scored on the same stimulus resamples]. Per "
                "trial: per trial removed ($\\times 10^{-6}$). " + _COMMON + " Two-sided bootstrap $p$; BH-FDR $q$ across the "
                "scales within dataset $\\times$ solution $\\times$ measure (Overall rows: across the panel's eligible design "
                f"tests, {_N9_TXT(True)})."}
_TSC = [("Overall", "\\textit{Overall}")] + [(s, s.capitalize()) for s in ("valence", "arousal", "similarity", "emotion", "threat")]
_TME = ["α", "ICC1/κ", "ICC(2,k)", "SH-between", "SH-within"]
out = ["% requires \\usepackage{rotating}",
       f"% Written by Replication_and_final_report.ipynb (pipeline version {PIPELINE_VERSION}) from Tables 2.10a-d (H1 "
       f"detector pairs), 5.20 (dynamic) and 5.21 (dynamic - control), {time.strftime('%Y-%m-%d %H:%M')}.",
       "% T2 = Cohort A (CA), T3 = Cohort B (CB) in the manuscript; these tables keep T2 and T3.", ""]
# the four supplementary H1 tables, from Tables 2.10a-d: per detector a Δ row (with the stars of its BH-adjusted p) and a
# (BH-adjusted p) row; the best detector and the detectors not separable from it in bold
_TEX_CRIT = {"η²_H": "$\\eta^2_H$", "Corr%": "$PorSig_{adj}$"}
_tex_txt = lambda s: re.sub(r"η²_H|Corr%|[_%&#]", lambda m: _TEX_CRIT.get(m.group(0), "\\" + m.group(0)), s)
_tex_name = lambda m, g: f"\\textbf{{\\boldmath {method_label(m)}}}" if g in H1_BOLD else method_label(m)
_tex_names = lambda o, grp, g: ", ".join(method_label(m) for m in o if grp[m] == g) or "none"
for _x, (_crit, _ds, _cfg, _order, _M, _D, _grp, _sdf, _B) in H1_MATS.items():
    _n, _dl = len(_order), _ds.split()[0]
    out += ["\\begin{sidewaystable}", "\t\\centering",
            f"\t\\caption{{Pairwise comparisons of mean {_TEX_CRIT[_crit]} between the {_n} detectors: difference in means "
            f"and BH-adjusted $p$ ({_dl})}}",
            f"\t\\label{{tab:supp_h1_{'eta' if _crit == 'η²_H' else 'corr'}_{_dl.lower()}}}", "\t\\scriptsize",
            "\t\\setlength{\\tabcolsep}{2pt}", f"\t\\begin{{tabular}}{{@{{}}l*{{{_n}}}{{c}}@{{}}}}", "\t\t\\toprule",
            "\t\tDetector & " + " & ".join(str(c + 1) for c in range(_n)) + " \\\\", "\t\t\\midrule"]
    for _r, _m in enumerate(_order):
        _st = [h1_stars(_M.iat[_r, c]) if c < _r else "" for c in range(_n)]
        out.append(f"\t\t{_r + 1}. {_tex_name(_m, _grp[_m])} & " + " & ".join(
            "---" if c == _r else "" if c > _r else h1_delta(_D.iat[_r, c], _crit, "$-$") + (f"$^{{{_st[c]}}}$" if _st[c] else "")
            for c in range(_n)) + " \\\\")
        out += ["\t\t & " + " & ".join(
            f"{{\\tiny ({h1_q(_M.iat[_r, c]).replace('< ', '$<$ ')})}}" if c < _r else "" for c in range(_n)) + " \\\\[2pt]"
                ] if _r else []
    _note = (f"Cells: $\\Delta$ ($p_{{adj}}$) of the row and the column detector: $\\Delta$ = the column detector's mean "
             f"{_TEX_CRIT[_crit]} minus the row detector's" + (" (percentage points)" if _crit == "Corr%" else "") + "; "
             "$p_{adj}$ = its BH-adjusted $p$ (two-sided participant-cluster paired bootstrap of the difference in means, "
             f"B = {_B:,}; BH-FDR over the {int(np.isfinite(_M.to_numpy()).sum())} pairs of the {_n} detectors of this "
             "test); --- = the detector itself. $p_{adj}$ to three decimals; four where three would round up to .050 or "
             ".010. "
             f"Configuration: {_tex_txt(_cfg)}. "
             + ("Detector 1 is the selected detector; the others are ordered by descending $p$ against it (ties: the "
                "smaller absolute difference in means first)." if _crit == "η²_H" else
                f"Detector 1 is the best detector (highest mean {_TEX_CRIT[_crit]}); the others are ordered by "
                "descending $p$ against it (ties: the smaller absolute difference in means first); the selected "
                f"detector, {method_label(WINNER_LVL1)}, is detector {_order.index(WINNER_LVL1) + 1}.")
             + " Bold: the best detector (highest mean; "
             f"{_tex_names(_order, _grp, 'best')}) and the detectors not separable from it, $p_{{adj}} > .05$ "
             f"({_tex_names(_order, _grp, 'not separable from the best')}).")
    out += ["\t\t\\bottomrule", "\t\\end{tabular}",
            f"\t\\par\\smallskip\\parbox{{\\textwidth}}{{\\scriptsize\\textit{{Note.}} {_note} $^{{*}}p_{{adj}} < .05$, "
            "$^{**}p_{adj} < .01$, $^{***}p_{adj} < .001$.}", "\\end{sidewaystable}", ""]
_n_h1 = len(out)
for tab, (cap, lab) in TITLES.items():
    t = t20 if tab.startswith("dyn") else t21
    out += ["\\begin{sidewaystable}", "\t\\centering", f"\t\\caption{{{cap}}}", f"\t\\label{{{lab}}}", "\t\\scriptsize",
            "\t\\setlength{\\tabcolsep}{2pt}", "\t\\begin{tabular}{@{}lll*{10}{c}@{}}", "\t\t\\toprule",
            "\t\t& & & \\multicolumn{5}{c}{Per trial ($\\times 10^{-6}$)} & \\multicolumn{5}{c}{Total} \\\\",
            "\t\t\\cmidrule(lr){4-8}\\cmidrule(l){9-13}",
            "\t\tCond. & Solution & Scale & $\\alpha$ & ICC(2,1)/$\\kappa$ & ICC(2,$k$) & SH$_{b}$ & SH$_{w}$ & $\\alpha$ & ICC(2,1)/$\\kappa$ & ICC(2,$k$) & SH$_{b}$ & SH$_{w}$ \\\\",
            "\t\t\\midrule"]
    first_ds = True
    for ds, dlab in (("T2 only", "T2"), ("T3 only", "T3")):
        for si, sol in enumerate(PICKS):
            if not (first_ds and si == 0):
                out.append("\t\t\\addlinespace")
            first_ds = False
            for j, (sc, slab) in enumerate(_TSC):
                est, itv = [], []
                for kind in ("per trial", "total"):
                    for m in _TME:
                        meas = ("Fleiss κ" if sc == "emotion" else "ICC(2,1)") if m == "ICC1/κ" else m
                        e, i = (("--", "") if (sc == "emotion" and m in ("ICC(2,k)", "SH-between", "SH-within"))
                                else _tex_cell(t, tab, ds, sol, meas, sc, kind))
                        est.append(e); itv.append(i)
                c0 = dlab if (si == 0 and j == 0) else ""
                c1 = sol if j == 0 else ""
                out.append(f"\t\t{c0} & {c1} & {slab} & " + " & ".join(est) + " \\\\")
                out.append("\t\t & & & " + " & ".join(f"{{\\tiny {x}}}" if x else "" for x in itv) + " \\\\[2pt]")
    out += ["\t\t\\bottomrule", "\t\\end{tabular}",
            f"\t\\par\\smallskip\\parbox{{\\textwidth}}{{\\scriptsize\\textit{{Note.}} {NOTES_TEX[tab]} $^{{*}}q < .05$, $^{{**}}q < .01$, $^{{***}}q < .001$.}}",
            "\\end{sidewaystable}", ""]
REPORT_TEX.write_text("\n".join(out))
print(f"wrote {REPORT_TEX} ({len(H1_MATS)} H1 and {len(TITLES)} H3 tables, {sum(l.count('[') for l in out[_n_h1:])} H3 "
      "cells with an interval)")
_failed = [c for c, ok, _ in CHECKS if not ok]
print("validation checks:", f"{len(CHECKS) - len(_failed)}/{len(CHECKS)} passed", _failed or "")
print(f"figures linked: {len(FIGURES)}")